# Natural Language Processing Project Presentation

# Authors:


*   Alessandro Marcassa
*   Lorenzo Franzé
*   Satvik Bisht
*   Teka Kimbi


#Task

1. Medical Meadow Medical Flashcards:

* Website: https://huggingface.co/datasets/medalpaca/medical_meadow_medical_flashcards

* Paper: https://arxiv.org/pdf/2304.08247.pdf
* Description: Information on medical curriculum flashcards has been given to GPT-3.5 and used to
create medical knowledge question answer pairs.
* Task: Medical Question Answering (i.e. train a model to answer medical questions.)




# Data Exploaration

# Introduction

**Basic Description of the Dataset and its Design**

The dataset in our study was derived from the Anki Medical Curriculum flashcards, a comprehensice collection curated and continually updated by medical students. Thes flashcards cover the full breadth of the medcial curriculum, encompassing subjects suche as anatomy, physiology, pathology and pharmacology. Each flashcard typically includes summaries to facilitate the learning and retention of key medical concepts. To create the dataset, the content was extracted from these flashcards, excluding those containing images.
Then, OpenAI's GPT 3.5- turbo was employed to transfrom the flashcard content into coherent, contextually relevant question-answer pairs.

The dataset consists of over 30,000 questions, each accompanied by a corresponding segment of text serving as its answer. Notably, these answers lack any additional context.

**Challenge in having only Question and Answers pairs without context**

A significant challenge of this dataset is the lack of extended context beyond the question and answer pairs. Typically, one of the most straightforward and simplet ways to train and fine-tune a question-answering model involves providing a context along with the question and answer.
Context aids the model in retriving the correct answer by grounding the question relevant information. However, the concise nature of the flashcard means that each pair must stand alone without additional explanatory medical concepts or see the connections between different pieces of information. Additionally, without context, it can be challenging to ensure that the questions are interpreted correctly and that the answers are sufficiently informative

**Our Work**

In tis notebook, we'll delve into a exploratory data analysis of the provided dataset. Our aim is to examine the data's structure and characteristics. Through this exploration, we seek to gain profound insights into both the dataset itself and the task at hand. Ultimately, our goal is to pinpoint a model that effectively addresses the complexity of this challenge

## Import and install

In [ ]:
!pip install -q python-terrier

In [ ]:
!pip install datasets

In [ ]:
pip install biopython

In [ ]:
!pip install keybert

In [ ]:
import string
from datasets import load_dataset
import re
import nltk
from nltk.corpus import stopwords
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
import altair as alt
from gensim.models import Word2Vec
from google.colab import drive
import os
import csv
import numpy as np
from nltk.stem import WordNetLemmatizer
import seaborn as sns
from wordcloud import WordCloud
from nltk.util import ngrams
from nltk import FreqDist
from Bio import Entrez
from keybert import KeyBERT
import pyterrier as pt
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm
import subprocess

## setup

In [ ]:
drive.mount('/content/drive')
os.chdir(f'/content/drive/MyDrive/Colab Notebooks/NLP/Assignment/datasets')
os.getcwd()
HUGGINGFACE = False
SAVE_DATASET = True
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
kw_model = KeyBERT()

## Utils

The function **preprocess_document** is designed to process a given document, performing various optional text preprocessing steps based on the specified parameters. These parameters include:

* isSorted: If set to True, the function sorts the processed document alphabetically.
* isSet: If set to True, the function converts the processed document into a set, removing any duplicate elements.
* noStopWords: If set to True, the function removes common English stopwords from the processed document.
* lemmatization: If set to True, the function lemmatizes the words in the processed document using WordNet.

The function begins by removing punctuation from the document using a regular expression. Then, depending on the parameters provided, it applies additional processing steps accordingly. Finally, it returns the processed document.

In [ ]:
def preprocess_document(document,noPunctuation=False ,isSorted=False, isSet=False, noStopWords=False, lemmatization=False):
  newDocument = document
  if noPunctuation==True:
    regex = '[' + string.punctuation + ']'
    newDocument = [re.sub(regex,'',item) for item in newDocument]
  if isSet==True:
    newDocument = set(newDocument)
  if noStopWords==True:
    newDocument =  [w for w in newDocument if w not in stopwords.words('english')]
  if isSorted==True:
    newDocument = sorted(newDocument)
  if lemmatization==True:
    lemmatizer = WordNetLemmatizer()
    newDocument = [lemmatizer.lemmatize(w) for w in newDocument]
  return newDocument


The function **create_global_vocabolary** operates on a DataFrame df containing document vocabularies. It aims to create a global vocabulary by aggregating all unique words from the document vocabularies within the DataFrame. The function offers optional parameters for further processing:

* isSorted: When set to True, the function sorts the global vocabulary alphabetically.
* isSet: If set to True, the function converts the global vocabulary into a set to remove any duplicate words.
* noStopWords: If set to True, the function removes common English stopwords from the global vocabulary.

The function iterates over each document vocabulary in the DataFrame, extending the global vocabulary with all unique words encountered. Depending on the specified parameters, it conducts additional processing such as sorting, converting to a set, and removing stopwords. Finally, it returns the resulting global vocabulary.

In [ ]:
def create_global_vocabolary(df,isSorted=False, isSet=False, noStopWords=False):
  global_voc = []
  for document_voc in df:
    global_voc.extend(word for word in document_voc)
  if isSet==True:
    global_voc = set(global_voc)
  if noStopWords==True:
    global_voc =  [w for w in global_voc if w not in stopwords.words('english')]
  if isSorted==True:
    global_voc = sorted(global_voc)
  return global_voc


This function, **nGrams**, generates and visualizes n-grams from a given document. It takes the document as input along with optional parameters for specifying the size of the n-grams and the number of most common n-grams to display. After generating the n-grams and counting their occurrences, it plots the frequency distribution of the top N most common n-grams. This visualization aids in understanding the distribution of word combinations in the document, helping to identify frequently occurring patterns.

In [ ]:
def nGrams(document,size=2,NmostCommon=10):
  # Generate n-grams from the list of words
  ngram_list = list(ngrams(document, size))
  # Count the occurrences of each n-gram
  ngram_counter = FreqDist(ngram_list)
  # Get the top N most common n-grams
  most_common_ngrams = ngram_counter.most_common(NmostCommon)
  # Separate n-grams and counts for plotting
  ngram_tuples, counts = zip(*most_common_ngrams)  # Change the variable name here
  # Convert n-gram tuples to strings for plotting
  ngram_strings = [" ".join(ngram) for ngram in ngram_tuples]
  # Plot the n-gram frequency distribution
  plt.figure(figsize=(10, 5), dpi=125)
  plt.bar(ngram_strings, counts)
  plt.xlabel("N-grams")
  plt.ylabel("Frequency")
  plt.title("Frequency Distribution of Top {} {}-grams after removing punctuation and stopwords".format(NmostCommon, size))
  plt.xticks(rotation=45)
  plt.show()


# Dataset loading

We initiate dataset download with this code snippet, offering flexibility in data retrieval. Depending on the value of the HUGGINGFACE flag, the script fetches the dataset from Hugging Face's repository or directly loads it from a CSV file stored on Google Drive. This versatile approach simplifies the process of acquiring the necessary data for analysis or modeling tasks.

## Huggining face loading

In [ ]:
if HUGGINGFACE == True:
  raw_datasets =  load_dataset("medalpaca/medical_meadow_medical_flashcards", download_mode="force_redownload")
  raw_dataframe = raw_datasets['train'].to_pandas()
  raw_dataframe.to_csv('MedFlashCards.csv', index=False)


## Drive loading

In [ ]:
if not HUGGINGFACE:
  file_dataframe = pd.read_csv('MedFlashCards.csv')
  print(type(file_dataframe))
  print(file_dataframe.head())

# Dataset description, Data preprocessing and Feature extraction

In the upcoming lines of code, we'll conduct data preprocessing tasks and explore the dataset to improve its quality and usability. This preprocessing is vital for facilitating data visualization, exploration, and subsequent analysis.

The dataset is organized into a dataframe with columns representing 'question' and 'answer' pairs. For each pair, we'll augment the dataframe with additional columns describing key features such as length and the preprocessed text after removing punctuation, stopwords, and applying lemmatization.

These preprocessing steps enable us to analyze the data more effectively, uncovering insights about its structure and regularities. Through statistical analysis, we can gain a deeper understanding of the dataset, paving the way for more informed analysis and decision-making.

## Number of questions and answers

In the upcoming code lines, our primary focus lies in comprehending the dataset's composition by examing thr quantity of questions and answers.
We will then proceed to detect and eliminat any empty or invalid entries within the dataset. This meticulous process ensures the dataset's intefrity and enhances its quality for subsequent analyses

In [ ]:
dataset_length = len(file_dataframe)
number_empty_entries = file_dataframe.isnull().sum()
total_number_of_empty_entries = number_empty_entries.sum()
file_dataframe.dropna(inplace=True)
dataset_length_no_empty_entries = len(file_dataframe)
number_of_questions = len(file_dataframe['input'])
number_of_answers = len(file_dataframe['output'])

In [ ]:
plt.figure(figsize=(10, 5), dpi=125)
plt.pie([total_number_of_empty_entries, (dataset_length-total_number_of_empty_entries)], labels=['Empty entries', 'Valid entries'], autopct='%1.1f%%')
plt.show()

In [ ]:
print("Number of total question and answer pairs: ",dataset_length )
print("Number of total question and answer pairs without empty entries: ",dataset_length_no_empty_entries )
print("\n")
print("Number of total questions: ",number_of_questions )
print("Number of total answers: ",number_of_questions )



## Number of words per question

The code block below is dedicated to severl key tasks:


1.   Text preprocessing of the questions, which involves removing punctuation, stopwords, and performing lemmatization. The processed text will be stored in the datarfame containing the questions

2.   Calcultion of various statistics related to the number of words per question such as maximum word count and length, based on the results obtained from the preprocessing step

3.   Visualization of the distribution of the number of words per question through two histograms and kernal density estimates. One histogram depicts the distribution using the rw form of the data, while the other represents the distribution after preprocessing (excluding stopwords and punctuation). These visualizations provide valuable insights into the characteristics of the dataset


In [ ]:
question_df = pd.DataFrame()
question_df['question'] = file_dataframe['input']

In [ ]:
# Number of words per question
question_df['length'] = question_df['question'].apply(lambda question: len(question.split(' ')))
question_df['noPunctuation'] = question_df['question'].apply(lambda question: preprocess_document(question.lower().split(),noPunctuation=True))
question_df['noStopWords'] = question_df['question'].apply(lambda question: preprocess_document(question.lower().split(),noPunctuation=True, noStopWords=True))
question_df['lemmatized'] = question_df['question'].apply(lambda question: preprocess_document(question.lower().split(),noPunctuation=True, lemmatization=True))
question_df['noStopWords-lemmatized'] = question_df['question'].apply(lambda question: preprocess_document(question.lower().split(),noPunctuation=True, lemmatization=True, noStopWords=True))

In [ ]:
print("Maximum number of words per question: ", max(question_df['length']))
print("Minimum number of words per question: ", min(question_df['length']))
print("Average number of words per question: ", np.mean(question_df['length']))
print("Maximum number of words per question without punctuation and stopwords: ", max(question_df['noStopWords'].apply(lambda question: len(question))))
print("Minimum number of words per question without punctuation and stopwords: ", min(question_df['noStopWords'].apply(lambda question: len(question))))
print("Average number of words per question without punctuation and stopwords: ", np.mean(question_df['noStopWords'].apply(lambda question: len(question))))

In [ ]:
plt.figure(figsize=(10, 5), dpi=125)
sns.histplot(question_df['length'], bins = max(question_df['length']), kde=True, label='with punctuation and stopwords')
sns.histplot(question_df['noStopWords'].apply(lambda question: len(question)), bins=max(question_df['length']), kde=True, label='without punctuation and stopwords')
plt.xlabel('Number of words')
plt.ylabel('Number of questions')
plt.title('Number of words per question')
plt.legend(loc='upper right')
plt.show()


In [ ]:
question_df

## Number of words per answer

The subsequent code block is dedicated to the following key objectives:


1.   Conducting text preprocessing on the answers by eliminating punctuation, stopwords, and performing lemmatiazion. The processed text will be intefrated into the dataframe containing the answers

2.   Computing various statistics related to the word count per answer, including maximum, minimum, average word count and length, based on the outcomes of the preprocessing phase

3.   Visualizing the distribution of the word count per answer through two histograms and kernel density estimates. One histogram illustrates the distribution using the original form of the data, while the other showcases the distribution post-preprocessing(excluding stopwords and punctuation). These visualization serve as valuable tools for gaining insights into the dataset's underlying patterns and features


In [ ]:
answers_df = pd.DataFrame()
answers_df['answer'] = file_dataframe['output']

In [ ]:
# Number of words per answers
answers_df['length'] = answers_df['answer'].apply(lambda answer: len(answer.split(' ')))
answers_df['noPunctuation'] = answers_df['answer'].apply(lambda answer: preprocess_document(answer.lower().split(),noPunctuation=True))
answers_df['noStopWords'] = answers_df['answer'].apply(lambda answer: preprocess_document(answer.lower().split(),noPunctuation=True, noStopWords=True))
answers_df['lemmatized'] = answers_df['answer'].apply(lambda answer: preprocess_document(answer.lower().split(),noPunctuation=True, lemmatization=True))
answers_df['noStopWords-lemmatized'] = answers_df['answer'].apply(lambda answer: preprocess_document(answer.lower().split(),noPunctuation=True, lemmatization=True, noStopWords=True))

In [ ]:
print("Maximum number of words per answer: ", max(answers_df['length']))
print("Minimum number of words per answer: ", min(answers_df['length']))
print("Average number of words per answer: ", np.mean(answers_df['length']))
print("Maximum number of words per answer without punctuation and stopwords: ", max(answers_df['noStopWords'].apply(lambda answer: len(answer))))
print("Minimum number of words per answer without punctuation and stopwords: ", min(answers_df['noStopWords'].apply(lambda answer: len(answer))))
print("Average number of words per answer without punctuation and stopwords: ", np.mean(answers_df['noStopWords'].apply(lambda answer: len(answer))))

In [ ]:
plt.figure(figsize=(10, 5), dpi=125)
sns.histplot(answers_df['length'], bins = max(answers_df['length']), kde=True, label='with punctuation and stopwords')
sns.histplot(answers_df['noStopWords'].apply(lambda answer: len(answer)), bins=max(answers_df['length']), kde=True, label='without punctuation and stopwords')
plt.xlabel('Number of words')
plt.ylabel('Number of answer')
plt.title('Number of words per answer')
plt.legend(loc='upper right')
plt.show()


## Vocabulary

Vocabulary analysis plays a crucial role in question answering systems for several reasons:



1.   **Understanding and Matching**:Through meticulous examination of the vocabulary, systems decipher the semantics of questions, pinpointing essential terms, phrases, and relationships. This deep comprehension facilitates precise matching with relevant answers, ensuring accurate responses to user queries.

2.  **Semantic Representation**: Vocabulary analysis enriches the semantic representation of questions and answers by identifying synonyms, related terms, and semantic parallels. This augmentation enhances the system's linguistic comprehension and retrieval capabilities.

3.  **Handling Variability**: The variability in vocabulary usage across questions necessitates adaptable mechanisms. Vocabulary analysis empowers systems to recognize diverse expressions of the same concept or intent, effectively addressing linguistic variability.

4.  **Filtering Noise**: Noise, such as stopwords and punctuation, hinders accurate comprehension of questions and answers. Vocabulary analysis effectively filters out these irrelevant elements, focusing solely on crucial content essential for comprehension and matching.

5.  **Detection of Biases**: Vocabulary analysis uncovers potential biases within the dataset, such as the over-representation of certain words or concepts. Recognizing these biases is critical as they could inadvertently influence the behavior and performance of our models.

6. **Guidance for Text Representation Methods**: Understanding the dataset's vocabulary guides the selection of appropriate text representation methods, such as word embeddings or transformers. It also assists in fine-tuning these methods to ensure optimal performance tailored to the dataset's characteristics.

In the forthcoming code block, vocabularies are extracted for both the set of questions and answers. Furthermore, vocabularies are computed individually for each question and answer, encompassing both raw and preprocessed data. Following this, histograms and kernel density estimates are plotted to showcase the distribution of word count within the vocabularies concerning the number of questions.
These visualizations proved valuable insights into the distribution patterns across the dataset's vocabularies

In [ ]:
question_df['Vocabolary'] = question_df['question'].apply(lambda question: preprocess_document(question.lower().split(),isSet=True, isSorted=True))
question_df['Vocabolary-noStopWords'] = question_df['question'].apply(lambda question: preprocess_document(question.lower().split(),noPunctuation=True, noStopWords=True,isSet=True, isSorted=True))

In [ ]:
plt.figure(figsize=(10, 5), dpi=125)
sns.histplot(question_df['Vocabolary'].apply(lambda question: len(question)), bins = max(question_df['length']), kde=True, label='with punctuation and stopwords')
sns.histplot(question_df['Vocabolary-noStopWords'].apply(lambda question: len(question)), bins=max(question_df['length']), kde=True, label='without punctuation and stopwords')
plt.xlabel('Number of words per in the vocabulary')
plt.ylabel('Number of question')
plt.title('Number of words per question')
plt.legend(loc='upper right')
plt.show()


In [ ]:
answers_df['Vocabolary'] = answers_df['answer'].apply(lambda answer: preprocess_document(answer.lower().split(),isSet=True, isSorted=True))
answers_df['Vocabolary-noStopWords'] = answers_df['answer'].apply(lambda answer: preprocess_document(answer.lower().split(),noPunctuation=True, noStopWords=True,isSet=True, isSorted=True))

In [ ]:
plt.figure(figsize=(10, 5), dpi=125)
sns.histplot(answers_df['Vocabolary'].apply(lambda answer: len(answer)), bins = max(answers_df['length']), kde=True, label='with punctuation and stopwords')
sns.histplot(answers_df['Vocabolary-noStopWords'].apply(lambda answer: len(answer)), bins=max(answers_df['length']), kde=True, label='without punctuation and stopwords')
plt.xlabel('Number of words in the vocabulary')
plt.ylabel('Number of answer')
plt.title('Number of words per answer')
plt.legend(loc='upper right')
plt.show()


In [ ]:
global_questions_vocabulary = create_global_vocabolary(question_df['Vocabolary'],isSorted=True, isSet=True)
global_questions_vocabulary_noStopwords_noPunctuation = create_global_vocabolary(question_df['Vocabolary-noStopWords'],isSorted=True, isSet=True)

global_answers_vocabulary = create_global_vocabolary(answers_df['Vocabolary'],isSorted=True, isSet=True)
global_answers_vocabulary_noStopwords_noPunctuation = create_global_vocabolary(answers_df['Vocabolary-noStopWords'],isSorted=True, isSet=True)

In [ ]:
print("Number of words in vocabolary of questions: ", len(global_questions_vocabulary))
print("Number of words in vocabolary of questions after removing punctuation and stopwords: ", len(global_questions_vocabulary_noStopwords_noPunctuation))
print("Number of words in vocabolary of answers: ", len(global_answers_vocabulary))
print("Number of words in vocabolary of answers after removing punctuation and stopwords: ", len(global_answers_vocabulary_noStopwords_noPunctuation))

## The Most Common 100 Words

In the upcoming code block, we showcase the 100 most common words for both the questions and the answers

In [ ]:
q = nltk.Counter(create_global_vocabolary(question_df['Vocabolary-noStopWords']))
print("The most common 100 words in the vocabolary of questions: \n", q.most_common(100))
a = nltk.Counter(create_global_vocabolary(answers_df['Vocabolary-noStopWords']))
print("The most common 100 words in the vocabolary of answers: \n", a.most_common(100))

## WordCloud

In the upcoming code block, we'll create word clouds to illustrate the most prevalent words found in both questions and the answers. Word clouds visually represent word frequency by displaying them in varying sizes: the more frequent a word appears in the text, the larger and more prominent it appears in the cloud. This visualization technique offers an intuitive insight into the mosto common words present.
Hence, word clouds are generated by analyzing the text to identify individual words, tallying their occurences, and then graphically rendering them with larger sizes attributed to more frequent words

### WordCloud for words in questions

In [ ]:
all_questions = ' '.join(question_df['question'])
all_questions_noPunctuation = preprocess_document(all_questions.lower().split(), noPunctuation=True)
all_questions_noStopWords = preprocess_document(all_questions_noPunctuation, noStopWords=True)

In [ ]:
# Create a WordCloud object
text = ' '.join(all_questions_noPunctuation)
wordcloud = WordCloud(width=800, height=400, background_color="white").generate(text)
# Plot the word cloud
plt.figure(figsize=(10, 5), dpi=125)
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title('Word Cloud for words in the questions')
plt.show()

### WordCloud for words in questions after removing stopwords

In [ ]:
# Create a WordCloud object
text = ' '.join(all_questions_noStopWords)
wordcloud = WordCloud(width=800, height=400, background_color="white").generate(text)
# Plot the word cloud
plt.figure(figsize=(10, 5), dpi=125)
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title('Word Cloud for words in the questions after removing stopwords')
plt.show()

### WordCloud for words in answers

In [ ]:
all_answers = ' '.join(answers_df['answer'])
all_answers_noPunctuation = preprocess_document(all_answers.lower().split(), noPunctuation=True)
all_answers_noStopWords = preprocess_document(all_answers_noPunctuation, noStopWords=True)

In [ ]:
# Create a WordCloud object
text = ' '.join(all_answers_noPunctuation)
wordcloud = WordCloud(width=800, height=400, background_color="white").generate(text)
# Plot the word cloud
plt.figure(figsize=(10, 5), dpi=125)
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title('Word Cloud for words in the answers')
plt.show()

### WordCloud for words in answers after removing stopwords

In [ ]:
text = ' '.join(all_answers_noStopWords)
wordcloud = WordCloud(width=800, height=400, background_color="white").generate(text)
# Plot the word cloud
plt.figure(figsize=(10, 5), dpi=125)
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title('Word Cloud for words in the answers after removing stopwords ')
plt.show()

## N-grams

N-grams are contiguos sequences of 'n' items (typically words) from a given text. For example, in a text, bigrams (2-grams) would be sequences of two words and trigrams (3-grams) would be sequences of three words. Analyzing n-grams is important because:


1.   **Contextual Insight**: They help in understanding the context and relationships between words in the text

2.   **Pattern Identification**: Common n-grams can reveal frequently occuring phrases and structures, which are useful for languafe modelling and improving text generation

3.   **Noise Reduction**: They assist in filtering out noise by focusing on meaningful word combinations rather than isolated words.




In [ ]:
nGrams(document=all_questions_noStopWords, size=2, NmostCommon=20)

In [ ]:
nGrams(document=all_answers_noStopWords, size=2, NmostCommon=20)

In [ ]:
nGrams(document=all_questions_noStopWords, size=3, NmostCommon=20)

In [ ]:
nGrams(document=all_answers_noStopWords, size=3, NmostCommon=20)

# Save dataset

This code snippet saves the current state of various dataframes to CSV files if the SAVE_DATASET flag is set to True. Each dataframe is saved without the index to keep the files clean and focused on the data.

In [ ]:
if SAVE_DATASET:
  file_dataframe.to_csv('pre_preprocessed_dataset.csv', index=False)
  question_df.to_csv('questions_dataframe.csv', index=False)
  answers_df.to_csv('answers_dataframe.csv', index=False)

# Clustering

#Clustering introduction
In this notebook, we conducted a comprehensive analysis using clustering methodologies of a medical dataset aimed at developing a question-answering model. Our analysis involved utilizing preprocessed data and conducting clustering and visualization techniques to gain insights into the structure and content of the dataset.



*   **Data Exploration and Preprocessing:**
We began by exploring the dataset's characteristics, including word counts, vocabulary richness, and distribution of words. Preprocessing steps such as removing punctuation, stopwords, and lemmatization were applied to standardize the text and improve model performance.
*   **Visualization and Analysis:**
We visualized the dataset using techniques like WordClouds, N-grams, and TF-IDF analysis to uncover common phrases, patterns, and significant terms. This allowed us to gain a deeper understanding of the dataset's content and identify prevalent medical themes.

# Import and install

In [ ]:
TEST_MODEL = True

In [ ]:
if TEST_MODEL:
  !git clone https://github.com/epfml/sent2vec.git
  %cd sent2vec
  !make
  !pip install .
  import sent2vec

In [ ]:
!pip install datasets

In [ ]:
import string
from datasets import load_dataset
import re
from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import LabelEncoder
import nltk
from nltk.corpus import stopwords
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
import altair as alt
import seaborn as sns
from gensim.models import Word2Vec
from google.colab import drive
import os
import csv
import numpy as np
import random
from nltk.stem import WordNetLemmatizer
from sklearn.cluster import KMeans
import time
from sklearn import metrics
from sklearn.cluster import KMeans
import time
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import MiniBatchKMeans

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

path = 'Colab Notebooks/NLP'

os.chdir(f'/content/drive/MyDrive/{path}')
os.getcwd()

In [ ]:
DATAS_PATH = '/content/drive/MyDrive/Colab Notebooks/NLP/project/Models/Datasets'

#Dataset import

In [ ]:
# For the clustering module we have used the pre-processed dataset from our preliminary analysis which has all the details such as
# input and output length, vocabulary and lemmatization
file_dataframe = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/NLP/NLP Project/Datasets/pre_preprocessed_dataset.csv')
print(type(file_dataframe))

In [ ]:
# The below is out preprocessed dataframe

file_dataframe.head()

In [ ]:
input, output = file_dataframe["input"], file_dataframe["output"]

In [ ]:
documents = input + " " + output

# Documents vectorisation

In order to cluster the documents, we need to first convert them into a vector format. We will use the `TfidfVectorizer` from Scikit-Learn to do this.

The `TfidfVectorizer` is very similar to the `CountVectorizer` we used in the text classification tutorial except that it multiplies the term frequency in the document by the inverse document frequecy of the term across the corpus: $\mathrm{tf}(t, d) \cdot \mathrm{idf}(t)$
- here $\mathrm{tf}(t, d)$ is the count of the term $t$ in the document $d$
- idf is inverse document frequency: $\mathrm{idf}(t) = \log{\frac{n + 1}{\mathrm{df}(t) + 1}} + 1$
- $n$ is the number of documents in collection
- and the document frequency, $\mathrm{df}(t)$ is the number of documents that contain the term $t$
- see https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html

The use of idf-weighting can be justified in term of information theory:
- the amount of **information** associated with a term quantifies the **amount of surprise** at seeing a term in a document
- the surprise decreases with its (prior) probability of occurrence and information must be additive, so we have:
  - $I(t) = \log{\frac{1}{P(t)}} = -\log{P(t)}$
- the probability of observing a particular term in a document is just the percentage of documents that contain the term:
  - $P(t) = \frac{\mathrm{df}(t)}{n}$
- We smooth this estimate so that small df(t) values don't cause unreasonably high idf values, we have:
  - $P(t) = \frac{\mathrm{df}(t)+1}{n+1}$
  - so $I(t) = \log{\frac{n+1}{\mathrm{df}(t)+1}}$
- The idf value used is then $\mathrm{idf}(t) = I(t) + 1$, where the $+1$ prevents idf from going to zero as df approaches $n$.

In [ ]:
vectorizer = TfidfVectorizer(max_df=0.8, min_df=5, stop_words='english', )

The vectorisation method takes a number of arguments that control the resulting vocabulary. We have set the following arguments:  
- **max_df = 0.8**: remove words occuring in more than half of the documents (note: this will get rid of any corpus-specific tags)
- **min_df = 5**: remove words occuring in less than 5 documents  
- **stop_words = 'english'**: remove stopwords using an english stopword list

We can now fit the vectorizer to the data:
- Note: we could transform the data at the same time, using the `fit_transform()` method, but we'll wait for now to transform the data

In [ ]:
vectorizer.fit(documents)

Let's have a quick look at the vocabulary. How big is it?

In [ ]:
vocab = vectorizer.get_feature_names_out()

print(f"Length of vocabulary: {len(vocab)}")

Wow, that's quite big!

Let's print out a random sample of 100 terms from it to see what they look like:

In [ ]:
sorted(random.sample(vocab.tolist(),100))

Most of the terms look pretty good, but the representation looks reasonable.

Now let's vectorize the dataset:

In [ ]:
vector_outputs = vectorizer.transform(output)

Here is the sparse vector for the first document:


In [ ]:
print(vector_outputs[0])

The decimial values are the TF-IDF scores for the terms. We can sort the terms by their TF-IDF values, and print them out as follows:

In [ ]:
sorted([(vocab[j], vector_outputs[0, j]) for j in vector_outputs[0].nonzero()[1]], key=lambda x: -x[1])

Do the top terms agree with what you expected for the document?

Print it out below:

In [ ]:
print(output[0])

## Data exploration

Let's play a bit with our data

### Measuring the similarity between vectorised documents

The vectorizer also normalizes the resulting document representations such that their vectors have length one.
- We can see this by computing the dot-product between a vector representation and itself.
- For example, for the first document in the collection we have:

In [ ]:
vec = vector_outputs[0]
vec.multiply(vec).sum()

To calculate the dot-product we multiplied the sparse vector by itself and then took the sum.

The fact that the vectors have length one (almost length of one due to approximations in the representation) means that the dot-product between vectors computes the cosine of the angle between them.
- The cosine of the angle between tf-idf vectors provides a value in the range [0,1], that is often used to measure the similarity between documents.
- Let's compute the similarity between the first two documents in the collection:

In [ ]:
vector_outputs[0].multiply(vector_outputs[1]).sum()

Here is the second document:

In [ ]:
print(output[1])

The similarity value is zero, which isn't surprising since most documents don't share vocabulary.

The average vocabulary size of a document in the collection is:

In [ ]:
nonzero_count = vector_outputs.count_nonzero()
doc_count = vector_outputs.get_shape()[0]

print(f"Average document vocabulary size: {nonzero_count/doc_count}")

The first two answers came from different categories:

In [ ]:
print(f"1st answer: {output[0]}")
print(f"2st answer: {output[1]}")

What if we try to find all the documents similar to the first one? Let's see:

In [ ]:
for i in range(len(output)):
  if vector_outputs[0].multiply(vector_outputs[i]).sum() >= 0.4:
    print(i, ': ', output[i])

### Searching the collection based on keywords

We could even use the same approach to compute the similarity between a search query and each of the documents in a collection in order to find the one that best matches with a query:

In [ ]:
import numpy as np

#query = 'pregnancy'
#query = 'heart attack'
query = 'PAP test'

query_vec = vectorizer.transform([query])[0]

index = np.argmax([query_vec.multiply(vector_outputs[i]).sum() for i in range(len(output))])
print(documents[index])
print("Similarity: ", query_vec.multiply(vector_outputs[index]).sum())

In [ ]:
vector_inputs = vectorizer.transform(input)

In [ ]:
similarity_matrix_io = np.dot(vector_inputs, vector_outputs.T)
similarity_matrix_ii = np.dot(vector_inputs, vector_inputs.T)
similarity_matrix_oo = np.dot(vector_outputs, vector_outputs.T)

In [ ]:
for i in range(similarity_matrix_io.shape[1]):
  if similarity_matrix_io[0, i] >= 0.4:
    print(i, ': ', output[i])

##Similarity Matrices

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
#@title question - answer similarity matrix
fig, ax = plt.subplots(figsize=(100, 100))

max_val = 100

intersection_matrix = similarity_matrix_io[:max_val, :max_val].todense()

ax.matshow(intersection_matrix, cmap=plt.cm.Blues)

for i in range(max_val):
    for j in range(max_val):
        c = np.round(intersection_matrix[j,i], 2)
        ax.text(i, j, str(c), va='center', ha='center')

In the similarity matrix generated from the initial 100 question-answer pairs, a notable trend emerges: there's a significant similarity between each question and its corresponding answer. This implies a close relationship between the question and its accurate response. Furthermore, some answers exhibit considerable similarity to different questions, suggesting that these questions may be about the same topics or fields.

In [ ]:
fig, ax = plt.subplots(figsize=(100, 100))
#@title question - question similarity matrix

max_val = 100

intersection_matrix = similarity_matrix_ii[:max_val, :max_val].todense()

ax.matshow(intersection_matrix, cmap=plt.cm.Blues)

for i in range(max_val):
    for j in range(max_val):
        c = np.round(intersection_matrix[j,i], 2)
        ax.text(i, j, str(c), va='center', ha='center')

In the similarity matrix derived from the initial 100 question-question pairs, it's notable that each question naturally shows high similarity with itself. However, the more interesting observation arises from the fact that without shuffling the dataset beforehand, similar questions consistently cluster together. This suggests that within the dataset, there are several questions per topic, and those related to the same topics tend to be grouped closely together.

In [ ]:
fig, ax = plt.subplots(figsize=(100, 100))

max_val = 100
#@title answer - answer similarity matrix


intersection_matrix = similarity_matrix_oo[:max_val, :max_val].todense()

ax.matshow(intersection_matrix, cmap=plt.cm.Blues)

for i in range(max_val):
    for j in range(max_val):
        c = np.round(intersection_matrix[j,i], 2)
        ax.text(i, j, str(c), va='center', ha='center')

In the similarity matrix derived from the initial 100 answer-answer pairs, a similar pattern emerges as observed in the analysis of the first 100 question-question pairs. However, this time, the clusters of answers with high similarity to themselves are even more pronounced.

# Clustering

Now we can start applying clustering to our vectorised documents

### Selecting a value for k

When used on low dimensional data, k-Means is often combined with the 'elbow method' (https://en.wikipedia.org/wiki/Elbow_method_(clustering)) for finding the 'right' number of clusters k

The method involves:
- running the clustering algorithm with increasing values of *k*
- plotting the intrinsic evaluation measure (within-cluster sum-of-squares)
- and looking for a point in which improvement in the measure decreases substantially from one time step to the next.

Let's try out the method using the MiniBatch version of k-Means since it is a bit faster to run.
- First generate the performance evaluation measure values across the range of k values:

In [ ]:
if True:
  performance_inertia = []
  performance_labels = []
  sum_time = 0
  k_values = range(10,200, 10)
  for k in k_values:
    print(f'Evaluation with k={k}')
    start =time.time()
    mini_batch = MiniBatchKMeans(n_clusters=k, batch_size=500, random_state=2307, n_init=2).fit(vector_outputs)
    end = time.time()
    execution_time = round(end-start, 2)
    sum_time += execution_time
    i = k_values.index(k)+1
    estimated_remaining_time = round((sum_time/i)*(len(k_values)-i), 2)
    performance_inertia.append(mini_batch.inertia_)
    performance_labels.append(mini_batch.labels_)
    print(f'Inertia:{mini_batch.inertia_}\nExecution time:{execution_time}s\nEstimated remaining time:{estimated_remaining_time}s\n\n')

Note that the within-cluster sum-of-squares almost always improves from one iteration to the next as k is increased.
- In theory it should always increase since the more cluster centroids there are, the more flexibility the model has for describing datapoints (assigning them to clusters)
- but in practice the stochasticity of k-Means and also the mini-batch procedure can cause the algorithm to not find the global optimum and thus produce a higher sum-of-squares value than a previous iteration (with lower k).

We'll now use some standard code to plot the performance measure against the value k:

In [ ]:
plt.figure()
plt.plot(k_values, performance_inertia)
plt.ylabel('Within-cluster sum-of-squares')
plt.xlabel('k')
plt.show()

Does it look to you like there is a point on the graph where performance suddenly stops getting a lot better?
- I don't see one ...
- That is likely because we have (i) very high dimensional data and (ii) quite a large number of documents.

So, we can try to analyze the silhouette score in the same range and look into the best scores to find a goof value of k

In [ ]:
from tqdm import tqdm
silhouette_scores = []
for labels in tqdm(performance_labels):
  silhouette_scores.append(metrics.silhouette_score(vector_outputs, labels))

In [ ]:
plt.figure()
plt.plot(k_values, silhouette_scores)
plt.ylabel('Silhouette_scores')
plt.xlabel('k')
plt.show()

In [ ]:
k = k_values[np.argmax(silhouette_scores)]
print(f'The value of K with the highest silhouette_scores is {k}')

### Clustering with k-Means

Now that we have computed a vector representation, we can perform the clustering.

There are many different clustering algorithms in common use, from k-Means and Hierarchical clustering to DBScan and Spectral clustering.  
- Many of them are implemented in Scikit-learn: https://scikit-learn.org/stable/modules/clustering.html
- You can easily change the code below to try them out.

We will use the k-Means algorithm since it is very popular, fast, scalable and relatively robust.
- k-Means models the dataset using k circlular clusters
- where each cluster is represented by the centroid of the datapoints it contains.

To apply k-Means we need to decide in advance how many clusters to look for.
- We will set the number of clusters to be exactly the number of categories in the dataset to see if the clustering can recover the original groups from the data.

In [ ]:
k = 10

kmeans = KMeans(n_clusters=k, max_iter=100, verbose=True, random_state=2307, n_init=2)
kmeans.fit(vector_outputs)


The inertia values are actually the *sum of squared distances between each sample to its closest cluster centroid*
- This is the measure that k-Means seeks to minimise
- Note: It is not the cosine distance between document and the cluster center (which is what we would like to minimise), but given that each vector has length one, the squared Euclidean distance is similar to it.  



### Clustering evaluation

Once we have run our clustering algorithm, we can

#### Investigating the Clusters

The clustering routine has now produced clusters in a very high dimensional feature space (with tens of thousands of dimansions).
- We can't plot such high dimensional data to see whether the clusters look coherent or not.
- Instead we will need to investigate the coherency of the clusters by looking at the terms occuring in them.

We can see the important terms for each cluster by inspecting the centroid vector for the cluster.
- Let's have a look at the terms with high weights in the centroid of the first cluster:

In [ ]:
# Get the centroid for the first cluster
centroid = kmeans.cluster_centers_[0]

# Sort terms according to their weights
# (argsort goes from lowest to highest, we reverse the order through slicing)
sorted_terms = centroid.argsort()[::-1]

# Print out the top 10 terms for the cluster
[vocab[j] for j in sorted_terms[:20]]

The top 10 terms look pretty consistent.

We could use our dot-product search trick from before to find the document in the collection that is the best exemplar of the cluster:

In [ ]:
index = np.argmax([np.dot(centroid,vec) for vec in vector_outputs.toarray()])

print(documents[index])

We now know what terms were important for defining cluster 0 and have seen a representative document,
- but still don't know how many documents were assigned to to the cluster.
- Let's find out:

In [ ]:
sum(kmeans.labels_ == 0)

We can see which clusters the first 10 documents have been assigned to:

In [ ]:
for i in range(10):
    print(f"document {i} is in cluster {kmeans.labels_[i]}")

Let's now repeat the process (algorithmically) to get the top terms for all the clusters.
- In the code below, we iterate over the clusters, getting the centroid for each, sort the values and printing out the top terms:

In [ ]:
print("Top terms per cluster:")
vocab = vectorizer.get_feature_names_out()

for i in range(kmeans.n_clusters):
    centroid = kmeans.cluster_centers_[i]
    sorted_terms = centroid.argsort()[::-1]
    print(f"Cluster {i}:\t{[vocab[j] for j in sorted_terms[:10]]}")

That's interesting.
- It looks like each of the clusters contains relatively consistent terms.

How many documents have been assigned to each cluster?

In [ ]:
print('Number of docs in: ')
number_of_elements = []
for i in range(kmeans.n_clusters):
    n_docs = np.sum(kmeans.labels_ == i)
    print(f"Cluster {i}: {n_docs}")
    number_of_elements.append(n_docs)

In [ ]:
plt.figure(figsize=(10, 5), dpi=125)
sns.histplot(number_of_elements, bins = k, kde=True)
plt.xlabel('Number of elemets')
plt.ylabel('Number of cluster')
plt.title('Number of elements per cluster')
plt.show()

Looks like there is a very large cluster corresponding to terms that might be more common across the corpus. Mostly, though, the counts are pretty well distributed across the clusters.

#### Quantitative Evaluation of Clustering Results

Let's now move on to the task of quantitatively evaluating the clustering algorithm. There are two ways to evaluate the results of a clustering algorithm:
- **Intrinsic evaluation** - If the ground truth is not known, you could use:
    - Within-cluster sum-of-squares: that is the *inertia* of the K-Means clustering
    - [Silhouette](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html#sklearn.metrics.silhouette_score) - "a higher Silhouette Coefficient score relates to a model with better defined clusters"
    
Let's first have a look at the *intrinsic measures*
- print out within-cluster sum-of-squares and the silhouette coefficient:


There are many possible metrics for clustering evaluation: some can be used when the ground truth labels are known, some can be used when the true labels are unknown.



In [ ]:
print("Intrinsic evaluation measures:")
print("Within-cluster sum-of-squares:", str(kmeans.inertia_))
print("Silhouette coefficient:", str(metrics.silhouette_score(vector_outputs, kmeans.labels_)))

The intrinsic evaluation measures are hard to interpret, since their values depend heavily on the difficulty of the clustering task, the amount of data, etc.
- Moreover, the sum-of-squares value will improve monotonically as the number of clusters is increased.


### Minibatch K-means clustering

MiniBatch k-Means is an approximate verions of the k-Means that is designed to scale up to massive datasets by making use of small samples (minibatches) in order to find the k centroids.
- It should be faster to run than k-Means. Let's give it a try:

In [ ]:
mb_kmeans = MiniBatchKMeans(n_clusters=k, batch_size=500, random_state=2307)
mb_kmeans.fit(vector_outputs)

Evaluate the minibatch clustering algorithm to see if the performance compares with the original k-Means on this data:

In [ ]:
print("Intrinsic evaluation measures:")
print("Within-cluster sum-of-squares:", str(mb_kmeans.inertia_))
print("Silhouette coefficient:", str(metrics.silhouette_score(vector_outputs, mb_kmeans.labels_)))

# Visualisation

Often visualisng the samples and the model output is useful to understand what's going on

### Problems visualising high dimensional data

For a bit of fun, we'll try to transform the high dimensional data into low dimensional data (just 3 dimensions) using a linear dimensionality reduction technique called Singular Value Decomposition.
- We'll transform the `vector_documents` to be 3 dimensional.

In [ ]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(3)
reduced_data = svd.fit_transform(vector_outputs)

[x,y,z] = np.transpose(reduced_data)
[x,y,z]

We can plot the data coloured according to the clusters found by the original k-Means algorithm:

In [ ]:
fig = plt.figure(figsize=(15, 10))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(x, y, z, c=kmeans.labels_, marker='.');

# Conclusion from clustering for Answers:

The clustering analysis reveals the main topics within the dataset, with terms consistently associated with specific medical themes, such as **vitamin deficiency**, **cancer types**, **thyroid conditions**, **heart issues**, **symptoms and treatments**, **kidney functions**, and **syndromes**.

Despite the comprehensive clustering, the **silhouette coefficient** is very **low (0.003886)**, indicating that the clusters might not be well-separated and suggesting a need for further refinement or exploration of other clustering techniques to achieve better-defined groupings.

The distribution of documents across clusters highlights varying concentrations, with some clusters (e.g., Cluster 6) containing significantly more documents than others, suggesting prevalent themes within the dataset.

**Term Frequency-Inverse Document Frequency (TF-IDF) Analysis:**
We analyzed the TF-IDF scores for terms within the clusters to identify the most significant terms. For example, terms such as **"levels"** and **"vitamin"** in **Cluster 0** have high TF-IDF scores, indicating their importance in the documents within that cluster. This approach helped us in understanding the key topics and terms that are characteristic of each cluster.

**Cluster Composition and Overlap:**
The cluster composition reveals that certain medical themes dominate the dataset. **Cluster 6**, with the highest number of documents **(15776)**, suggests a predominant focus on patient-related discussions and common diseases. On the other hand, clusters like **Cluster 2** (**330 documents**) focus on more specialized topics such as **thyroid conditions**, indicating less frequent but highly specific content.

**Evaluating Optimal Number of Clusters (k):**
We conducted an evaluation of MiniBatch K-Means clustering over a range of cluster numbers (k) from 10 to 190. The within-cluster sum-of-squares (inertia) decreases as k increases, but the silhouette coefficient remains low throughout. This suggests that while increasing the number of clusters might slightly improve cohesion within clusters, it does not necessarily enhance separation between them.

**Computational Performance:**
The computational performance analysis of MiniBatch K-Means clustering showed varying execution times with different values of k. The execution time generally increases with the number of clusters, reflecting the higher computational complexity. For example, k=10 took 0.44s, whereas k=190 took 17.44s. This information is crucial for optimizing the trade-off between clustering accuracy and computational efficiency.

#Context

#Clustering with contexts

In this section we have used the datasets which we generated using our context retrieval algorithm as the hugging face dataset did not have the context.

In [ ]:
file_dataframe = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/NLP/NLP Project/Datasets/questions_context_dataframe.csv')
print(type(file_dataframe))

In [ ]:
file_dataframe.head()

In [ ]:
contexts = file_dataframe["contexts"]

##Context vectorization

We processesed a set of text contexts by first converting them into TF-IDF vectors using TfidfVectorizer.

Then, we evaluated the optimal number of clusters for K-Means clustering by iterating over different k values.

Futhermore, we computed the inertia (within-cluster sum-of-squares) and silhouette scores for each k value, aiding in the assessment of cluster quality.

Finally, we identified the k value with the highest silhouette score as the optimal number of clusters for the given text data.

In [ ]:
vectorizer = TfidfVectorizer(max_df=0.8, min_df=5, stop_words='english', )

In [ ]:
vectorizer.fit(contexts)

In [ ]:
vocab = vectorizer.get_feature_names_out()

print(f"Length of vocabulary: {len(vocab)}")

It's a lot bigger than using only the answers

In [ ]:
vector_contexts = vectorizer.transform(contexts)

In [ ]:
sorted([(vocab[j], vector_contexts[0, j]) for j in vector_contexts[0].nonzero()[1]], key=lambda x: -x[1])

We can try to compare the top words of the first context with the question and the answer to see if they are relevant

In [ ]:
print("Input: ", input[0], "\n")
print("Output: ", output[0], "\n")
print("Context: ", contexts[0])

Mg2, Ca2 and PTH are stricly connected with both the answer and the question

##Selecting a value of k

We evaluated the performance of MiniBatch K-Means clustering over a range of cluster numbers (k) from 10 to 190.

Futhermore, we measured and printed the inertia and execution time for each value of k and estimate the remaining time for the evaluations.

Finally, stored the results for further analysis to determine the optimal number of clusters.

In [ ]:
if True:
  performance_inertia = []
  performance_labels = []
  sum_time = 0
  k_values = range(10,200, 10)
  for k in k_values:
    print(f'Evaluation with k={k}')
    start =time.time()
    mini_batch = MiniBatchKMeans(n_clusters=k, batch_size=500, random_state=2307, n_init=2).fit(vector_contexts)
    end = time.time()
    execution_time = round(end-start, 2)
    sum_time += execution_time
    i = k_values.index(k)+1
    estimated_remaining_time = round((sum_time/i)*(len(k_values)-i), 2)
    performance_inertia.append(mini_batch.inertia_)
    performance_labels.append(mini_batch.labels_)
    print(f'Inertia:{mini_batch.inertia_}\nExecution time:{execution_time}s\nEstimated remaining time:{estimated_remaining_time}s\n\n')

In [ ]:
plt.figure()
plt.plot(k_values, performance_inertia)
plt.ylabel('Within-cluster sum-of-squares')
plt.xlabel('k')
plt.show()

From the graph it is impossible to understand the right value of k so we calculate the silhouette scores for the same k values

In [ ]:
from tqdm import tqdm
silhouette_scores = []
for labels in tqdm(performance_labels):
  silhouette_scores.append(metrics.silhouette_score(vector_contexts, labels))

In [ ]:
plt.figure()
plt.plot(k_values, silhouette_scores)
plt.ylabel('Silhouette_scores')
plt.xlabel('k')
plt.show()

In [ ]:
k = k_values[np.argmax(silhouette_scores)]
print(f'The value of K with the highest silhouette_scores is {k}')

##K-means clustering

We performed K-Means clustering to group similar text documents together. Here's what we learned from this analysis:

**Cluster Themes:** We identified 10 distinct groups (clusters) of text documents. Each cluster has a set of keywords that best represent the common themes or topics found within the documents in that group.

**Understanding Each Group:** We can see the top 10 most important words for each cluster. This helped us understand the core ideas that define each group of documents.

**Finding Representative Documents:** We found a document that closely resembles the central point (centroid) of the first cluster. This document served as a good example of the typical text found within that cluster.

**Distribution of Documents:** We explored how many documents belong to each cluster. We then visualized this distribution using a histogram. This helps us see how many documents fall into each group size category.

**Cluster Quality Check:** We performed some calculations to assess the quality of the clustering. These calculations tell us how tightly documents within each group resemble each other.

In short, this analysis helped us understand how the text documents can be grouped based on their content. We can see the key themes for each group, how many documents fall into each group, and how well the documents within each group are clustered together.

In [ ]:
k = 10

kmeans = KMeans(n_clusters=k, max_iter=100, verbose=True, random_state=2307, n_init=2)
kmeans.fit(vector_contexts)

In [ ]:
# Get the centroid for the first cluster
centroid = kmeans.cluster_centers_[0]

# Sort terms according to their weights
# (argsort goes from lowest to highest, we reverse the order through slicing)
sorted_terms = centroid.argsort()[::-1]

# Print out the top 10 terms for the cluster
[vocab[j] for j in sorted_terms[:20]]

In [ ]:
index = np.argmax([np.dot(centroid,vec) for vec in vector_contexts.toarray()])

print(contexts[index])

In [ ]:
print("Top terms per cluster:")
vocab = vectorizer.get_feature_names_out()

for i in range(kmeans.n_clusters):
    centroid = kmeans.cluster_centers_[i]
    sorted_terms = centroid.argsort()[::-1]
    print(f"Cluster {i}:\t{[vocab[j] for j in sorted_terms[:10]]}")

In [ ]:
print('Number of docs in: ')
number_of_elements = []
for i in range(kmeans.n_clusters):
    n_docs = np.sum(kmeans.labels_ == i)
    print(f"Cluster {i}: {n_docs}")
    number_of_elements.append(n_docs)

In [ ]:
plt.figure(figsize=(10, 5), dpi=125)
sns.histplot(number_of_elements, bins = k, kde=True)
plt.xlabel('Number of elemets')
plt.ylabel('Number of cluster')
plt.title('Number of elements per cluster')
plt.show()

In [ ]:
print("Intrinsic evaluation measures:")
print("Within-cluster sum-of-squares:", str(kmeans.inertia_))
print("Silhouette coefficient:", str(metrics.silhouette_score(vector_contexts, kmeans.labels_)))

# Conclusion from clustering for context:

Our clustering analysis reveals distinct themes in the dataset, including viral infections, case reports, syndromes, biological studies, treatments, cancer research, pulmonary diseases, epidemiological studies, cardiac conditions, and cellular biology.

The largest clusters are focused on biological studies and treatments, indicating a significant emphasis in these areas.


**Cluster Size Distribution:**



*   **Observation:** Cluster sizes vary significantly, with the largest cluster (Cluster 3) containing 12,542 documents and the smallest (Cluster 6) containing 742 documents.
*   **Implication:** This suggests that some topics are much more prevalent in the dataset. For instance, biological studies and treatments are heavily represented, indicating these are major areas of focus or interest in the dataset.

**Dominant Themes:**



*   **Clusters with Common Themes:** Multiple clusters (e.g., Clusters 1, 2, and 4) relate to patient care and treatment, though they differ in focus (case reports, clinical syndromes, and therapies).
*   **Implication:** There's a strong emphasis on clinical aspects of medicine, showing a potential overlap in topics, which might suggest a need for more granular clustering.

**Term Overlap:**

*   **Observation:** Some terms like "patients" and "case" appear in multiple clusters, indicating common topics across different clusters.
*   **Implication:** The dataset might benefit from a hierarchical clustering approach to better differentiate between overlapping themes.


**Intrinsic Evaluation Measures:**


*   **Within-cluster sum-of-squares (Inertia):** 32764.73
*   **Silhouette Coefficient:** 0.00296


The high inertia value suggests some clusters are not very compact, and the low silhouette coefficient indicates that the clusters are not well-separated.
This could be because we have a closed-domain dataset (medical one) covering a wide range of highly specilized topics.  To achieve a high level of internal cohesion within each cluster, we would need a large number of clusters, ensuring that each cluster contrains only closely related elements.




# Word Embedding

# Word Embeddings Introduction

Word embeddings are a type of word representation that allows words to be expressed as vectors in a continuous vector space. These vectors capture semantic menaings and relationshps between words bases on their usage in large text of corpora

## Import and Package installation

Let's install some important packages and libraries

In [ ]:
!pip install --upgrade gensim
!pip install scikit-learn
!pip install matplotlib
!pip install datasets
!pip install plotly

In [ ]:
!pip install fasttext

Let's import some useful libraries

In [ ]:
from google.colab import drive
import os
from gensim.models.word2vec import Word2Vec
import gensim.downloader as api
from gensim.models import FastText, Word2Vec, KeyedVectors
import string
import numpy as np
import pandas as pd
from pandas.core.common import flatten
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from datasets import load_dataset
import re
import csv
import random
import plotly.express as px
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import fasttext
import seaborn as sns
from scipy.spatial.distance import pdist, squareform

## Setup

In [ ]:
drive.mount('/content/drive')
os.chdir(f'/content/drive/MyDrive/Colab Notebooks/NLP/Assignment/datasets')
os.getcwd()

Set to true **HUGGINGFACE** to download the dataset directly from HuggingFace

In [ ]:
HUGGINGFACE = False

In [ ]:
nltk.download('wordnet')

In [ ]:
nltk.download('punkt')

## Utils

Define a dictionary of contractions and their expansions

In [ ]:
contractions_dict = {
    "can't": "cannot",
    "won't": "will not",
    "I'm": "I am",
    "he's": "he is",
    "she's": "she is",
    "it's": "it is",
    "they're": "they are",
    "we're": "we are",
    "you're": "you are",
    "isn't": "is not",
    "aren't": "are not",
    "wasn't": "was not",
    "weren't": "were not",
    "don't": "do not",
    "doesn't": "does not",
    "didn't": "did not",
    "haven't": "have not",
    "hasn't": "has not",
    "hadn't": "had not",
    "I'll": "I will",
    "you'll": "you will",
    "he'll": "he will",
    "she'll": "she will",
    "we'll": "we will",
    "they'll": "they will",
    "I'd": "I would",
    "you'd": "you would",
    "he'd": "he would",
    "she'd": "she would",
    "we'd": "we would",
    "they'd": "they would",
    "there's": "there is",
    "here's": "here is",
    "how's": "how is",
    "let's": "let us",
    "that's": "that is",
    "what's": "what is",
    "who's": "who is",
    "where's": "where is",
    "why's": "why is",

}

The **expand_contractions** function is designed to expand contractions in a given text based on a provided dictionary of contractions and their corresponding expanded forms.

In [ ]:
def expand_contractions(text, contractions_dict=contractions_dict):
    contractions_pattern = re.compile('({})'.format('|'.join(contractions_dict.keys())),
                                      flags=re.IGNORECASE | re.DOTALL)

    def replace(match):
        match_text = match.group(0)
        expanded_text = contractions_dict.get(match_text.lower())
        return expanded_text

    expanded_text = contractions_pattern.sub(replace, text)
    return expanded_text

The function **preprocess_document** is designed to process a given document, performing various optional text preprocessing steps based on the specified parameters. These parameters include:

* expand: If set to True, the function expands contractions in the document (e.g., "can't" to "cannot").
* isSorted: If set to True, the function sorts the processed document alphabetically.
* isSet: If set to True, the function converts the processed document into a set, removing any duplicate elements.
* noStopWords: If set to True, the function removes common English stopwords from the processed document.
* lemmatization: If set to True, the function lemmatizes the words in the processed document using WordNet.

The function begins by expand contractions, removing punctuation from the document using a regular expression. Then, depending on the parameters provided, it applies additional processing steps accordingly. Finally, it returns the processed document.

In [ ]:
def preprocess_document(document, expand=False, noPunctuation=False, isSorted=False, isSet=False, noStopWords=False, lemmatization=False):
    newDocument = document
    if expand:
        newDocument = [expand_contractions(word) for word in newDocument]
    if noPunctuation:
        regex = '[' + string.punctuation + ']'
        newDocument = [re.sub(regex, '', item) for item in newDocument]
    if isSet:
        newDocument = set(newDocument)
    if noStopWords:
        newDocument = [w for w in newDocument if w not in stopwords.words('english')]
    if isSorted:
        newDocument = sorted(newDocument)
    if lemmatization:
        lemmatizer = WordNetLemmatizer()
        newDocument = [lemmatizer.lemmatize(w) for w in newDocument]
    return ' '.join(newDocument)

The upcoming snippet of code demonstrates how to use vector arithmetci with a pre-trained word embedding model to find terms semantically similar to a given concept given the vocabulary of the embedding.
This technique leverages the semantic relationships captured by the word embeddings to explore and understand the conceptual adjustments within the embedding space

In [ ]:
def adjust_term_vector(model,term, term_to_adjust_from, term_to_adjust_to, topN=10):
  term_vec = model.get_vector(term)
  adjust_from_vec = model.get_vector(term_to_adjust_from)
  adjust_to_vec = model.get_vector(term_to_adjust_to)
  adjusted_vec = term_vec + (adjust_from_vec - adjust_to_vec)
  closest_terms = model.most_similar(adjusted_vec, topn=topN)
  return closest_terms

In [ ]:
def plot_PCA_decomposition(model, words):
  words_vec = np.array([model[word] for word in words])
  pca = PCA(n_components=2)
  words_vec_2d = pca.fit_transform(words_vec)
  plt.figure(figsize=(7, 7))
  plt.scatter(words_vec_2d[:, 0], words_vec_2d[:, 1], edgecolors='k', c='g')
  for word, (x, y) in zip(words, words_vec_2d):
      plt.text(x, y, word, fontsize=12)
  plt.show()

# Dataset loading

**Basic Description of the Dataset and its Design**

The dataset in our study was derived from the Anki Medical Curriculum flashcards, a comprehensice collection curated and continually updated by medical students. Thes flashcards cover the full breadth of the medcial curriculum, encompassing subjects suche as anatomy, physiology, pathology and pharmacology. Each flashcard typically includes summaries to facilitate the learning and retention of key medical concepts. To create the dataset, the content was extracted from these flashcards, excluding those containing images.
Then, OpenAI's GPT 3.5- turbo was employed to transfrom the flashcard content into coherent, contextually relevant question-answer pairs.

The dataset consists of over 30,000 questions, each accompanied by a corresponding segment of text serving as its answer. Notably, these answers lack any additional context.



**Dataset Loading**

We initiate dataset download with this code snippet, offering flexibility in data retrieval. Depending on the value of the HUGGINGFACE flag, the script fetches the dataset from Hugging Face's repository or directly loads it from a CSV file stored on Google Drive. This versatile approach simplifies the process of acquiring the necessary data for analysis or modeling tasks.

## Huggining face loading

In [ ]:
if HUGGINGFACE == True:
  raw_datasets =  load_dataset("medalpaca/medical_meadow_medical_flashcards", download_mode="force_redownload")
  raw_dataframe = raw_datasets['train'].to_pandas()
  raw_dataframe.to_csv('MedFlashCards.csv', index=False)


## Drive loading

In [ ]:
if HUGGINGFACE == False:
  medFlashCards_dataframe = pd.read_csv('MedFlashCards.csv')

question_df = pd.read_csv('questions_dataframe.csv')
answer_df = pd.read_csv('answers_dataframe.csv')


# Learning word embeddings

## Data Preprocessing

Data preprocessing is crucial before training word embeddings because it ensures the text data is clean, consistent, and meaningful. Here's why each operation is important:


*   **Remove Punctuation and Special Characters**: Punctuation and special characters do not carry meaningful semantic information for word embeddings and can introduce noise. Removing them helps focus on the actual words and their relationships

*   **Expand Contractions**: Contractions like "can't" or "won't" are expanded to "cannot" and "will not" to standardize the text. This ensures that the model treats them as the same words rather than different tokens, improving the quality of embeddings

*  **Lowercase Words**: Converting all words to lowercase standardizes the text, so "Apple" and "apple" are treated as the same word. This reduces the vocabulary size and improves the embeddings' consistency.

*  **Lemmatization**: Lemmatization reduces words to their base or root form, such as converting "running" to "run." This helps in grouping similar words together, ensuring that the embeddings capture the core meaning rather than different forms of the same word.

By performing these preprocessing steps, the text data becomes cleaner and more uniform, allowing the word embeddings to capture more accurate and meaningful representations of the words.

In [ ]:
train_questions = question_df['question'].apply(lambda question: preprocess_document(question.lower().split(), noPunctuation=True, expand=True, lemmatization=True ))
train_answers = answer_df['answer'].apply(lambda answer: preprocess_document(answer.lower().split(), noPunctuation=True, expand=True, lemmatization=True ))

First, we will train word embeddings using the Word2Vec implementation from the Gensim package

In [ ]:
train_questions = list(flatten(train_questions))
train_answers = list(flatten(train_answers))

## Tokenization


In the upcoming code block, the tokenization process is performed. Tokenization involves splitting text into smaller units called tokens, which can be words, subwords, or characters. For word embeddings, this typically means breaking down a text into individual words or subwords that can then be converted into vectors.

In [ ]:
train_questions_tokenized = [word_tokenize(question) for question in train_questions]
train_answers_tokenized = [word_tokenize(answer) for answer in train_answers]
tokenized_sentences = train_questions_tokenized + train_answers_tokenized

## Word2Vec

Word2Vec is a method for generating word embeddings by training a shallow neural network on a large corpus of text data. It learns dense vector representations of words based on their context, capturing semantic relationships between words. These embeddings are widely used in NLP tasks to measure semantic similarity and improve performance in various applications



Finally we have the data in the right format for training Word2Vec, so we can provide it to the algorithm. For parameters, we set:
- the embedding size to be 30,
- the minimum count for any vocabulary term to be 5
- the size of the context window to 10.

In [ ]:
model = Word2Vec(tokenized_sentences, vector_size=30, min_count=5, window=10)

Let's see how big the vocabulary is that Word2Vec ended up using, i.e. how many word vectors did it learn:

In [ ]:
num_vectors = len(model.wv)
print('Number of learned vectors: ', num_vectors)

### Inspecting embeddings and finding similar words

Let's print out one of the vectors to see what looks like now that we have a word2vec model trained:

In [ ]:
term = 'pregnancy'
model.wv[term]

We could now use the model to compute similarities between terms based on their cosine distance in the embedding space.


In [ ]:
term = 'heart'
model.wv.most_similar(term)

In [ ]:
term = 'virus'
model.wv.most_similar(term)

In [ ]:
term = 'antidepressant'
model.wv.most_similar(term)

In [ ]:
term = 'insulin'
model.wv.most_similar(term)


The following snippet of code shows how a word's meaning shifts based on an analogy. Imagine insulin is the base word. We want its meaning to move away from diabetes and closer to pneumonia. The code uses a fancy method (PCA) to create a visual map where closer words have similar meaning

In [ ]:
term_vec = 'insulin'
adjust_from_vec = 'pneumonia'
adjust_to_vec = 'diabetes'
closest_terms = adjust_term_vector(model.wv,term_vec, adjust_from_vec, adjust_to_vec)
words = [word[0] for word in closest_terms] + [term_vec, adjust_from_vec, adjust_to_vec]
plot_PCA_decomposition(model.wv, words)

### Visualising the embedding vectors using t-SNE

We'll now visualise some of the word vectors in a 3 dimensional space using t-SNE.
The vocabulary of word vectors is quite large (around 25,000). Giving them all to t-SNE will cause it to take far too long to converge. So let's first choose a random subset of 500 terms to show.

In [ ]:
sample = random.sample(list(model.wv.key_to_index), 500)
print(sample)

Now we'll get the word vectors for the sampled terms:

In [ ]:
word_vectors = model.wv[sample]
word_vectors

And we'll provide the vectors to TSNE to fit a model and transform the data to 3 dimensions:

In [ ]:
tsne = TSNE(n_components=3, n_iter=2000)
tsne_embedding = tsne.fit_transform(word_vectors)

Now transform the data into 3 columns (for x, y, and z):

In [ ]:
x, y, z = np.transpose(tsne_embedding)

And generate the 3d plot:

In [ ]:
fig = px.scatter_3d(x=x, y=y, z=z)
fig.update_traces(marker=dict(size=3,line=dict(width=2)))
fig.show()

Well that's a not a particularly interesting 3d plot!
- How about we label some of the points on the graph to see what words they correspond to:

In [ ]:
fig = px.scatter_3d(x=x[:200],y=y[:200],z=z[:200],text=sample[:200])
fig.update_traces(marker=dict(size=3,line=dict(width=2)),textfont_size=10)
fig.show()

Let's extend the random set of terms with a set of colours to see if they cluster:

In [ ]:
# Add some specific terms to sample:
colours = ['red','green','blue','orange','yellow','purple','pink','cream','brown','black','white','gray']

word_vectors = model.wv[colours+sample]

tsne = TSNE(n_components=3)
tsne_embedding = tsne.fit_transform(word_vectors)

x, y, z = np.transpose(tsne_embedding)

In [ ]:
import plotly.express as px

r = (-200,200)
fig = px.scatter_3d(x=x, y=y, z=z, range_x=r, range_y=r, range_z=r, text=colours + [None] * 500)
fig.update_traces(marker=dict(size=3,line=dict(width=2)),textfont_size=10)
fig.show()

Note: t-SNE is a stochastic algorithm, so run it a couple of times to see how the visualisation changes.

Have a play around with the visualisation to see whether other sets of terms cluster together.

# Loading Pre-trained Embedding

## Glove-wiki-Gigaword-200

Using GloVe embedddings pre-trained on the Wikipedia 2014 + Gigaword 5 dataset (glove-wiki-gigaword-200) can enhance the performance of question answering systems on our dataset. These embeddings are trained on an extensive corpus, capturing deep semantic and synactic nuances. This pretraining provides a wealth of linguistic knowledge. With 200 dimensions, GloVe embeddings achieve a balance between capturing intricate word relationships and maintaning computational efficiency.
Additionally, employing pretrained embeddings accelerates the training process, as the model starts with rich word representations instead of learning them from scratch. Consequently, glove-wiki-gigaword-200 is an effective, efficient, and practical choice for enhancing question-answering systems across various datasets.

**GloVe-Wiki-Gigaword 200 overview**:


*   **Algorithm**: GloVe which captures global word-word co-occurence statistics
*   **Training Data**: Wikipedia and Gigaword corpus
*   **Vector dimension**: 200-dimensional embeddings




In [ ]:
model_wiki = api.load('glove-wiki-gigaword-200')

Let's see how big the vocabulary:

In [ ]:
print('Vocabulary size of Wikipedia model: ', len(model_wiki))

### Inspecting embeddings and finding similar words

We could now use the model to compute similarities between terms based on their cosine distance in the embedding space

In [ ]:
term = 'virus'
model_wiki.most_similar(term)

This code demonstrates how a word's meaning shifts based on analogies using vector arithmetic, such as "king" to "queen" and "man" to "woman." While a pretrained model on a large open-domain corpus can effectively capture complex word relationships, it often struggles with domain-specific analogies, such as "insulin" to "antibiotics" and "diabetes" to "pneumonia." In the following sections, we will explore how using a domain-specific pretrained model can enhance our performance in these specialized contexts.

In [ ]:
term_vec = 'king'
adjust_from_vec = 'woman'
adjust_to_vec = 'man'
closest_terms = adjust_term_vector(model_wiki,term_vec, adjust_from_vec, adjust_to_vec)
words = [word[0] for word in closest_terms] + [term_vec, adjust_from_vec, adjust_to_vec]
plot_PCA_decomposition(model_wiki, words)

In [ ]:
term_vec = 'insulin'
adjust_from_vec = 'pneumonia'
adjust_to_vec = 'diabetes'
closest_terms = adjust_term_vector(model_wiki,term_vec, adjust_from_vec, adjust_to_vec)
words = [word[0] for word in closest_terms] + [term_vec, adjust_from_vec, adjust_to_vec]
plot_PCA_decomposition(model_wiki, words)

## Word2Vec - PMC - 100

Word embeddings create vector representations of text data, but not all embeddings effectively capture clinical, medical, and biomedical information. These embeddings provide a low-dimensional representation of semantic meaning, facilitating efficient text data analysis. While they are useful across various domains, their application to clinical, medical, and biomedical encounter notes presents several challenges. Commonly available pre-trained embeddings are often derived from broad, non-medical text sources, which may not accurately capture the specific word senses, linguistic relationships, or vocabulary needed in a clinical context. To address this, a training corpus based on clinical case reports from the PubMed Open Access subset is used. Consequently, a medical and biomedical pre-trained embedding model was employed.

**Word2Vec-PMC-Open-Access-All-Manuscripts overview**:


*   **Algorithm**: Word2Vec
*   **Training Data**: PMC Open Access All Manuscripts
*   **Vector dimension**: 100-dimensional embeddings

First download and extract the files from each archive

In [ ]:
!tar -xvf w2v_100d_oa_all.tar.gz

In [ ]:
# Load the model
model = Word2Vec.load('W2V_100/w2v_oa_all_100d.bin')

Let's see how big the vocabulary:

In [ ]:
print(f'Size of the vocabulary: {len(model.wv)}')

### Inspecting embeddings and finding similar words

Return 100-dimensional vector representations of each word

In [ ]:
term = 'diabetes'
model.wv.get_vector(term)

In [ ]:
term = 'heart'
model.wv.get_vector(term)

In [ ]:
term = 'pregnancy'
model.wv.get_vector(term)

In [ ]:
model.wv.similarity('myocardial_infarction', 'heart_attack')

We could now use the model to compute similarities between terms based on their cosine distance in the embedding space.


In [ ]:
term = 'delirium'
model.wv.most_similar(term)

In [ ]:
term = 'heart'
model.wv.most_similar(term)

The upcoming snippet of code demonstrates how to use vector arithmetic for word's meaning shifts based on analogies with a pre-trained word embedding model - Word2Vec on PMC Open Access All Manuscripts - to find terms semantically similar to a given concept in the context of medical literature.
This technique leverages the semantic relationships captured by the word embeddings to explore and understand the conceptual adjustments within the embedding space

In [ ]:
term_vec = 'insulin'
adjust_from_vec = 'pneumonia'
adjust_to_vec = 'diabetes'
closest_terms = adjust_term_vector(model.wv,term_vec, adjust_from_vec, adjust_to_vec)
words = [word[0] for word in closest_terms] + [term_vec, adjust_from_vec, adjust_to_vec]
plot_PCA_decomposition(model.wv, words)

## FastText - Common Crawl + Wikipedia - 300

fasttext is an extension of the Word2Vec model developed by Facebook AI Research. Unlike traditional Word2Vec, which considers words as atomic units, fasttext trates words as a bags of character n-grams. This approach allows fasttext to generate embeddings for out-of-vocabulary words by aggregating embeddings of their character n-grams, enabling it to handle rare or misspelled words effectively.

In addition to word embeddings, fasttext also supports sentence embeddings. It achieves this by averaging the word embeddings within a sentence to create a fixed-size vector representation of the entire sentence.

**fasttext word vectors for 157 languages overview**:

**Algorithm**: fasttext

**Training Data**: Wikipedia + Commn Crawl

**Vector Dimension**: 300 - dimension embedding

Let's download the model

In [ ]:
# !wget https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.en.zip
!wget http://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz
!gzip -d cc.en.300.bin.gz

Let's load the model

In [ ]:
ft_model = fasttext.load_model('cc.en.300.bin')

Let's see how big the vocabulary:

In [ ]:
print(f'Size of the vocabulary: {len(ft_model.get_words())}')

### Inspecting embeddings and finding similar words

Return 300-dimensional vector representations of each word

In [ ]:
term = 'diabetes'
ft_model.get_word_vector(term)

In [ ]:
term = 'heart'
ft_model.get_word_vector(term)

In [ ]:
term = 'pregnancy'
ft_model.get_word_vector(term)

We could now use the model to compute similarities between terms based on their cosine distance in the embedding space.


In [ ]:
term = 'delirium'
ft_model.get_nearest_neighbors(term)

In [ ]:
term = 'heart'
ft_model.get_nearest_neighbors(term)

The upcoming snippet of code demonstrates how to use vector arithmetic for word's meaning shifts based on analogies with a pre-trained word embedding model

In [ ]:
term_vec = 'king'
adjust_from_vec = 'woman'
adjust_to_vec = 'man'

closest_terms = ft_model.get_analogies(adjust_from_vec, adjust_to_vec, term_vec)
closest_terms

Let's combine the retrived words with the ones used to retrive them

In [ ]:
words = [word[1] for word in closest_terms] + [term_vec, adjust_from_vec, adjust_to_vec]

In [ ]:
words

Let's plot the embeddings in the 2D space using PCA

In [ ]:
words_vec = np.array([ft_model.get_word_vector(word) for word in words])
pca = PCA(n_components=2)
words_vec_2d = pca.fit_transform(words_vec)
plt.figure(figsize=(7, 7))
plt.scatter(words_vec_2d[:, 0], words_vec_2d[:, 1], edgecolors='k', c='g')
for word, (x, y) in zip(words, words_vec_2d):
    plt.text(x, y, word, fontsize=12)
plt.show()

In [ ]:
term_vec = 'insulin'
adjust_from_vec = 'pneumonia'
adjust_to_vec = 'diabetes'

closest_terms = ft_model.get_analogies(adjust_from_vec,adjust_to_vec,term_vec)

In [ ]:
words = [word[1] for word in closest_terms] + [term_vec, adjust_from_vec, adjust_to_vec]

In [ ]:
words

In [ ]:
words_vec = np.array([ft_model.get_word_vector(word) for word in words])
pca = PCA(n_components=2)
words_vec_2d = pca.fit_transform(words_vec)
plt.figure(figsize=(7, 7))
plt.scatter(words_vec_2d[:, 0], words_vec_2d[:, 1], edgecolors='k', c='g')
for word, (x, y) in zip(words, words_vec_2d):
    plt.text(x, y, word, fontsize=12)
plt.show()

In the upcoming line of code, we compute the embeddings for the first 100 questions of the dataset using the FastText model

In [ ]:
word_vectors_questions = np.array([ft_model.get_word_vector(doc) for doc in question_df['question'][:100] ])

Let's proceed to compute the distance matrix of the vector embeddings for the first 100 questions. Subsequently, we'll visualize the similarity matrix using a heatmap. Heatmapes depicting distance metrics offer valuable insights into the similarities between elements and can aid in rudimentary clustering, identifying similar elements within the dataset

In [ ]:
distance_matrix = squareform(pdist(word_vectors_questions, metric='cosine'))
plt.figure(figsize=(100, 100))
sns.heatmap(1 - distance_matrix[:100, :100], annot=True, cmap='coolwarm', square=True)
plt.show()

The same can be done for the first 100 answers

In [ ]:
word_vectors_answers = np.array([ft_model.get_word_vector(doc) for doc in answer_df['answer'][:100] ])

In [ ]:
distance_matrix = squareform(pdist(word_vectors_answers, metric='cosine'))
plt.figure(figsize=(100, 100))
sns.heatmap(1 - distance_matrix[:100, :100], annot=True, cmap='coolwarm', square=True)
plt.show()

By inspecting the heatmap and manually examing the dataset, it becomes evident that there are many similar questions within the dataset. The heatmap reveals clusters of  closely related answers through regions of similar colors, indicating high similarity, Moreover, performing clustering on the dataset could further confirm and identify these groups of similar questions more systematically

 While this approach may not be as sophisticated as more advanced clustering algorithms, it can provide useful initial insights into the structure of the data and potential groupings of similar elements.

##Conclusions

The analysis clearly demonstrates that utilizing pre-trained word embedding models leads to enhanced performance. One compelling analysis underscoring this advantage is the word analogy test (within the medical domain). It vividly illutrastes that pre-trained models talilored to specific domains consistently outperform their conterparts

# Context Retrival

# Context Retrival Introduction

**Introduction to Contextual Question Answering**

In the development of question-answering systems, having a context for each question is often crucial. Typically, models are trained to understand and retrieve answers within a given context. However, our dataset contains only question and answers without any additional context. This presents a unique challenge: how can we train an effective question-answering model without inherent contextual information ?

**Importance of Context**

Context is often vital because it provides the backgrounf information needed for understanding and accurately answering. Without context, the model may struggle to comprehend the nuances and specifics of the questions, leasinf to lower accuracy and reliablity

**Challenges without context**



1.   **Ambiguity**: question cab be ambiguous without context
2.   **Relevance**: without context, it is challenging to determine which information is relevant to the question
3.   **Depth of understanding**: context allows the model to provide more comprehensive and detailed answers

**Proposed Solutions**


1. **Using External Sources for Context Retrieval**:

 To address the absence of context in our dataset, we propose a method that involves retrieving relevant context from external sources, specifically leveraging the PubMed API to access medical and biomedical papers and their abstracts. Here's an overview of our approach:


 -  **Keyword Extraction**: for each question, we employ keyBERT, a pre-trained mdoel, to extract a set of keywords that capture the essence of the question. These keywords serve as the query terms for retrieving relevant papers and their contexts

 - **Context Retrival**: using the extracted keywords as queries, we utilize the PubMedAPI to search through a vast repository of medical literature. This step aims to retrieve potential contexts from papers and their abstracts, ensuring that the context is domain-specific and directly relevant to the question at hand

 - **Preprocessing and Cleaning**: the retrieved contexts undergo preprocessing and cleaning to ensure they are in the appropriate format for futher analysis. This preprocessing involves removing irrelevant information, normalizing text, and ensuring consistency across retrieved contexts.

 - **Context Ranking**: to prioritize the retrieved contexts, we employ various ranking techniques:

    - **TF-IDF and BM25 Index Search**: These methods assess the importance of terms based on their frequencies within the retrieved contexts and the question itself. This allows us to build a document search engine that matches contexts to a given query (question).

    - **Document vectorization**: utilizing a TD-IDF vectorizer, we compute the textual documents into numerical vectors based on the relevancy of the words and how often they appear in the documents, enabling quantitative comparison.

    - **Sentence Embedding and Cosine Similarity**: We compute sentence embeddings for both the question and the retrieved contexts, then evaluate the cosine similarities between them to obtain similarity scores. This process allows us to rank the contexts based on their relevance to the question. Unlike the previous methods that only consider term frequency, this approach captures the semantic meaning and relationships between words within the sentences.

The last method indeed yielded the best results, as it effectively maintains the semantic relationships between words. This capability enchances the matching process, enabling a more accurate alignment of the question with the context. By considering not only the frequency of terms but also their semantic meaning, this methos provides a more  nuanced understanding of the relationship between the question and the retrieved contexts. This ensures that the generated responses are not only relevant in terms of topic but also in terms of undrlying semantic context, resulting in more precise and informative answers



2.  **Fine-Tuning Pre-Trained Models**

Another solution is to leverage pre-trained models and fine-tune them for our question-answering task. This approach takes advantage of the vast knowledge these models have already acquired during their training


3. **Leveraging Open-Domain Pre-Trained Models**

In addition to domain-specific models, we can also consider open-domain pre-trained models. Literature shows that models trained on vast, open-domain knowledge have performed well on question-answering tasks after fine-tuning.

**Our Work**

In the forthcoming notebook, we will delve into the process of retrieving external context for each question in our dataset. This endeavor will empower us to construct a comprehensive context for every question, a crucial asset in training models that rely on contextual information for optimal performance.

## Import and Install

Let's install some useful libraries

In [ ]:
!pip install -q python-terrier

In [ ]:
!pip install datasets

In [ ]:
!pip install biopython

In [ ]:
!pip install keybert

In [ ]:
!pip install gensim

In [ ]:
!git clone https://github.com/epfml/sent2vec.git
%cd sent2vec
!make
!pip install .

Let's import some useful libraries

In [ ]:
from google.colab import drive
import os

from datasets import load_dataset
import numpy as np
import pandas as pd
import csv

from collections import Counter

import matplotlib.pyplot as plt

import re
import string
from string import punctuation

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

import pyterrier as pt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import euclidean_distances

from keybert import KeyBERT

from gensim.models import Word2Vec
from gensim.models.doc2vec import Doc2Vec, TaggedDocument


from Bio import Entrez

from sentence_transformers import SentenceTransformer

import sent2vec

## Setup

In [ ]:
drive.mount('/content/drive')
os.chdir(f'/content/drive/MyDrive/Colab Notebooks/NLP/Assignment/datasets') #Teka's path
#os.chdir(f'/content/drive/MyDrive/Colab Notebooks/NLP/NLP Project/Datasets')# Alessandro's path
os.getcwd()

Variables designated as constants and flags are employed to selectively execute specific segments of the code.

In [ ]:
HUGGINGFACE = False
SAVE_DATASET = True
SET_RANDOM_INDEX = False

Load models and packages

In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

In [ ]:
kw_model = KeyBERT()

In [ ]:
import pyterrier as pt
if not pt.started():
  pt.init()

## Utils

The function **preprocess_document** is designed to process a given document, performing various optional text preprocessing steps based on the specified parameters. These parameters include:

* isSorted: If set to True, the function sorts the processed document alphabetically.
* isSet: If set to True, the function converts the processed document into a set, removing any duplicate elements.
* noStopWords: If set to True, the function removes common English stopwords from the processed document.
* lemmatization: If set to True, the function lemmatizes the words in the processed document using WordNet.

The function begins by removing punctuation from the document using a regular expression. Then, depending on the parameters provided, it applies additional processing steps accordingly. Finally, it returns the processed document.

In [ ]:
def preprocess_document(document,noPunctuation=False ,isSorted=False, isSet=False, noStopWords=False, lemmatization=False):
  newDocument = document
  if noPunctuation==True:
    regex = '[' + string.punctuation + ']'
    newDocument = [re.sub(regex,'',item) for item in newDocument]
  if isSet==True:
    newDocument = set(newDocument)
  if noStopWords==True:
    newDocument =  [w for w in newDocument if w not in stopwords.words('english')]
  if isSorted==True:
    newDocument = sorted(newDocument)
  if lemmatization==True:
    lemmatizer = WordNetLemmatizer()
    newDocument = [lemmatizer.lemmatize(w) for w in newDocument]
  return newDocument

The upcoming snippet of code defines three functions for interacting with the PubMed databaes:


1.   **search_pubmed(query, max_results=10)**: Searches PubMed using a specified query and returns a list of PubMed IDs (PMIDs) for the retrieved results, with a default maximum of 10 results.

2.   **fetch_abstracts(pubmed_ids)**: Fetches the abstracts corresponding to a list of PubMed IDs and returns them as text.

3.   **fetch_abstracts_and_summaries(pubmed_ids)**: Retrieves summaries for the PubMed records identified by a list of PubMed IDs and returns them as structured data.


In [ ]:
def search_pubmed(query, max_results=10):
    Entrez.email = "SemanticSurgeons@gmail.com"
    handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results)
    results = Entrez.read(handle)
    handle.close()
    return results["IdList"]

def fetch_abstracts(pubmed_ids):
    Entrez.email = "SemanticSurgeons@gmail.com"
    handle = Entrez.efetch(db="pubmed", id=pubmed_ids, rettype="abstract", retmode="text")
    abstracts = handle.read()
    handle.close()
    return abstracts

def fetch_abstracts_and_summaries(pubmed_ids):
    Entrez.email = "SemanticSurgeons@gmail.com"
    summaries = Entrez.esummary(db="pubmed", id=','.join(pubmed_ids))
    summary_records = Entrez.read(summaries)
    summaries.close()
    return summary_records

The following function, **create_query**, generates a query based on the input question. If the question length exceeds a specified threshold (queryLength), the function extracts keywords using a keyword extraction model and constructs a query using these keywords. Otherwise, it returns the original question

In [ ]:
def create_query(question,queryLength):
  if len(question) > queryLength:
    query = []
    keywords = kw_model.extract_keywords(question, keyphrase_ngram_range=(1, 1), stop_words='english',top_n=queryLength)
    for j in range(len(keywords)):
      key = keywords[j][0]
      query.append(key)
    query = ' '.join(query)
    return query
  else:
    return question

In [ ]:
def most_similar(doc_id,similarity_matrix,matrix):
    print (f'Document: {question_documents_train_set[doc_id]}')
    print ('\n')
    print (f'Similar Documents using {matrix}:')
    if matrix=='Cosine Similarity':
        similar_ix=np.argsort(similarity_matrix[doc_id])[::-1]
        print(similar_ix)
    elif matrix=='Euclidean Distance':
        similar_ix=np.argsort(similarity_matrix[doc_id])
    for ix in similar_ix:
        if ix==doc_id:
            continue
        print('\n')
        print (f'Document: {question_documents_train_set[ix]}')
        print (f'{matrix} : {similarity_matrix[doc_id][ix]}')

# Dataset Loading and preprocessing

We'll begin by loading and preprocessing the dataset to ensure it's in the appropriate format for the context retrieval phase.

In [ ]:
medFlashCards_df = pd.read_csv('MedFlashCards.csv')
medFlashCards_df.dropna(inplace=True)
medFlashCards_df

In [ ]:
questions_df = pd.DataFrame(medFlashCards_df['input'])
answers_df = pd.DataFrame(medFlashCards_df['output'])

In [ ]:
questions_df['Cleaned Questions'] = questions_df["input"].apply(lambda question: ' '.join(preprocess_document(question.split(), noPunctuation=True,lemmatization=True )) )

# Context Retrival

In [ ]:
if SET_RANDOM_INDEX == True:
  random_question_index = np.random.randint(0, len(medFlashCards_df)-1)
else:
  random_question_index = 1460

In [ ]:
question = questions_df['Cleaned Questions'][random_question_index]
answer = answers_df['output'][random_question_index]

In [ ]:
print(f'Question: {question}')
print(f'Answers: {answer}')

We'll now generate the query for retrieving documents from the PubMed API. Considering a query length of 6, we'll use the create_query function to construct the query based on the provided question.

In [ ]:
query_length = 6
query = create_query(question, query_length)

The upcoming code segment starts by setting the maximum number of results to retrieve from a PubMed search and performs the search using the provided query. If the initial search yields no results, it retries the search using the last keyword from the query which is the least important

In [ ]:
max_results = 30  # Maximum number of results to retrieve
pubmed_ids = search_pubmed(query, max_results)
if not pubmed_ids:
    pubmed_ids = search_pubmed(query[:-1], max_results)

This code fetches abstracts for PubMed IDs and creates a DataFrame called "documents" with two columns: "docno" and "text," where each row represents an abstract with an incremented ID.

In [ ]:
abstracts = fetch_abstracts(pubmed_ids)
documents = pd.DataFrame(columns=["docno", "text"])
for idx, abstract in enumerate(abstracts.split("\n\n")):
  newEntry = {'docno': idx + 1, 'text': abstract}
  documents = documents._append(newEntry, ignore_index = True)

Let's preprocess the retrieved contexts to ensure they are in the optimal format for subsequent operations.

In [ ]:
documents['Cleaned Context'] = documents['text'].apply(lambda question: " ".join(preprocess_document(question.split(),noPunctuation=True, lemmatization=True)))

In [ ]:
documents

## Document Search Engine using Indexing

These methods assess the importance of terms based on their frequencies within the retrieved contexts and the question itself

The first step in order to apply any kind of search is to index the whole dataset of contexts

In [ ]:
indexer = pt.DFIndexer("./index_contexts", overwrite=True)
documents['docno'] = documents['docno'].astype(str)
index_ref = indexer.index(documents["Cleaned Context"],documents['docno'])
index_ref.toString()

In [ ]:
!ls -lh index_contexts/

Let's now load the index and print  some information about our collection

In [ ]:
index = pt.IndexFactory.of(index_ref)
print(index.getCollectionStatistics().toString())

### TF-IDF Index Search


This code segment initializes a TF-IDF-based batch retrieval system with a specified index and then executes a search using the provided query.

In [ ]:
tf_idf = pt.BatchRetrieve(index, wmodel="TF_IDF")
ranking_td_idf = tf_idf.search(query)
ranking_td_idf

This code snippet checks if the ranking list is not empty. If it's not empty, it retrieves the most relevant result and, if its length is less than 300 characters, also retrieves the second relevant result and concatenates them. If the ranking list is empty, it assigns 'NAN' to the most relevant result.

In [ ]:
ranking_length = len(ranking_td_idf)
if(ranking_length)!=0:
    most_relevant_result_docno = ranking_td_idf.loc[ranking_td_idf['rank'] == 0, 'docno'].values[0]
    most_relevant_result = documents.loc[documents['docno'] ==  most_relevant_result_docno, 'text'].values[0]

    if len(most_relevant_result) < 300:
      second_relevant_result_docno = ranking_td_idf.loc[ranking['rank'] == 1, 'docno'].values[0]
      second_relevant_result = documents.loc[documents['docno'] ==  most_relevant_result_docno, 'text'].values[0]
      most_relevant_result = most_relevant_result + ' ' + second_relevant_result

else:
      most_relevant_result = 'NAN'

In [ ]:
context_tfIdf = most_relevant_result
print(context_tfIdf)

### BM25 Index Search

This code segment initializes a BM25-based batch retrieval system with a specified index and then executes a search using the provided query.

In [ ]:
bm25 = pt.BatchRetrieve(index, wmodel="BM25")
ranking_bm25 = bm25.search(query)
ranking_bm25

In [ ]:
ranking_length = len(ranking_bm25)
if(ranking_length)!=0:
    most_relevant_result_docno = ranking_td_idf.loc[ranking_bm25['rank'] == 0, 'docno'].values[0]
    most_relevant_result = documents.loc[documents['docno'] ==  most_relevant_result_docno, 'text'].values[0]

    if len(most_relevant_result) < 300:
      second_relevant_result_docno = ranking_bm25.loc[ranking_bm25['rank'] == 1, 'docno'].values[0]
      second_relevant_result = documents.loc[documents['docno'] ==  most_relevant_result_docno, 'text'].values[0]
      most_relevant_result = most_relevant_result + ' ' + second_relevant_result

else:
      most_relevant_result = 'NAN'

In [ ]:
context_bm25 = most_relevant_result
print(context_bm25)

### Composition

This operator allow to re-rank the output of one retriever using a second retriever

In [ ]:
composition = tf_idf >> bm25
composition_ranking = composition.search(query)
composition_ranking

In [ ]:
if(len(composition_ranking))!=0:
    most_relevant_result_docno = composition_ranking.loc[composition_ranking['rank'] == 0, 'docno'].values[0]
    most_relevant_result = documents.loc[documents['docno'] ==  most_relevant_result_docno, 'text'].values[0]
    if len(most_relevant_result) < 300:
      second_relevant_result_docno = composition_ranking.loc[composition_ranking['rank'] == 1, 'docno'].values[0]
      second_relevant_result = documents.loc[documents['docno'] ==  most_relevant_result_docno, 'text'].values[0]
      most_relevant_result = most_relevant_result + ' ' + second_relevant_result
else:
      most_relevant_result = 'NAN'

In [ ]:
context_composition_tfIdf_bm25 = most_relevant_result
print(context_composition_tfIdf_bm25)

## Document Search Engine using Document Vectorization

Compute the textual documents into numerical vectors based on the relevancy of the words and how often they appear in the documents

Create lists of contexts and questions from the corresponding dataframes for building the train set

In [ ]:
contexts_list = documents['text'].tolist()
questions_list  = questions_df['input'].tolist()

In [ ]:
train_set = questions_list + contexts_list

This line fits the vectorizer to the training set, learning the vocabulary and the idf (inverse document frequency) values from the training data

In [ ]:
vectorizer = TfidfVectorizer(max_df=0.8, min_df=5, stop_words='english')

In [ ]:
vectorizer.fit(train_set)

In [ ]:
learned_vocabulary = vectorizer.get_feature_names_out()
print(f'Length of Vocabulary: {len(learned_vocabulary)}')

The upcoming lines of code transform the question and contexts list into their vector representations using the fitted vectorizer. This process converts the text data into numerical vectors based on the learned vocabulary and idf values

In [ ]:
vector_question = vectorizer.transform([question])
vector_contexts = vectorizer.transform(contexts_list)

Let's compute the similarities

In [ ]:
question_contexts_similarities = np.dot(vector_question, vector_contexts.T)

In [ ]:
best_matches = np.argmax(question_contexts_similarities, axis=1)
value = best_matches[0][0, 0]
print(f"The more similar context to question 0 is {best_matches[0]}")
print('CONTEXT:' ,contexts_list[value] )

## Document Search Engine using Document Embedding

A document search engine using embeddings transforms how search results are generated by focusing on the semantic meaning of text rather than just matching keywords. Choosing embeddings for this purpose offers significant advantages. Embeddings capture the semantic meaning of words, enabling more accurate and contextually relevant search results by understanding synonyms and polysemous words. They enhance contextual understanding, improve disambiguation, support scalable and efficient searches, and enable cross-lingual capabilitie

**Doc2Vec**

Doc2Vec is an extension of the Word2Vec model that learns continuous representations for pieces of text, such as sentences, paragraphs, or documents, while also capturing their semantic meanings. In Doc2Vec, each document is represented as a fixed-length vector, just like words are represented as vectors in Word2Vec.

The following code  prepares the data for training a Doc2Vec model by creating a list of tagged documents. Each document is tokenized into words, and a unique tag is assigned to it. The resulting tagged_data list contains TaggedDocument objects, where each object represents a document (question or context) with its corresponding tag.

In [ ]:
question_documents_train_set = [question] + documents['Cleaned Context'].tolist()
tagged_data = [TaggedDocument(words=word_tokenize(doc), tags=[i]) for i, doc in enumerate(question_documents_train_set)]

This code initializes a Doc2Vec model with a vector size of 100 and trains it for 100 epochs using the provided tagged data. During training, the model learns to generate document vectors that represent the semantic meaning of the input documents.

In [ ]:
model_d2v = Doc2Vec(vector_size=100,alpha=0.025, min_count=1)

model_d2v.build_vocab(tagged_data)

for epoch in range(100):
    model_d2v.train(tagged_data,
                total_examples=model_d2v.corpus_count,
                epochs=model_d2v.epochs)

Let's initialize an array to store document embeddings with dimensions corresponding to the number of documents in the dataset and a vector size of 100. Then, we are going  iterate over each document in the dataset, retrieve its corresponding Doc2Vec embedding from the trained model, and assign it to the corresponding position in the document_embeddings array.

In [ ]:
document_embeddings=np.zeros((documents.shape[0],100))

for i in range(len(document_embeddings)):
    document_embeddings[i]= model_d2v.docvecs[i]

Let's calculates pairwise similarities and differences between document embeddings using cosine similarity

In [ ]:
pairwise_similarities=cosine_similarity(document_embeddings)

Let's retrieve the most similar documents

In [ ]:
most_similar(0,pairwise_similarities,'Cosine Similarity')

**Sentence Transformer**


Sentence Transformers is a versatile toolkit designed for creating fixed-length numerical representations, or embeddings, of sentences and short texts. It harnesses the power of pre-trained transformer models like BERT, RoBERTa, or DistilBERT to encode input sentences into dense vector representations. These embeddings effectively capture the semantic meaning of the sentences, enabling a wide range of downstream tasks such as semantic similarity assessment, text classification, or clustering. In our case, we're utilizing the **all-mpnet-base-v2** variant, which is built on the MPNet architecture.

The upcoming  code initializes a Sentence Transformers model using the **all-mpnet-base-v2** variant, which is based on the MPNet architecture. The model is capable of generating high-quality sentence embeddings for a wide range of natural language processing task

In [ ]:
sbert_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

We'll employ the previously loaded model to create embeddings for a collection of documents containing both questions and answers, which together constitute the training set

In [ ]:
document_embeddings = sbert_model.encode(question_documents_train_set)

Let's calculates pairwise similarities and differences between document embeddings using cosine similarity

In [ ]:
pairwise_similarities=cosine_similarity(document_embeddings)
pairwise_differences=euclidean_distances(document_embeddings)

Let's retrieve the most similar documents

In [ ]:
most_similar(0,pairwise_similarities,'Cosine Similarity')

**BioSentVec**

BioSentVec, a pre-trained model, transforms sentences into numerical forms, effectively capturing the intricate connections between words within the realm of biology and medicine. Its training dataset includes a vast collection of over 30 million documents sourced from scholarly articles found in PubMed and clinical notes extracted from the MIMIC-III Clinical Database. This extensive training ensures its proficiency in understanding the language and concepts specific to these fields.

In [ ]:
model_BioSentVec_path = '/content/drive/MyDrive/Colab Notebooks/NLP/project/Models/Datasets/BioSentVec_PubMed_MIMICIII-bigram_d700.bin'
testing_model = sent2vec.Sent2vecModel()

Let's load the model

In [ ]:
try:
    testing_model.load_model(model_BioSentVec_path)
except Exception as e:
    print(e)
print('model successfully loaded')

We'll employ the previously loaded model to create embeddings for a collection of documents containing both questions and answers, which together constitute the training set

In [ ]:
document_embeddings = testing_model.embed_sentences(question_documents_train_set)

Let's calculates pairwise similarities and differences between document embeddings using cosine similarity

In [ ]:
pairwise_similarities = cosine_similarity(document_embeddings)

Let's retrieve the most similar documents

In [ ]:
most_similar(0,pairwise_similarities,'Cosine Similarity')

# Evaluation

The code cells above showcase three methods for documents retrival based on a given query. Each method successfully organizes the retrieved documents from most relevant to least relevant. It's evident that approaches solely relying on term frequency were surpassed by those utilizing sentence embeddings, which capture semantic relatioships of words within a sentence.
Additionaly, models pre-trained within our specific domain, such as BioSentVec, demonstrated superior accuracy


In [ ]:
tf_idf_bm_25 = "INTRODUCTION: Primary amoebic meningoencephalitis (PAM) is a rare disease caused\
by the free-living amoeba Naegleria fowleri. Infection occurs by insufflation of\
water containing amoebae into the nasal cavity, and is usually associated with\
bathing in freshwater. Nasal irrigation is a more rarely reported route of\
infection.\
CASE PRESENTATION: A fatal case of PAM in a previously healthy Norwegian woman,\
acquired during a holiday trip to Thailand, is described. Clinical findings were\
consistent with rapidly progressing meningoencephalitis. The cause of infection\
was discovered by chance, owing to the unexpected detection of N. fowleri DNA by\
a PCR assay targeting fungi. A conclusive diagnosis was established based on\
sequencing of N. fowleri DNA from brain biopsies, supported by histopathological\
findings. Nasal irrigation using contaminated tap water is suspected as the\
source of infection.\
CONCLUSION: The clinical presentation of PAM is very similar to severe bacterial\
meningitis. This case is a reminder that when standard investigations fail to\
identify a cause of infection in severe meningoencephalitis, it is of crucial\
importance to continue a broad search for a conclusive diagnosis. PAM should be\
considered as a diagnosis in patients with symptoms of severe\
meningoencephalitis returning from endemic areas."

tf_idf_vectorizer = "Fatal primary amoebic meningoencephalitis in a Norwegian tourist returning from\
Thailand."

doc2vec = " Author information 1National Center for Emerging and Zoonotic Infectious Diseases Centers\
 for Disease Control and Prevention1600 Clifton Road Atlanta GA 30329 USA jey9cdcgov"

sentence_transformer = "Balamuthia mandrillaris is an amoeba found in fresh water and soil that cause\
 granulomatous amoebic encephalitis We report herein an autopsy case of B mandrillaris amoebic\
  encephalitis which wa definitely diagnosed by PCR An 81yearold man who had Sjögrens syndrome\
   manifested drowsiness 2 month before his death with progressive deterioration Neuroimaging\
    demonstrated focus of T2 and fluidattenuated inversion recovery high and T1 lowintensity with\
     irregular postcontrast ring enhancement in the cerebral hemisphere thalamus and midbrain\
      Pathologically multiple hemorrhagic and necrotic lesion were found in the cerebrum thalamus\
       midbrain pons medulla and cerebellum which were characterized by liquefactive necrosis marked \
        edema hemorrhage and necrotizing vasculitis associated with the perivascular accumulation of amoebic\
         trophozoite a few cyst and the infiltration of numerous neutrophil and microgliamacrophages The trophozoite\
          were ovoid or round 1060 μm in diameter and they showed foamy cytoplasm and a round nucleus with small\
           karyosome in the center The PCR and immunohistochemistry from paraffinembedded brain specimen revealed\
            angioinvasive encephalitis due to B mandrillaris Human case of B mandrillaris brain infection are rare\
             in Japan with only a few brief report in the literature"

bioSentVec = "INTRODUCTION Primary amoebic meningoencephalitis PAM is a rare disease caused by the freeliving\
 amoeba Naegleria fowleri Infection occurs by insufflation of water containing amoeba into the nasal cavity\
  and is usually associated with bathing in freshwater Nasal irrigation is a more rarely reported route of infection\
   CASE PRESENTATION A fatal case of PAM in a previously healthy Norwegian woman acquired during a holiday trip\
    to Thailand is described Clinical finding were consistent with rapidly progressing meningoencephalitis\
     The cause of infection wa discovered by chance owing to the unexpected detection of N fowleri DNA by a\
      PCR assay targeting fungi A conclusive diagnosis wa established based on sequencing of N fowleri DNA\
       from brain biopsy supported by histopathological finding Nasal irrigation using contaminated tap water\
        is suspected a the source of infection CONCLUSION The clinical presentation of PAM is very similar to\
         severe bacterial meningitis This case is a reminder that when standard investigation fail to identify\
          a cause of infection in severe meningoencephalitis it is of crucial importance to continue a broad search\
           for a conclusive diagnosis PAM should be considered a a diagnosis in patient with symptom of severe\
            meningoencephalitis returning from endemic area"

In [ ]:
print(f'Question: {question}\n')
print(f'TF-IDF>>BM25: {tf_idf_bm_25 }\n')
print(f'TF-IDF Vectorizer: {tf_idf_vectorizer}\n')
print(f'Doc2Vec: {doc2vec}\n')
print(f'sentence transformers: {sentence_transformer}\n')
print(f'BioSentVec: {bioSentVec}\n')

# Models

Every model were tested both before and after fine-tuning, utilizing both BioSentVec embeddings and BERTscore.

To expedite the evaluation process, a stopping criterion was employed. If, in the last 20 iterations of the for loop calculating the mean score, the mean does not change significantly, the execution stops, and the score is printed.

Additionally, the models were evaluated for execution time in answering questions.

#[Mistral](https://mistral.ai/news/announcing-mistral-7b/)

Mistral is an open-source large language model developed by Mistral AI.

In this notebook, we are using Mistral 7B with 7 billion parameters. We will define prompts for tuning our model and create functions to map our data to these prompts. By doing so, we can tailor the model’s responses to the specific question answering problem.

##Configurations and imports

Let's import everything we need for the execution!

**Remember to set the flags in the right way to execute everything you need**

In [ ]:
USE_PRETRAINED = True       # Set to False if you want to fine-tune the model and save it, otherwise it will be loaded the previous fine-tuned LoRA model
EXPORT_MODEL = False        # Set to True if you want to export the GGUF model
DOWNLOAD_BIOSENT = False    # Set to True if you want to download the biosentvec dataset to test the model
TEST_MODEL = True           # Set to True if you want to test the model
DOWNLOAD_DATASET = False    # Set to True if you want to download the huggingface dataset and re-split it (note that by doing it you will have a different split of the train-test)

In [ ]:
MODEL_PATH = '/content/drive/MyDrive/Colab Notebooks/NLP/project/Models/Mistral'
DATAS_PATH = '/content/drive/MyDrive/Colab Notebooks/NLP/project/Models/Datasets'
model_name = "unsloth/mistral-7b-v0.3-bnb-4bit"

In [ ]:
%%capture
# Installs Unsloth, Xformers (Flash Attention) and all other packages!
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from google.colab import drive
from unsloth import FastLanguageModel
from unsloth import is_bfloat16_supported
import torch
from datasets import load_dataset
from datasets import Dataset
import os
from trl import SFTTrainer
import sys
from transformers import TrainingArguments
from transformers.utils import logging
from nltk import word_tokenize
from nltk.corpus import stopwords
from string import punctuation
from scipy.spatial import distance
import nltk
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

In [ ]:
if TEST_MODEL:
  !pip install bert-score
  from bert_score import BERTScorer

  !git clone https://github.com/epfml/sent2vec.git
  %cd sent2vec
  !make
  !pip install .
  import sent2vec

  sys.setrecursionlimit(4000)

In [ ]:
drive.mount('/content/drive')

In [ ]:
if DOWNLOAD_BIOSENT:
  os.chdir(f'{DATAS_PATH}')
  os.getcwd()
  !wget https://ftp.ncbi.nlm.nih.gov/pub/lu/Suppl/BioSentVec/BioSentVec_PubMed_MIMICIII-bigram_d700.bin

In [ ]:
os.chdir(f'{MODEL_PATH}')
os.getcwd()

In [ ]:
nltk.download('stopwords')
nltk.download('punkt')

## Model definition

To accelerate the fine-tuning process of our model, we opted for a lightweight tool developed by the community: [Unsloth](https://github.com/unslothai/unsloth).

Unsloth works by overwriting some parts of the modeling code with optimized operations. By manually deriving backpropagation steps and rewriting all Pytorch modules into Triton kernels, Unsloth can both reduce memory usage and make fine-tuning faster.

Crucially, **accuracy degradation is 0%** with respect to normal QLoRA, because no approximations are made in the optimized code.

In [ ]:
max_seq_length = 4096   # We use RoPE Scaling to extend it to 4096 with Unsloth!
dtype = None            # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True     # Use 4bit quantization to reduce memory usage. Can be False.

In [ ]:
if USE_PRETRAINED:
    #@title Loading finetuned models
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

In [ ]:
if not USE_PRETRAINED:
  model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
  )


To speedup the tuning even more and decrease the needed computational power we used [LoRA](https://arxiv.org/abs/2106.09685). LoRA is a novel technique introduced by Microsoft researchers to deal with the problem of fine-tuning large-language models.

LoRA proposes freezing pre-trained model weights, which reduces the number of trainable parameters and GPU memory requirements.

In [ ]:
if not USE_PRETRAINED:
  model = FastLanguageModel.get_peft_model(
      model,
      r = 32,    #16 in previous implementation                    # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
      target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj",],
      lora_alpha = 32, #16 in previous implementation
      lora_dropout = 0,                                            # Currently only supports dropout = 0
      bias = "none",                                               # Currently only supports bias = "none"
      use_gradient_checkpointing = "unsloth",                          # We set `gradient_checkpointing=False` ONLY for TinyLlama since Unsloth saves tonnes of memory usage. @@@ IF YOU GET OUT OF MEMORY - set to True @@@
      random_state = 3407,
      use_rslora = False,                                          # We support rank stabilized LoRA
      loftq_config = None,                                         # And LoftQ
  )

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass



## Dataset import
To maintain the integrity of model testing, we pre-saved the dataset split in a dataframe. This ensures that during testing, no training samples are utilized, thereby preserving the model evaluation process.

In [ ]:
if DOWNLOAD_DATASET:
  dataset_path = "medalpaca/medical_meadow_medical_flashcards"
  raw_dataset = load_dataset(dataset_path)
  dataset = raw_dataset["train"].train_test_split(test_size=0.2)
  train_dataset = dataset["train"]
  test_dataset = dataset["test"]
  if False:
    df_train = dataset["train"].to_pandas()
    df_test = dataset["test"].to_pandas()
    os.chdir(f'{DATAS_PATH}')
    os.getcwd()
    df_train.to_csv("train_split.csv", index=False)
    df_test.to_csv("test_split.csv", index=False)
    os.chdir(f'{MODEL_PATH}')
    os.getcwd()
else:
  os.chdir(f'{DATAS_PATH}')
  os.getcwd()
  test_dataset = Dataset.from_pandas(pd.read_csv('test_split.csv'))
  train_dataset = Dataset.from_pandas(pd.read_csv('train_split.csv'))
  os.chdir(f'{MODEL_PATH}')
  os.getcwd()

train_dataset = train_dataset.map(formatting_prompts_func, batched = True)
test_dataset = test_dataset.map(formatting_prompts_func, batched = True)

## Train the model
Now let's use Huggingface TRL's `SFTTrainer`!

In [ ]:
if not USE_PRETRAINED:
  logging.set_verbosity_info()

  training_args = TrainingArguments(
      per_device_train_batch_size = 8,
      gradient_accumulation_steps = 4,
      warmup_ratio = 0.1,
      num_train_epochs = 3,
      save_steps = 100,
      learning_rate = 2e-4,
      fp16 = not is_bfloat16_supported(),
      bf16 = is_bfloat16_supported(),
      #fp16 = not torch.cuda.is_bf16_supported(),
      #bf16 = torch.cuda.is_bf16_supported(),
      logging_steps = 1,
      optim = "adamw_8bit",
      weight_decay = 0.01,
      lr_scheduler_type = "linear",
      seed = 3407,
      output_dir = "outputs",
      overwrite_output_dir = True
  )

  trainer = SFTTrainer(
      model = model,
      tokenizer = tokenizer,
      train_dataset = train_dataset,
      dataset_text_field = "text",
      max_seq_length = max_seq_length,
      dataset_num_proc = 2,
      packing = False, # Packs short sequences together to save time!
      args = training_args,
  )

In [ ]:
if not USE_PRETRAINED:
  #@title Show current memory stats
  gpu_stats = torch.cuda.get_device_properties(0)
  start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
  max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
  print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
  print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
if not USE_PRETRAINED:
  if len(os.listdir("outputs")) <= 1:
    trainer_stats = trainer.train()
  else:
    trainer_stats = trainer.train(resume_from_checkpoint=True)

In [ ]:
if not USE_PRETRAINED:
  #@title Show final memory and time stats
  used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
  used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
  used_percentage = round(used_memory         /max_memory*100, 3)
  lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
  print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
  print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
  print(f"Peak reserved memory = {used_memory} GB.")
  print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
  print(f"Peak reserved memory % of max memory = {used_percentage} %.")
  print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

##Functions
Some useful functions!

In [ ]:
stop_words = set(stopwords.words('english'))
def preprocess_sentence(text):
    text = text.replace('/', ' / ')
    text = text.replace('.-', ' .- ')
    text = text.replace('.', ' . ')
    text = text.replace('\'', ' \' ')
    text = text.lower()

    tokens = [token for token in word_tokenize(text) if token not in punctuation and token not in stop_words]

    return ' '.join(tokens)

In [ ]:
def generate_answer(question):
  # alpaca_prompt = Copied from above
  FastLanguageModel.for_inference(model) # Enable native 2x faster inference
  inputs = tokenizer(
  [
      alpaca_prompt.format(
          "Answer this question truthfully", # instruction
          question, # input
          "", # output - leave this blank for generation!
      )
  ], return_tensors = "pt").to("cuda")

  outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True, pad_token_id=tokenizer.eos_token_id)
  tokenizer.batch_decode(outputs)
  predicted_answer = tokenizer.batch_decode(outputs[:, inputs['input_ids'].shape[1]:])[0]
  predicted_answer = predicted_answer[:(len(predicted_answer)-5)]
  return predicted_answer

## Inference
Let's run the model! You can change the instruction and input - leave the output blank!

 We can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Answer this question truthfully", # instruction
        "What is the PAP test?", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128, use_cache = False)

##Testing

To evaluate the model is essential to establish metrics that evaluate the accuracy of a generated answer relative to the correct one. In our context, we have identified two key metrics: **BERTScore** and **Evaluation with Embedding**

We chose these metrics over more statistical ones because both BERTScores and embeddings take into account the contextual meaning of sentences when assessing their similarity, making them excellent choices for evaluating our models.

To expedite the evaluation process, a stopping criterion has been used. If, in the last 20 iterations of the for loop calculating the mean score, the mean does not change significantly, the execution stops, and the score is printed.

Additionally, the model has been evaluated for execution time in answering questions.

In [ ]:
index = 300
#@title Manual testing
real_answer = test_dataset[index]["output"]
generated_answer = generate_answer(test_dataset[index]["input"])
question = test_dataset[index]["input"]

print("Question: ", question)
print("Generated answer: ", generated_answer)
print("Real answer: ", real_answer)

###BioSentVec loading...

In [ ]:
if TEST_MODEL:
  model_path = f'{DATAS_PATH}/BioSentVec_PubMed_MIMICIII-bigram_d700.bin'
  testing_model = sent2vec.Sent2vecModel()

In [ ]:
if TEST_MODEL:
  def evaluate_answer(real, predicted):
    sentence_vector1 = testing_model.embed_sentence(preprocess_sentence(real))
    sentence_vector2 = testing_model.embed_sentence(preprocess_sentence(predicted))
    cosine_sim = 1 - distance.cosine(sentence_vector1[0], sentence_vector2[0])
    return cosine_sim

In [ ]:
if TEST_MODEL:
  try:
      testing_model.load_model(model_path)
  except Exception as e:
      print(e)
  print('model successfully loaded')

###BioSentVec score

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_list = []
  patience = 20
  k = 0
  for i in range(len(test_dataset)):
    real_answer = test_dataset[i]["output"]
    question = test_dataset[i]["input"]
    generated_answer = generate_answer(question)
    if (generated_answer) is not None and (real_answer) is not None:
      score = evaluate_answer(real_answer, generated_answer)
      if i == 0:
        mean = score
      sum += score
      n_try += 1
      old_mean = mean
      mean = sum/n_try
      mean_list.append(mean)

      if abs(mean - old_mean) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot(mean_list, label='cosine similiarity')
  plt.ylabel('Mean cosine similiarity')
  plt.xlabel('Iteration')
  plt.title("Mean cosine similarity per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print(f'The average cosine similarity is: {mean}')

###BERTScore

In [ ]:
# Initialize the BERTScorer object
if TEST_MODEL:
  scorer = BERTScorer(lang='en')

In [ ]:
if TEST_MODEL:
  sum = [0, 0, 0]
  n_try = 0
  mean_score_list = []
  patience = 20
  k = 0
  for i in range(len(test_dataset)):
    real_answer = test_dataset[i]["output"]
    question = test_dataset[i]["input"]

    generated_answer = generate_answer(question)
    if (generated_answer) is not None and (real_answer) is not None:
      P, R, F1 = scorer.score([generated_answer], [real_answer])
      score = [float(P), float(R), float(F1)]
      if i == 0:
        mean_score = score
      sum = [x + y for x, y in zip(sum, score)]
      n_try += 1
      old_mean = mean_score
      mean_score = [(x / n_try) for x in sum]
      mean_score_list.append(mean_score)

      if abs(mean_score[0] - old_mean[0]) < 0.001 and abs(mean_score[1] - old_mean[1]) < 0.001 and abs(mean_score[2] - old_mean[2]) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean_score} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot([score[0] for score in mean_score_list], color='#2ca02c',label='Precision')
  plt.plot([score[1] for score in mean_score_list], color='#d62728',label='Recall')
  plt.plot([score[2] for score in mean_score_list],label='F1 score')

  plt.ylabel('Mean scores')
  plt.xlabel('Iteration')
  plt.title("Mean scores  per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print("Precision: ", mean_score[0])
  print("Recall: ", mean_score[1])
  print("F1 score: ", mean_score[2])

###Execution time

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_time_list = []
  patience = 20
  k = 0
  for i in tqdm(range(200)):
    question = test_dataset[i]["input"]

    start = time.time()
    generated_answer = generate_answer(question)
    end = time.time()

    score_time = end-start
    if i == 0:
      mean_time = score_time
    sum += score_time
    n_try += 1
    old_mean_time = mean_time
    mean_time = sum/n_try
    mean_time_list.append(mean_time)

In [ ]:
plt.plot(mean_time_list)
plt.plot([mean_time for i in mean_time_list], '--', color='blue', label='Mean execution time', linewidth=1)
plt.ylabel('Mean execution time')
plt.xlabel('Iteration')
plt.title("Mean execution time per iteartion")
plt.legend()
plt.show()

In [ ]:
if TEST_MODEL:
  print(f'The average execution time is: {mean_time}')

## Saving
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.


In [ ]:
if not USE_PRETRAINED:
  model.save_pretrained("lora_model") # Local saving
  tokenizer.save_pretrained("lora_model")

##GGUF / llama.cpp Conversion
Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods:
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

In [ ]:
if EXPORT_MODEL:
  # Save to 8bit Q8_0
  if False: model.save_pretrained_gguf("model", tokenizer,)
  if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

  # Save to 16bit GGUF
  if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
  if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

  # Save to q4_k_m GGUF
  if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
  if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

#[Gemma](https://ai.google.dev/gemma/docs)
Gemma is an open-source large language model developed by Google.

In this notebook, we are using Gemma with 7 billion parameters. We will define prompts for tuning our model and create functions to map our data to these prompts. By doing so, we can tailor the model’s responses to the specific question answering problem.

##Configurations and imports

Let's import everything we need for the execution!

**Remember to set the flags in the right way to execute everything you need**

In [ ]:
USE_PRETRAINED = True       # Set to False if you want to fine-tune the model and save it, otherwise it will be loaded the previous fine-tuned LoRA model
EXPORT_MODEL = False        # Set to True if you want to export the GGUF model
DOWNLOAD_BIOSENT = False    # Set to True if you want to download the biosentvec dataset to test the model
TEST_MODEL = True           # Set to True if you want to test the model
DOWNLOAD_DATASET = False    # Set to True if you want to download the huggingface dataset and re-split it (note that by doing it you will have a different split of the train-test)

In [ ]:
MODEL_PATH = '/content/drive/MyDrive/Colab Notebooks/NLP/project/Models/Gemma'
DATAS_PATH = '/content/drive/MyDrive/Colab Notebooks/NLP/project/Models/Datasets'
model_name = "unsloth/gemma-7b-bnb-4bit"

In [ ]:
%%capture
# Installs Unsloth, Xformers (Flash Attention) and all other packages!
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from google.colab import drive
from unsloth import FastLanguageModel
from unsloth import is_bfloat16_supported
import torch
from datasets import load_dataset
from datasets import Dataset
import os
from trl import SFTTrainer
import sys
from transformers import TrainingArguments
from transformers.utils import logging
from nltk import word_tokenize
from nltk.corpus import stopwords
from string import punctuation
from scipy.spatial import distance
import nltk
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

In [ ]:
if TEST_MODEL:
  !pip install bert-score
  from bert_score import BERTScorer

  !git clone https://github.com/epfml/sent2vec.git
  %cd sent2vec
  !make
  !pip install .
  import sent2vec

  sys.setrecursionlimit(4000)

In [ ]:
drive.mount('/content/drive')

In [ ]:
if DOWNLOAD_BIOSENT:
  os.chdir(f'{DATAS_PATH}')
  os.getcwd()
  !wget https://ftp.ncbi.nlm.nih.gov/pub/lu/Suppl/BioSentVec/BioSentVec_PubMed_MIMICIII-bigram_d700.bin

In [ ]:
os.chdir(f'{MODEL_PATH}')
os.getcwd()

In [ ]:
nltk.download('stopwords')
nltk.download('punkt')

## Model definition

To accelerate the fine-tuning process of our model, we opted for a lightweight tool developed by the community: [Unsloth](https://github.com/unslothai/unsloth).

Unsloth works by overwriting some parts of the modeling code with optimized operations. By manually deriving backpropagation steps and rewriting all Pytorch modules into Triton kernels, Unsloth can both reduce memory usage and make fine-tuning faster.

Crucially, **accuracy degradation is 0%** with respect to normal QLoRA, because no approximations are made in the optimized code.

In [ ]:
max_seq_length = 4096   # TinyLlama's internal maximum sequence length is 2048. We use RoPE Scaling to extend it to 4096 with Unsloth!
dtype = None            # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True     # Use 4bit quantization to reduce memory usage. Can be False.

In [ ]:
if USE_PRETRAINED:
    #@title Loading finetuned models
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

In [ ]:
if not USE_PRETRAINED:
  model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
  )


To speedup the tuning even more and decrease the needed computational power we used [LoRA](https://arxiv.org/abs/2106.09685). LoRA is a novel technique introduced by Microsoft researchers to deal with the problem of fine-tuning large-language models.

LoRA proposes freezing pre-trained model weights, which reduces the number of trainable parameters and GPU memory requirements.

In [ ]:
if not USE_PRETRAINED:
  model = FastLanguageModel.get_peft_model(
      model,
      r = 32,    #16 in previous implementation                    # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
      target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj",],
      lora_alpha = 32, #16 in previous implementation
      lora_dropout = 0,                                            # Currently only supports dropout = 0
      bias = "none",                                               # Currently only supports bias = "none"
      use_gradient_checkpointing = "unsloth",                          # We set `gradient_checkpointing=False` ONLY for TinyLlama since Unsloth saves tonnes of memory usage. @@@ IF YOU GET OUT OF MEMORY - set to True @@@
      random_state = 3407,
      use_rslora = False,                                          # We support rank stabilized LoRA
      loftq_config = None,                                         # And LoftQ
  )

Now, let's define the prompt used for tuning our model and create the function to map our data to this prompt.*testo in corsivo*

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass



## Dataset import
To maintain the integrity of model testing, we pre-saved the dataset split in a dataframe. This ensures that during testing, no training samples are utilized, thereby preserving the model evaluation process.

In [ ]:
if DOWNLOAD_DATASET:
  dataset_path = "medalpaca/medical_meadow_medical_flashcards"
  raw_dataset = load_dataset(dataset_path)
  dataset = raw_dataset["train"].train_test_split(test_size=0.2)
  train_dataset = dataset["train"]
  test_dataset = dataset["test"]
  if False:
    df_train = dataset["train"].to_pandas()
    df_test = dataset["test"].to_pandas()
    os.chdir(f'{DATAS_PATH}')
    os.getcwd()
    df_train.to_csv("train_split.csv", index=False)
    df_test.to_csv("test_split.csv", index=False)
    os.chdir(f'{MODEL_PATH}')
    os.getcwd()
else:
  os.chdir(f'{DATAS_PATH}')
  os.getcwd()
  test_dataset = Dataset.from_pandas(pd.read_csv('test_split.csv'))
  train_dataset = Dataset.from_pandas(pd.read_csv('train_split.csv'))
  os.chdir(f'{MODEL_PATH}')
  os.getcwd()

train_dataset = train_dataset.map(formatting_prompts_func, batched = True)
test_dataset = test_dataset.map(formatting_prompts_func, batched = True)

## Train the model
Now let's use Huggingface TRL's `SFTTrainer`!

In [ ]:
if not USE_PRETRAINED:
  logging.set_verbosity_info()

  training_args = TrainingArguments(
      per_device_train_batch_size = 8,
      gradient_accumulation_steps = 4,
      warmup_ratio = 0.1,
      num_train_epochs = 3,
      save_steps = 100,
      learning_rate = 2e-4,
      fp16 = not is_bfloat16_supported(),
      bf16 = is_bfloat16_supported(),
      #fp16 = not torch.cuda.is_bf16_supported(),
      #bf16 = torch.cuda.is_bf16_supported(),
      logging_steps = 1,
      optim = "adamw_8bit",
      weight_decay = 0.01,
      lr_scheduler_type = "linear",
      seed = 3407,
      output_dir = "outputs",
      overwrite_output_dir = True
  )

  trainer = SFTTrainer(
      model = model,
      tokenizer = tokenizer,
      train_dataset = train_dataset,
      dataset_text_field = "text",
      max_seq_length = max_seq_length,
      dataset_num_proc = 2,
      packing = False, # Packs short sequences together to save time!
      args = training_args,
  )

In [ ]:
if not USE_PRETRAINED:
  #@title Show current memory stats
  gpu_stats = torch.cuda.get_device_properties(0)
  start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
  max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
  print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
  print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
if not USE_PRETRAINED:
  if len(os.listdir("outputs")) <= 1:
    trainer_stats = trainer.train()
  else:
    trainer_stats = trainer.train(resume_from_checkpoint=True)

In [ ]:
if not USE_PRETRAINED:
  #@title Show final memory and time stats
  used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
  used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
  used_percentage = round(used_memory         /max_memory*100, 3)
  lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
  print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
  print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
  print(f"Peak reserved memory = {used_memory} GB.")
  print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
  print(f"Peak reserved memory % of max memory = {used_percentage} %.")
  print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

##Functions
Some useful functions!

In [ ]:
stop_words = set(stopwords.words('english'))
def preprocess_sentence(text):
    text = text.replace('/', ' / ')
    text = text.replace('.-', ' .- ')
    text = text.replace('.', ' . ')
    text = text.replace('\'', ' \' ')
    text = text.lower()

    tokens = [token for token in word_tokenize(text) if token not in punctuation and token not in stop_words]

    return ' '.join(tokens)

In [ ]:
def generate_answer(question):
  # alpaca_prompt = Copied from above
  FastLanguageModel.for_inference(model) # Enable native 2x faster inference
  inputs = tokenizer(
  [
      alpaca_prompt.format(
          "Answer this question truthfully", # instruction
          question, # input
          "", # output - leave this blank for generation!
      )
  ], return_tensors = "pt").to("cuda")

  outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
  tokenizer.batch_decode(outputs)
  predicted_answer = tokenizer.batch_decode(outputs[:, inputs['input_ids'].shape[1]:])[0]
  predicted_answer = predicted_answer[:(len(predicted_answer)-5)]
  return predicted_answer

## Inference
Let's run the model! You can change the instruction and input - leave the output blank!

 We can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Answer this question truthfully", # instruction
        "How it is possible to lose fat?", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128, use_cache = False)

##Testing

To evaluate the model is essential to establish metrics that evaluate the accuracy of a generated answer relative to the correct one. In our context, we have identified two key metrics: **BERTScore** and **Evaluation with Embedding**

We chose these metrics over more statistical ones because both BERTScores and embeddings take into account the contextual meaning of sentences when assessing their similarity, making them excellent choices for evaluating our models.

To expedite the evaluation process, a stopping criterion has been used. If, in the last 20 iterations of the for loop calculating the mean score, the mean does not change significantly, the execution stops, and the score is printed.

Additionally, the model has been evaluated for execution time in answering questions.

In [ ]:
index = 300
#@title Manual testing
real_answer = test_dataset[index]["output"]
generated_answer = generate_answer(test_dataset[index]["input"])
question = test_dataset[index]["input"]

print("Question: ", question)
print("Generated answer: ", generated_answer)
print("Real answer: ", real_answer)

###BioSentVec loading...

In [ ]:
if TEST_MODEL:
  model_path = f'{DATAS_PATH}/BioSentVec_PubMed_MIMICIII-bigram_d700.bin'
  testing_model = sent2vec.Sent2vecModel()

In [ ]:
if TEST_MODEL:
  def evaluate_answer(real, predicted):
    sentence_vector1 = testing_model.embed_sentence(preprocess_sentence(real))
    sentence_vector2 = testing_model.embed_sentence(preprocess_sentence(predicted))
    cosine_sim = 1 - distance.cosine(sentence_vector1[0], sentence_vector2[0])
    return cosine_sim

In [ ]:
if TEST_MODEL:
  try:
      testing_model.load_model(model_path)
  except Exception as e:
      print(e)
  print('model successfully loaded')

###BioSentVec score

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_list = []
  patience = 20
  k = 0
  for i in range(len(test_dataset)):
    real_answer = test_dataset[i]["output"]
    question = test_dataset[i]["input"]
    generated_answer = generate_answer(question)
    if (generated_answer) is not None and (real_answer) is not None:
      score = evaluate_answer(real_answer, generated_answer)
      if i == 0:
        mean = score
      sum += score
      n_try += 1
      old_mean = mean
      mean = sum/n_try
      mean_list.append(mean)

      if abs(mean - old_mean) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot(mean_list, label='cosine similiarity')
  plt.ylabel('Mean cosine similiarity')
  plt.xlabel('Iteration')
  plt.title("Mean cosine similarity per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print(f'the average cosine similarity is: {mean}')

###BERTScore

In [ ]:
# Initialize the BERTScorer object
if TEST_MODEL:
  scorer = BERTScorer(lang='en')

In [ ]:
if TEST_MODEL:
  sum = [0, 0, 0]
  n_try = 0
  mean_score_list = []
  patience = 20
  k = 0
  for i in range(len(test_dataset)):
    real_answer = test_dataset[i]["output"]
    question = test_dataset[i]["input"]

    generated_answer = generate_answer(question)
    if (generated_answer) is not None and (real_answer) is not None:
      P, R, F1 = scorer.score([generated_answer], [real_answer])
      score = [float(P), float(R), float(F1)]
      if i == 0:
        mean_score = score
      sum = [x + y for x, y in zip(sum, score)]
      n_try += 1
      old_mean = mean_score
      mean_score = [(x / n_try) for x in sum]
      mean_score_list.append(mean_score)

      if abs(mean_score[0] - old_mean[0]) < 0.001 and abs(mean_score[1] - old_mean[1]) < 0.001 and abs(mean_score[2] - old_mean[2]) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean_score} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot([score[0] for score in mean_score_list], color='#2ca02c',label='Precision')
  plt.plot([score[1] for score in mean_score_list], color='#d62728',label='Recall')
  plt.plot([score[2] for score in mean_score_list],label='F1 score')

  plt.ylabel('Mean scores')
  plt.xlabel('Iteration')
  plt.title("Mean scores  per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print("Precision: ", mean_score[0])
  print("Recall: ", mean_score[1])
  print("F1 score: ", mean_score[2])

###Execution time

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_time_list = []
  patience = 20
  k = 0
  for i in tqdm(range(200)):
    question = test_dataset[i]["input"]

    start = time.time()
    generated_answer = generate_answer(question)
    end = time.time()

    score_time = end-start
    if i == 0:
      mean_time = score_time
    sum += score_time
    n_try += 1
    old_mean_time = mean_time
    mean_time = sum/n_try
    mean_time_list.append(mean_time)

In [ ]:
plt.plot(mean_time_list)
plt.plot([mean_time for i in mean_time_list], '--', color='blue', label='Mean execution time', linewidth=1)
plt.ylabel('Mean execution time')
plt.xlabel('Iteration')
plt.title("Mean execution time per iteartion")
plt.legend()
plt.show()

In [ ]:
if TEST_MODEL:
  print(f'The average execution time is: {mean_time}')

## Saving
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.


In [ ]:
if not USE_PRETRAINED:
  model.save_pretrained("lora_model") # Local saving
  tokenizer.save_pretrained("lora_model")

##GGUF / llama.cpp Conversion
Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods:
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

In [ ]:
if EXPORT_MODEL:
  # Save to 8bit Q8_0
  if False: model.save_pretrained_gguf("model", tokenizer,)
  if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

  # Save to 16bit GGUF
  if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
  if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

  # Save to q4_k_m GGUF
  if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
  if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

#[Llama](https://llama.meta.com/llama3/)
LLama (Large Language Model Meta AI) is an open-source large language model developed by Meta.

In this notebook, we are using Llama 3 with 8 billion parameters. We will define prompts for tuning our model and create functions to map our data to these prompts. By doing so, we can tailor the model’s responses to the specific question answering problem.

##Configurations and imports

Let's import everything we need for the execution!

**Remember to set the flags in the right way to execute everything you need**

In [ ]:
USE_PRETRAINED = True       # Set to False if you want to fine-tune the model and save it, otherwise it will be loaded the previous fine-tuned LoRA model
EXPORT_MODEL = False        # Set to True if you want to export the GGUF model
DOWNLOAD_BIOSENT = False    # Set to True if you want to download the biosentvec dataset to test the model
TEST_MODEL = True          # Set to True if you want to test the model
DOWNLOAD_DATASET = False    # Set to True if you want to download the huggingface dataset and re-split it (note that by doing it you will have a different split of the train-test)

In [ ]:
MODEL_PATH = '/content/drive/MyDrive/Colab Notebooks/NLP/project/Models/llama'
DATAS_PATH = '/content/drive/MyDrive/Colab Notebooks/NLP/project/Models/Datasets'
model_name = "unsloth/llama-3-8b-bnb-4bit"

In [ ]:
%%capture
# Installs Unsloth, Xformers (Flash Attention) and all other packages!
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from google.colab import drive
from unsloth import FastLanguageModel
from unsloth import is_bfloat16_supported
import torch
from datasets import load_dataset
from datasets import Dataset
import os
from trl import SFTTrainer
import sys
from transformers import TrainingArguments
from transformers.utils import logging
from nltk import word_tokenize
from nltk.corpus import stopwords
from string import punctuation
from scipy.spatial import distance
import nltk
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

In [ ]:
if TEST_MODEL:
  !pip install bert-score
  from bert_score import BERTScorer

  !git clone https://github.com/epfml/sent2vec.git
  %cd sent2vec
  !make
  !pip install .
  import sent2vec

  sys.setrecursionlimit(2500)

In [ ]:
drive.mount('/content/drive')

In [ ]:
if DOWNLOAD_BIOSENT:
  os.chdir(f'{DATAS_PATH}')
  os.getcwd()
  !wget https://ftp.ncbi.nlm.nih.gov/pub/lu/Suppl/BioSentVec/BioSentVec_PubMed_MIMICIII-bigram_d700.bin

In [ ]:
os.chdir(f'{MODEL_PATH}')
os.getcwd()

In [ ]:
nltk.download('stopwords')
nltk.download('punkt')

## Model definition

To accelerate the fine-tuning process of our model, we opted for a lightweight tool developed by the community: [Unsloth](https://github.com/unslothai/unsloth).

Unsloth works by overwriting some parts of the modeling code with optimized operations. By manually deriving backpropagation steps and rewriting all Pytorch modules into Triton kernels, Unsloth can both reduce memory usage and make fine-tuning faster.

Crucially, **accuracy degradation is 0%** with respect to normal QLoRA, because no approximations are made in the optimized code.

In [ ]:
max_seq_length = 4096   # TinyLlama's internal maximum sequence length is 2048. We use RoPE Scaling to extend it to 4096 with Unsloth!
dtype = None            # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True     # Use 4bit quantization to reduce memory usage. Can be False.

In [ ]:
if USE_PRETRAINED:
    #@title Loading finetuned models
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

In [ ]:
if not USE_PRETRAINED:
  model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
  )


To speedup the tuning even more and decrease the needed computational power we used [LoRA](https://arxiv.org/abs/2106.09685). LoRA is a novel technique introduced by Microsoft researchers to deal with the problem of fine-tuning large-language models.

LoRA proposes freezing pre-trained model weights, which reduces the number of trainable parameters and GPU memory requirements.

In [ ]:
if not USE_PRETRAINED:
  model = FastLanguageModel.get_peft_model(
      model,
      r = 32,    #16 in previous implementation                    # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
      target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj",],
      lora_alpha = 32, #16 in previous implementation
      lora_dropout = 0,                                            # Currently only supports dropout = 0
      bias = "none",                                               # Currently only supports bias = "none"
      use_gradient_checkpointing = "unsloth",                          # We set `gradient_checkpointing=False` ONLY for TinyLlama since Unsloth saves tonnes of memory usage. @@@ IF YOU GET OUT OF MEMORY - set to True @@@
      random_state = 3407,
      use_rslora = False,                                          # We support rank stabilized LoRA
      loftq_config = None,                                         # And LoftQ
  )

Now, let's define the prompt used for tuning our model and create the function to map our data to this prompt.*testo in corsivo*

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass



## Dataset import
To maintain the integrity of model testing, we pre-saved the dataset split in a dataframe. This ensures that during testing, no training samples are utilized, thereby preserving the model evaluation process.

In [ ]:
if DOWNLOAD_DATASET:
  dataset_path = "medalpaca/medical_meadow_medical_flashcards"
  raw_dataset = load_dataset(dataset_path)
  dataset = raw_dataset["train"].train_test_split(test_size=0.2)
  train_dataset = dataset["train"]
  test_dataset = dataset["test"]
  if False:
    df_train = dataset["train"].to_pandas()
    df_test = dataset["test"].to_pandas()
    os.chdir(f'{DATAS_PATH}')
    os.getcwd()
    df_train.to_csv("train_split.csv", index=False)
    df_test.to_csv("test_split.csv", index=False)
    os.chdir(f'{MODEL_PATH}')
    os.getcwd()
else:
  os.chdir(f'{DATAS_PATH}')
  os.getcwd()
  test_dataset = Dataset.from_pandas(pd.read_csv('test_split.csv'))
  train_dataset = Dataset.from_pandas(pd.read_csv('train_split.csv'))
  os.chdir(f'{MODEL_PATH}')
  os.getcwd()

train_dataset = train_dataset.map(formatting_prompts_func, batched = True)
test_dataset = test_dataset.map(formatting_prompts_func, batched = True)

## Train the model
Now let's use Huggingface TRL's `SFTTrainer`!

In [ ]:
if not USE_PRETRAINED:
  logging.set_verbosity_info()

  training_args = TrainingArguments(
      per_device_train_batch_size = 8,
      gradient_accumulation_steps = 4,
      warmup_ratio = 0.1,
      num_train_epochs = 3,
      save_steps = 100,
      learning_rate = 2e-4,
      fp16 = not is_bfloat16_supported(),
      bf16 = is_bfloat16_supported(),
      #fp16 = not torch.cuda.is_bf16_supported(),
      #bf16 = torch.cuda.is_bf16_supported(),
      logging_steps = 1,
      optim = "adamw_8bit",
      weight_decay = 0.01,
      lr_scheduler_type = "linear",
      seed = 3407,
      output_dir = "outputs",
      overwrite_output_dir = True
  )

  trainer = SFTTrainer(
      model = model,
      tokenizer = tokenizer,
      train_dataset = train_dataset,
      dataset_text_field = "text",
      max_seq_length = max_seq_length,
      dataset_num_proc = 2,
      packing = False, # Packs short sequences together to save time!
      args = training_args,
  )

In [ ]:
if not USE_PRETRAINED:
  #@title Show current memory stats
  gpu_stats = torch.cuda.get_device_properties(0)
  start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
  max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
  print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
  print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
if not USE_PRETRAINED:
  if len(os.listdir("outputs")) <= 1:
    trainer_stats = trainer.train()
  else:
    trainer_stats = trainer.train(resume_from_checkpoint=True)

In [ ]:
if not USE_PRETRAINED:
  #@title Show final memory and time stats
  used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
  used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
  used_percentage = round(used_memory         /max_memory*100, 3)
  lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
  print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
  print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
  print(f"Peak reserved memory = {used_memory} GB.")
  print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
  print(f"Peak reserved memory % of max memory = {used_percentage} %.")
  print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

##Functions
Some useful functions!

In [ ]:
stop_words = set(stopwords.words('english'))
def preprocess_sentence(text):
    text = text.replace('/', ' / ')
    text = text.replace('.-', ' .- ')
    text = text.replace('.', ' . ')
    text = text.replace('\'', ' \' ')
    text = text.lower()

    tokens = [token for token in word_tokenize(text) if token not in punctuation and token not in stop_words]

    return ' '.join(tokens)

In [ ]:
def generate_answer(question):
  # alpaca_prompt = Copied from above
  FastLanguageModel.for_inference(model) # Enable native 2x faster inference
  inputs = tokenizer(
  [
      alpaca_prompt.format(
          "Answer this question truthfully", # instruction
          question, # input
          "", # output - leave this blank for generation!
      )
  ], return_tensors = "pt").to("cuda")

  outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True, pad_token_id=tokenizer.eos_token_id)
  tokenizer.batch_decode(outputs)
  predicted_answer = tokenizer.batch_decode(outputs[:, inputs['input_ids'].shape[1]:])[0]
  predicted_answer = predicted_answer[:(len(predicted_answer)-5)]
  return predicted_answer

## Inference
Let's run the model! You can change the instruction and input - leave the output blank!

 We can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Answer this question truthfully", # instruction
        "What is the PAP test?", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

##Testing

To evaluate the model is essential to establish metrics that evaluate the accuracy of a generated answer relative to the correct one. In our context, we have identified two key metrics: **BERTScore** and **Evaluation with Embedding**

We chose these metrics over more statistical ones because both BERTScores and embeddings take into account the contextual meaning of sentences when assessing their similarity, making them excellent choices for evaluating our models.

To expedite the evaluation process, a stopping criterion has been used. If, in the last 20 iterations of the for loop calculating the mean score, the mean does not change significantly, the execution stops, and the score is printed.

Additionally, the model has been evaluated for execution time in answering questions.

In [ ]:
index = 300
#@title Manual testing
real_answer = test_dataset[index]["output"]
generated_answer = generate_answer(test_dataset[index]["input"])
question = test_dataset[index]["input"]

print("Question: ", question)
print("Generated answer: ", generated_answer)
print("Real answer: ", real_answer)

###BioSentVec loading...

In [ ]:
if TEST_MODEL:
  model_path = f'{DATAS_PATH}/BioSentVec_PubMed_MIMICIII-bigram_d700.bin'
  testing_model = sent2vec.Sent2vecModel()

In [ ]:
if TEST_MODEL:
  def evaluate_answer(real, predicted):
    sentence_vector1 = testing_model.embed_sentence(preprocess_sentence(real))
    sentence_vector2 = testing_model.embed_sentence(preprocess_sentence(predicted))
    cosine_sim = 1 - distance.cosine(sentence_vector1[0], sentence_vector2[0])
    return cosine_sim

In [ ]:
if TEST_MODEL:
  try:
      testing_model.load_model(model_path)
  except Exception as e:
      print(e)
  print('model successfully loaded')

###BioSentVec score

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_list = []
  patience = 20
  k = 0
  for i in range(len(test_dataset)):
    real_answer = test_dataset[i]["output"]
    question = test_dataset[i]["input"]
    generated_answer = generate_answer(question)
    if (generated_answer) is not None and (real_answer) is not None:
      score = evaluate_answer(real_answer, generated_answer)
      if i == 0:
        mean = score
      sum += score
      n_try += 1
      old_mean = mean
      mean = sum/n_try
      mean_list.append(mean)

      if abs(mean - old_mean) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot(mean_list, label='cosine similiarity')
  plt.ylabel('Mean cosine similiarity')
  plt.xlabel('Iteration')
  plt.title("Mean cosine similarity per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print(f'The average cosine similarity is: {mean}')

###BERTScore

In [ ]:
# Initialize the BERTScorer object
if TEST_MODEL:
  scorer = BERTScorer(lang='en')

In [ ]:
if TEST_MODEL:
  sum = [0, 0, 0]
  n_try = 0
  mean_score_list = []
  patience = 20
  k = 0
  for i in range(len(test_dataset)):
    real_answer = test_dataset[i]["output"]
    question = test_dataset[i]["input"]

    generated_answer = generate_answer(question)
    if (generated_answer) is not None and (real_answer) is not None:
      P, R, F1 = scorer.score([generated_answer], [real_answer])
      score = [float(P), float(R), float(F1)]
      if i == 0:
        mean_score = score
      sum = [x + y for x, y in zip(sum, score)]
      n_try += 1
      old_mean = mean_score
      mean_score = [(x / n_try) for x in sum]
      mean_score_list.append(mean_score)

      if abs(mean_score[0] - old_mean[0]) < 0.001 and abs(mean_score[1] - old_mean[1]) < 0.001 and abs(mean_score[2] - old_mean[2]) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean_score} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot([score[0] for score in mean_score_list], color='#2ca02c',label='Precision')
  plt.plot([score[1] for score in mean_score_list], color='#d62728',label='Recall')
  plt.plot([score[2] for score in mean_score_list],label='F1 score')

  plt.ylabel('Mean scores')
  plt.xlabel('Iteration')
  plt.title("Mean scores  per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print("Precision: ", mean_score[0])
  print("Recall: ", mean_score[1])
  print("F1 score: ", mean_score[2])

###Execution time

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_time_list = []
  patience = 20
  k = 0
  for i in tqdm(range(200)):
    question = test_dataset[i]["input"]

    start = time.time()
    generated_answer = generate_answer(question)
    end = time.time()

    score_time = end-start
    if i == 0:
      mean_time = score_time
    sum += score_time
    n_try += 1
    old_mean_time = mean_time
    mean_time = sum/n_try
    mean_time_list.append(mean_time)

In [ ]:
if TEST_MODEL:
  plt.plot(mean_time_list)
  plt.plot([mean_time for i in mean_time_list], '--', color='blue', label='Mean', linewidth=1)
  plt.ylabel('Mean execution time')
  plt.xlabel('Iteration')
  plt.title("Mean execution time per iteartion")
  plt.show()

In [ ]:
if TEST_MODEL:
  print(f'The average execution time is: {mean_time}')

## Saving
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.


In [ ]:
if not USE_PRETRAINED:
  model.save_pretrained("lora_model") # Local saving
  tokenizer.save_pretrained("lora_model")

##GGUF / llama.cpp Conversion
Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods:
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

In [ ]:
if EXPORT_MODEL:
  # Save to 8bit Q8_0
  if False: model.save_pretrained_gguf("model", tokenizer,)
  if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

  # Save to 16bit GGUF
  if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
  if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

  # Save to q4_k_m GGUF
  if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
  if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

#[Tiny Llama](https://arxiv.org/pdf/2401.02385)

TinyLlama, an open-source small language model developed by Peiyuan Zhang, Guangtao Zeng, Tianduo Wang, and Wei Lu, distinguishes itself from its larger counterpart, Llama, by operating with reduced computational demands.

Despite having 1.1 billion parameters, this figure is modest compared to other LLMs with hundreds of billions or even trillions of parameters.


##Configurations and imports

Let's import everything we need for the execution!

**Remember to set the flags in the right way to execute everything you need**

In [ ]:
USE_PRETRAINED = True       # Set to False if you want to fine-tune the model and save it, otherwise it will be loaded the previous fine-tuned LoRA model
EXPORT_MODEL = False        # Set to True if you want to export the GGUF model
DOWNLOAD_BIOSENT = False    # Set to True if you want to download the biosentvec dataset to test the model
TEST_MODEL = True           # Set to True if you want to test the model
DOWNLOAD_DATASET = False    # Set to True if you want to download the huggingface dataset and re-split it (note that by doing it you will have a different split of the train-test)

In [ ]:
MODEL_PATH = '/content/drive/MyDrive/Colab Notebooks/NLP/project/Models/Tiny llama'
DATAS_PATH = '/content/drive/MyDrive/Colab Notebooks/NLP/project/Models/Datasets'
model_name = "unsloth/tinyllama-bnb-4bit"

In [ ]:
%%capture
# Installs Unsloth, Xformers (Flash Attention) and all other packages!
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from google.colab import drive

from unsloth import FastLanguageModel
from unsloth import is_bfloat16_supported

import torch

from datasets import load_dataset
from datasets import Dataset

from trl import SFTTrainer

from transformers import TrainingArguments
from transformers.utils import logging

from nltk import word_tokenize
from nltk.corpus import stopwords
import nltk

from string import punctuation

from scipy.spatial import distance

from tqdm import tqdm

import numpy as np

import pandas as pd

import sys
import time
import os

import matplotlib.pyplot as plt

In [ ]:
if TEST_MODEL:
  !pip install bert-score
  from bert_score import BERTScorer

  !git clone https://github.com/epfml/sent2vec.git
  %cd sent2vec
  !make
  !pip install .
  import sent2vec

  sys.setrecursionlimit(4000)

In [ ]:
drive.mount('/content/drive')

In [ ]:
if DOWNLOAD_BIOSENT:
  os.chdir(f'{DATAS_PATH}')
  os.getcwd()
  !wget https://ftp.ncbi.nlm.nih.gov/pub/lu/Suppl/BioSentVec/BioSentVec_PubMed_MIMICIII-bigram_d700.bin

In [ ]:
os.chdir(f'{MODEL_PATH}')
os.getcwd()

In [ ]:
nltk.download('stopwords')
nltk.download('punkt')

## Model definition

To accelerate the fine-tuning process of our model, we opted for a lightweight tool developed by the community: [Unsloth](https://github.com/unslothai/unsloth).

Unsloth works by overwriting some parts of the modeling code with optimized operations. By manually deriving backpropagation steps and rewriting all Pytorch modules into Triton kernels, Unsloth can both reduce memory usage and make fine-tuning faster.

Crucially, **accuracy degradation is 0%** with respect to normal QLoRA, because no approximations are made in the optimized code.

In [ ]:
max_seq_length = 4096   # We use RoPE Scaling to extend it to 4096 with Unsloth!
dtype = None            # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True     # Use 4bit quantization to reduce memory usage. Can be False.

In [ ]:
if USE_PRETRAINED:
    #@title Loading finetuned models
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

In [ ]:
if not USE_PRETRAINED:
  model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
  )


To speedup the tuning even more and decrease the needed computational power we used [LoRA](https://arxiv.org/abs/2106.09685). LoRA is a novel technique introduced by Microsoft researchers to deal with the problem of fine-tuning large-language models.

LoRA proposes freezing pre-trained model weights, which reduces the number of trainable parameters and GPU memory requirements.

In [ ]:
if not USE_PRETRAINED:
  model = FastLanguageModel.get_peft_model(
      model,
      r = 32,    #16 in previous implementation                    # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
      target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj",],
      lora_alpha = 32, #16 in previous implementation
      lora_dropout = 0,                                            # Currently only supports dropout = 0
      bias = "none",                                               # Currently only supports bias = "none"
      use_gradient_checkpointing = "unsloth",                          # We set `gradient_checkpointing=False` ONLY for TinyLlama since Unsloth saves tonnes of memory usage. @@@ IF YOU GET OUT OF MEMORY - set to True @@@
      random_state = 3407,
      use_rslora = False,                                          # We support rank stabilized LoRA
      loftq_config = None,                                         # And LoftQ
  )

Now, let's define the prompt used for tuning our model and create the function to map our data to this prompt.

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass



## Dataset import
To maintain the integrity of model testing, we pre-saved the dataset split in a dataframe. This ensures that during testing, no training samples are utilized, thereby preserving the model evaluation process.

In [ ]:
if DOWNLOAD_DATASET:
  dataset_path = "medalpaca/medical_meadow_medical_flashcards"
  raw_dataset = load_dataset(dataset_path)
  dataset = raw_dataset["train"].train_test_split(test_size=0.2)
  train_dataset = dataset["train"]
  test_dataset = dataset["test"]
  if False:
    df_train = dataset["train"].to_pandas()
    df_test = dataset["test"].to_pandas()
    os.chdir(f'{DATAS_PATH}')
    os.getcwd()
    df_train.to_csv("train_split.csv", index=False)
    df_test.to_csv("test_split.csv", index=False)
    os.chdir(f'{MODEL_PATH}')
    os.getcwd()
else:
  os.chdir(f'{DATAS_PATH}')
  os.getcwd()
  test_dataset = Dataset.from_pandas(pd.read_csv('test_split.csv'))
  train_dataset = Dataset.from_pandas(pd.read_csv('train_split.csv'))
  os.chdir(f'{MODEL_PATH}')
  os.getcwd()

train_dataset = train_dataset.map(formatting_prompts_func, batched = True)
test_dataset = test_dataset.map(formatting_prompts_func, batched = True)

## Tuning the model
Now let's use Huggingface TRL's `SFTTrainer`!

In [ ]:
if not USE_PRETRAINED:
  logging.set_verbosity_info()

  training_args = TrainingArguments(
      per_device_train_batch_size = 8,
      gradient_accumulation_steps = 4,
      warmup_ratio = 0.1,
      num_train_epochs = 3,
      save_steps = 100,
      learning_rate = 2e-4,
      fp16 = not is_bfloat16_supported(),
      bf16 = is_bfloat16_supported(),
      #fp16 = not torch.cuda.is_bf16_supported(),
      #bf16 = torch.cuda.is_bf16_supported(),
      logging_steps = 1,
      optim = "adamw_8bit",
      weight_decay = 0.01,
      lr_scheduler_type = "linear",
      seed = 3407,
      output_dir = "outputs",
      overwrite_output_dir = True
  )

  trainer = SFTTrainer(
      model = model,
      tokenizer = tokenizer,
      train_dataset = train_dataset,
      dataset_text_field = "text",
      max_seq_length = max_seq_length,
      dataset_num_proc = 2,
      packing = False, # Packs short sequences together to save time!
      args = training_args,
  )

In [ ]:
if not USE_PRETRAINED:
  #@title Show current memory stats
  gpu_stats = torch.cuda.get_device_properties(0)
  start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
  max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
  print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
  print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
if not USE_PRETRAINED:
  if len(os.listdir("outputs")) <= 1:
    trainer_stats = trainer.train()
  else:
    trainer_stats = trainer.train(resume_from_checkpoint=True)

In [ ]:
if not USE_PRETRAINED:
  #@title Show final memory and time stats
  used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
  used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
  used_percentage = round(used_memory         /max_memory*100, 3)
  lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
  print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
  print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
  print(f"Peak reserved memory = {used_memory} GB.")
  print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
  print(f"Peak reserved memory % of max memory = {used_percentage} %.")
  print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

##Functions
Some useful functions!

The preprocess_sentence function is utilized to eliminate punctuation and stopwords from the input text.

In [ ]:
stop_words = set(stopwords.words('english'))
def preprocess_sentence(text):
    text = text.replace('/', ' / ')
    text = text.replace('.-', ' .- ')
    text = text.replace('.', ' . ')
    text = text.replace('\'', ' \' ')
    text = text.lower()

    tokens = [token for token in word_tokenize(text) if token not in punctuation and token not in stop_words]

    return ' '.join(tokens)

The generate_answer function utilizes the input model to generate a response to the provided question.

In [ ]:
def generate_answer(question):
  # alpaca_prompt = Copied from above
  FastLanguageModel.for_inference(model) # Enable native 2x faster inference
  inputs = tokenizer(
  [
      alpaca_prompt.format(
          "Answer this question truthfully", # instruction
          question, # input
          "", # output - leave this blank for generation!
      )
  ], return_tensors = "pt").to("cuda")

  outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
  tokenizer.batch_decode(outputs)
  predicted_answer = tokenizer.batch_decode(outputs[:, inputs['input_ids'].shape[1]:])[0]
  predicted_answer = predicted_answer[:(len(predicted_answer)-5)]
  return predicted_answer

## Inference
Let's run the model! You can change the instruction and input - leave the output blank!

 We can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Answer this question truthfully", # instruction
        "What is the PAP test?", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

##Testing

To evaluate the model is essential to establish metrics that evaluate the accuracy of a generated answer relative to the correct one. In our context, we have identified two key metrics: **BERTScore** and **Evaluation with Embedding**

We chose these metrics over more statistical ones because both BERTScores and embeddings take into account the contextual meaning of sentences when assessing their similarity, making them excellent choices for evaluating our models.

To expedite the evaluation process, a stopping criterion has been used. If, in the last 20 iterations of the for loop calculating the mean score, the mean does not change significantly, the execution stops, and the score is printed.

Additionally, the model has been evaluated for execution time in answering questions.

In [ ]:
index = 300
#@title Manual testing
real_answer = test_dataset[index]["output"]
generated_answer = generate_answer(test_dataset[index]["input"])
question = test_dataset[index]["input"]

print("Question: ", question)
print("Generated answer: ", generated_answer)
print("Real answer: ", real_answer)

###BioSentVec loading...

In [ ]:
if TEST_MODEL:
  model_path = f'{DATAS_PATH}/BioSentVec_PubMed_MIMICIII-bigram_d700.bin'
  testing_model = sent2vec.Sent2vecModel()

In [ ]:
if TEST_MODEL:
  def evaluate_answer(real, predicted):
    sentence_vector1 = testing_model.embed_sentence(preprocess_sentence(real))
    sentence_vector2 = testing_model.embed_sentence(preprocess_sentence(predicted))
    cosine_sim = 1 - distance.cosine(sentence_vector1[0], sentence_vector2[0])
    return cosine_sim

In [ ]:
if TEST_MODEL:
  try:
      testing_model.load_model(model_path)
  except Exception as e:
      print(e)
  print('model successfully loaded')

###BioSentVec score

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_list = []
  patience = 20
  k = 0
  for i in range(len(test_dataset)):
    real_answer = test_dataset[i]["output"]
    question = test_dataset[i]["input"]
    generated_answer = generate_answer(question)
    if (generated_answer) is not None and (real_answer) is not None:
      score = evaluate_answer(real_answer, generated_answer)
      if i == 0:
        mean = score
      sum += score
      n_try += 1
      old_mean = mean
      mean = sum/n_try
      mean_list.append(mean)

      if abs(mean - old_mean) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot(mean_list, label='cosine similiarity')
  plt.ylabel('Mean cosine similiarity')
  plt.xlabel('Iteration')
  plt.title("Mean cosine similarity per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print(f'the average cosine similarity is: {mean}')

###BERTScore

In [ ]:
# Initialize the BERTScorer object
if TEST_MODEL:
  scorer = BERTScorer(lang='en')

In [ ]:
if TEST_MODEL:
  sum = [0, 0, 0]
  n_try = 0
  mean_score_list = []
  patience = 20
  k = 0
  for i in range(len(test_dataset)):
    real_answer = test_dataset[i]["output"]
    question = test_dataset[i]["input"]

    generated_answer = generate_answer(question)
    if (generated_answer) is not None and (real_answer) is not None:
      P, R, F1 = scorer.score([generated_answer], [real_answer])
      score = [float(P), float(R), float(F1)]
      if i == 0:
        mean_score = score
      sum = [x + y for x, y in zip(sum, score)]
      n_try += 1
      old_mean = mean_score
      mean_score = [(x / n_try) for x in sum]
      mean_score_list.append(mean_score)

      if abs(mean_score[0] - old_mean[0]) < 0.001 and abs(mean_score[1] - old_mean[1]) < 0.001 and abs(mean_score[2] - old_mean[2]) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean_score} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot([mean_score[0] for score in mean_score_list], color='#2ca02c',label='Precision')
  plt.plot([mean_score[1] for score in mean_score_list], color='#d62728',label='Recall')
  plt.plot([mean_score[2] for score in mean_score_list],label='F1 score')

  plt.ylabel('Mean scores')
  plt.xlabel('Iteration')
  plt.title("Mean scores  per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print("Precision: ", mean_score[0])
  print("Recall: ", mean_score[1])
  print("F1 score: ", mean_score[2])

###Execution time

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_time_list = []
  patience = 20
  k = 0
  for i in tqdm(range(200)):
    question = test_dataset[i]["input"]

    start = time.time()
    generated_answer = generate_answer(question)
    end = time.time()

    score_time = end-start
    if i == 0:
      mean_time = score_time
    sum += score_time
    n_try += 1
    old_mean_time = mean_time
    mean_time = sum/n_try
    mean_time_list.append(mean_time)

In [ ]:
if TEST_MODEL:
  plt.plot(mean_time_list)
  plt.plot([mean_time for i in mean_time_list], '--', color='blue', label='Mean', linewidth=1)
  plt.ylabel('Mean execution time')
  plt.xlabel('Iteration')
  plt.title("Mean execution time per iteartion")
  plt.show()

In [ ]:
if TEST_MODEL:
  print(f'The average execution time is: {mean_time}')

## Saving
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.


In [ ]:
if not USE_PRETRAINED:
  model.save_pretrained("lora_model") # Local saving
  tokenizer.save_pretrained("lora_model")

##GGUF / llama.cpp Conversion
Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods:
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

In [ ]:
if EXPORT_MODEL:
  # Save to 8bit Q8_0
  if False: model.save_pretrained_gguf("model", tokenizer,)
  if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

  # Save to 16bit GGUF
  if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
  if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

  # Save to q4_k_m GGUF
  if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
  if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# GPT2 fine tuning for medical questions

In the notebook **GPT2-medium** is fine tuned to answer to some medical questions.
The GPT2-medium version was used with 345 million parameters. It differs from other since it is based on the Transformer architecture, specifically the decoder-only version and is a general purpose model and can handle different language tasks across different domains.

During training phase a marker [Q] is concatenated to the question and a marker [A] to its answer.

At inference time at the beginning of the question the marker [Q] must me added.

The model can answer to a wide variety of questions related to medical topics it was treined on.

In [ ]:
TRAIN_MODEL = False
DOWNLOAD_DATASET = False    # Set to True if you want to download the huggingface dataset and re-split it (note that by doing it you will have a different split of the train-test)

##Configurations and imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

path = 'Colab Notebooks/NLP_project'

os.chdir(f'/content/drive/MyDrive/{path}')
os.getcwd()

In [ ]:
!pip install datasets
!pip install -U sentence-transformers

!pip install transformers torch tensorflow pandas numpy
!pip3 install accelerate -U
!pip install bert-score

In [ ]:
from datasets import load_dataset
from datasets import load_from_disk
from transformers import TextDataset, DataCollatorForLanguageModeling
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import Trainer, TrainingArguments
from transformers import PreTrainedTokenizerFast, GPT2LMHeadModel, GPT2TokenizerFast, GPT2Tokenizer

from transformers import BertTokenizer, BertModel
from bert_score import BERTScorer
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Dataset import

In [ ]:
if DOWNLOAD_DATASET:
  raw_dataset =  load_dataset("medalpaca/medical_meadow_medical_flashcards", download_mode="force_redownload")
  dataset = raw_dataset["train"].train_test_split(test_size=0.08)
  train_validation_split = dataset["train"].train_test_split(test_size=0.08)
  dataset["train"] = train_validation_split["train"]
  dataset["validation"] = train_validation_split["test"]
  dataset
else:
  dataset = load_from_disk("dataset_gpt_2")

In [ ]:
dataset

In [ ]:
#data are saved in order not to mix test and training
if DOWNLOAD_DATASET:
  dataset.save_to_disk("dataset_gpt_2")

In [ ]:
dataset = dataset.filter(lambda example: example["input"] != "")

## Train the model

In [ ]:
if TRAIN_MODEL:
  # Extract input and output columns
  inputs = dataset["train"]["input"]
  outputs = dataset["train"]["output"]

  # Concatenate input-output pairs

  def get_concat_text(inputs, outputs):
    concatenated_text = ""
    for input_text, output_text in zip(inputs, outputs):
        concatenated_text += f"[Q] {input_text}\n[A] {output_text}\n "
    return re.sub(r'\n+', '\n', concatenated_text).strip()  # Remove excess newline characters

  # Print concatenated text
  text_train = get_concat_text(inputs, outputs)

  with open("train.txt", "w") as f:
      f.write(text_train)

  # validation set
  inputs = dataset["validation"]["input"]
  outputs = dataset["validation"]["output"]

  text_val = get_concat_text(inputs, outputs)

  with open("valid.txt", "w") as f:
      f.write(text_val)

In [ ]:
def load_dataset(file_path, tokenizer, block_size = 128):
    dataset = TextDataset(
        tokenizer = tokenizer,
        file_path = file_path,
        block_size = block_size,
    )
    return dataset

def load_data_collator(tokenizer, mlm = False):
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=mlm,
    )
    return data_collator

In [ ]:
def train(train_file_path,
          eval_file_path,
          model_name,
          output_dir,
          overwrite_output_dir,
          per_device_train_batch_size,
          num_train_epochs,
          save_steps):
  tokenizer = GPT2Tokenizer.from_pretrained(model_name)
  data_collator = load_data_collator(tokenizer)

  train_dataset = load_dataset(train_file_path, tokenizer)
  eval_dataset = load_dataset(eval_file_path, tokenizer)


  tokenizer.save_pretrained(output_dir)

  model = GPT2LMHeadModel.from_pretrained(model_name)

  model.save_pretrained(output_dir)

  training_args = TrainingArguments(
          output_dir=output_dir,
          overwrite_output_dir=overwrite_output_dir,
          per_device_train_batch_size=per_device_train_batch_size,
          num_train_epochs=num_train_epochs,
          eval_steps=500,
          eval_strategy="steps",
          load_best_model_at_end=True,
          save_steps=save_steps
      )

  trainer = Trainer(
          model=model,
          args=training_args,
          data_collator=data_collator,
          train_dataset=train_dataset,
          eval_dataset=eval_dataset
  )

  trainer.train()
  trainer.save_model()
  trainer.evaluate()

In [ ]:

train_file_path = "train.txt"
eval_file_path = "valid.txt"
model_name = 'gpt2-medium'
output_dir = 'medium-val-gpt2-custom_q_and_a'
overwrite_output_dir = False
per_device_train_batch_size = 8
num_train_epochs = 5.0
save_steps = 500

In [ ]:
# Train
if TRAIN_MODEL:
  train(
      train_file_path=train_file_path,
      eval_file_path=eval_file_path,
      model_name=model_name,
      output_dir=output_dir,
      overwrite_output_dir=overwrite_output_dir,
      per_device_train_batch_size=per_device_train_batch_size,
      num_train_epochs=num_train_epochs,
      save_steps=save_steps
  )

##inference

We test our model extracting a random question-answer pair from the dataset and we test on two scores: BLEU and BertScore, it is possible to show that BLEU isn't a good metric for comparison since it doesn't take into consideration the semantic of the sentence

In [ ]:
def load_model(model_path):
    model = GPT2LMHeadModel.from_pretrained(model_path)
    return model


def load_tokenizer(tokenizer_path):
    tokenizer = GPT2Tokenizer.from_pretrained(tokenizer_path)
    return tokenizer

def generate_text(tokenizer, sequence, max_length):


    ids = tokenizer.encode(f'{sequence}', return_tensors='pt')
    final_outputs = model.generate(
        ids,
        do_sample=True,
        max_length=max_length,
        pad_token_id=model.config.eos_token_id,
        top_k=50,
        top_p=0.95,
    )

    return (tokenizer.decode(final_outputs[0], skip_special_tokens=True)).split("[Q]")[1].split("[A]")[1]

In [ ]:
#retrieve model and tokenizer
model_path = "medium-val-gpt2-custom_q_and_a"
model = load_model(model_path)
tokenizer = load_tokenizer(model_path)
tokenizer.padding_side ='left'

In [ ]:
import random

random_index = random.randint(0, len(dataset["test"])-1)

question = "[Q]" + dataset["test"][random_index]["input"]
max_len = 150

print(question)
generated_response = generate_text(tokenizer, question, max_len)
print("generated_response: " + generated_response)
print("true_response: " + dataset["test"][random_index]["output"])

Exection time

In [ ]:
import time
start = time.time()
generated_response = generate_text(tokenizer, question, max_len)
end = time.time()

print("exection time: "+ str(end - start))

In [ ]:
#bleu score: not a good metric

from nltk.translate.bleu_score import sentence_bleu
right_response = dataset["test"][random_index]["output"]
bleu = sentence_bleu( [right_response.split()] , generated_response.split() )
print(f"BLEU: {bleu}")

In [ ]:
scorer = BERTScorer(model_type='bert-base-uncased')
right_response = dataset["test"][random_index]["output"]
P, R, F1 = scorer.score([generated_response], [right_response])
print(f"BERTScore Precision: {P.mean():.4f}, Recall: {R.mean():.4f}, F1: {F1.mean():.4f}")

##Testing

In [ ]:

def load_tokenizer(tokenizer_path):
    tokenizer = GPT2Tokenizer.from_pretrained(tokenizer_path)
    return tokenizer

def load_model(model_path):
    model = GPT2LMHeadModel.from_pretrained(model_path)
    return model

def generate_text_parall(tokenizer, sequence, max_length):
    tokenizer.pad_token = tokenizer.eos_token

    ids = tokenizer(sequence, return_tensors='pt', padding=True ,truncation=True)

    final_outputs = model.generate(
        **ids,
        do_sample=True,
        max_length=max_length,
        pad_token_id=model.config.eos_token_id,
        top_k=50,
        top_p=0.95,
    )

    return [ tokenizer.decode(x, skip_special_tokens=True).split("[Q]")[1].split("[A]")[1] for x in final_outputs ]

In [ ]:
#retrieve model and tokenizer
model_path = "medium-val-gpt2-custom_q_and_a"
model = load_model(model_path)
tokenizer = load_tokenizer(model_path)
tokenizer.padding_side ='left'

scorer = BERTScorer(lang='en')

In [ ]:
max_len = 150
questions = ["[Q]" + x for x in dataset["test"]["input"]]

Inference from the model is performed in batch, test set is splitted into n_splits parts, this is done in order to avoid overloading RAM resources

In [ ]:
n_splits = 160
sections = np.linspace(0, len(questions), n_splits+1)
n_sections = 6

For visualization purpose show only 6 sections

In [ ]:
P = np.array([])
R = np.array([])
F1 = np.array([])

for i in range(n_sections):
  print(i)
  first_index = int(sections[i])
  second_index= int(sections[i+1])
  responses = generate_text_parall(tokenizer, questions[first_index : second_index], max_len)

  P_2, R_2, F1_2 = scorer.score(responses, dataset["test"]["output"][first_index : second_index])
  P = np.append(P, P_2)
  R = np.append(R, R_2)
  F1 = np.append(F1, F1_2)
  print(f"BERTScore Precision : {np.mean(P):.4f}, Recall: {np.mean(R):.4f}, F1: {np.mean(F1):.4f}  #samples: {int((len(questions)/n_splits)*(i+1))}")


In [ ]:
def cum_mean(arr):
  cum_sum = np.cumsum(arr)
  return cum_sum/np.arange(1, len(arr) + 1)

plt.plot(cum_mean(P), color='#2ca02c',label='Precision')
plt.plot(cum_mean(R), color='#d62728',label='Recall')
plt.plot(cum_mean(F1),label='F1 score')

plt.ylabel('Mean scores')
plt.xlabel('Iteration')
plt.title("Mean scores  per iteartion")
plt.legend()
plt.show()

##comparison with GPT2 without fine-tuning

In [ ]:
#retrieve model and tokenizer
model_path = "gpt2-medium"
model = load_model(model_path)
tokenizer = load_tokenizer(model_path)
tokenizer.padding_side ='left'

scorer = BERTScorer(lang='en')

In [ ]:
def generate_text_parall_raw(tokenizer, sequence, max_length):
    tokenizer.pad_token = tokenizer.eos_token

    ids = tokenizer(sequence, return_tensors='pt', padding=True ,truncation=True)

    final_outputs = model.generate(
        **ids,
        do_sample=True,
        max_length=max_length,
        pad_token_id=model.config.eos_token_id,
        top_k=50,
        top_p=0.95,
    )

    return [ tokenizer.decode(x, skip_special_tokens=True) for x in final_outputs ]

In [ ]:
P = np.array([])
R = np.array([])
F1 = np.array([])
merged_true_answers = [questions[i] + dataset["test"]["output"][i] for i in range(len(questions))]

n_sections = 6

for i in range(n_sections):
  print(i)
  first_index = int(sections[i])
  second_index= int(sections[i+1])
  responses = generate_text_parall_raw(tokenizer, questions[first_index : second_index], max_len)

  P_2, R_2, F1_2 = scorer.score(responses, merged_true_answers[first_index : second_index])
  P = np.append(P, P_2)
  R = np.append(R, R_2)
  F1 = np.append(F1, F1_2)
  print(f"BERTScore Precision : {np.mean(P):.4f}, Recall: {np.mean(R):.4f}, F1: {np.mean(F1):.4f}  #samples: {int((len(questions)/n_splits)*(i+1))}")

In [ ]:
def cum_mean(arr):
  cum_sum = np.cumsum(arr)
  return cum_sum/np.arange(1, len(arr) + 1)

plt.plot(cum_mean(P), color='#2ca02c',label='Precision')
plt.plot(cum_mean(R), color='#d62728',label='Recall')
plt.plot(cum_mean(F1),label='F1 score')

plt.ylabel('Mean scores')
plt.xlabel('Iteration')
plt.title("Mean scores  per iteartion")
plt.legend()
plt.show()

##Conclusions

As visible comparing the BertScore on Precision, Recall and F1 there is a visible improvement due to fine-tuning. Even handcraft valuation shows this result.
Answers provided by the model are quite good and accurate.

# Context-based model: BioBert


For this type of models a context is needed in the training phase and inference phase. The **context** is a sequence of text from which the answer can be extracted, it was retrieved, as previously explained, from MedPub dataset using their API.

BioBERT is a biomedical language representation model designed for biomedical text mining tasks such as biomedical named entity recognition, relation extraction, question answering, etc.

The overall process of pre-training and fine-tuning BioBERT is ashown above.
* First, BioBERT is intialized with weights from BERT, which was pretrained on general domain corpora (English Wikipedia and BooksCorpus).
* Then, BioBERT is pre-trained on biomedical domain corpora (PubMed abstracts and PMC full-text articles).
* Finally, BioBERT is fine-tuned and evaluated on three popular biomedical text mining tasks (NER, RE and QA).

We use Biobert for our dataset, fine tuning it accoording to question-answering task

In [ ]:
TRAIN_MODEL = False
DOWNLOAD_DATASET = False    # Set to True if you want to download the huggingface dataset and re-split it (note that by doing it you will have a different split of the train-test)

## Configurations and imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

path = 'Colab Notebooks/NLP_project'

os.chdir(f'/content/drive/MyDrive/{path}')
os.getcwd()

In [ ]:
!pip install datasets
!pip install -U sentence-transformers
!pip install datasets
!pip install transformers torch tensorflow pandas numpy
!pip3 install accelerate -U
!pip install bert-score


In [ ]:
import pandas as pd
from datasets import load_dataset, Dataset
from datasets import load_from_disk
from bert_score import BERTScorer
from transformers import pipeline
from transformers import AutoTokenizer
from transformers import AutoModelForQuestionAnswering
import numpy as np
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer
from transformers import DefaultDataCollator
from datasets import load_metric
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import pandas as pd
from datasets import Dataset
import matplotlib.pyplot as plt


## Dataset import

On this part the dataset is imported from the source, furthermore the context is extracted from .csv file. There is a small number of questions that doesn't have a context and they were discarded. At the end the join on the two tables is performed

In [ ]:
if DOWNLOAD_DATASET:
  raw_dataset =  load_dataset("medalpaca/medical_meadow_medical_flashcards", download_mode="force_redownload")

  #read the contextes from dataframe and join with answer-questions pairs
  df_context = pd.read_csv('questions_context_dataframe.csv')
  df_context.drop(df_context.columns[[1,2,3,4,5,6]], axis=1, inplace=True)
  raw_dataset_df = raw_dataset['train'].to_pandas()
  merged_df = pd.merge(raw_dataset_df, df_context, left_on='input',right_on='question', how='inner')
  merged_df.drop( merged_df.columns[[2,3]], axis=1, inplace=True)
  dataset = Dataset.from_pandas(merged_df)
  dataset = dataset.train_test_split(test_size=0.1)

  #dataset = dataset["train"].train_test_split(test_size=0.1)
  train_validation_split = dataset["train"].train_test_split(test_size=0.1)
  dataset["train"] = train_validation_split["train"]
  dataset["validation"] = train_validation_split["test"]
  dataset
else:
  dataset = load_from_disk("dataset_biobert")

In [ ]:
if DOWNLOAD_DATASET:
  #data are saved in order not to mix test and training
  dataset.save_to_disk("dataset_biobert")

In [ ]:
dataset = dataset.filter(lambda example: example["input"] != "")

## Train the model

Here we train only Biobert and fine tune it on our dataset, however there is a data we don't have: starting_index: it specifies where the answer is located in the context, since we don't have this information we set it to 0 index, also the ending_index is set to 0

###BioBert

In [ ]:
if TRAIN_MODEL:
  tokenizer = AutoTokenizer.from_pretrained("biobert-base-cased-v1.2")
  output_dir = "2_biobert_qa_model"
  model = AutoModelForQuestionAnswering.from_pretrained("biobert-base-cased-v1.2")
  data_collator = DefaultDataCollator()

  ### TRAINING SET ###
  # Convert the dataset to a dictionary
  data_dict = dataset["train"].to_dict()
  # Create a DataFrame from the dictionary
  df = pd.DataFrame.from_dict(data_dict)

  questions = [q.strip() for q in df["input"]]
  context = [q.strip() for q in df["contexts"]]
  inputs = tokenizer(
          questions,
          context,
          max_length=512,
          truncation="only_second",
          return_offsets_mapping=True,
          padding="max_length",
      )

  offset_mapping = inputs.pop("offset_mapping")

  start_positions = []
  end_positions = []
  answers = df['output']
  for i, offset in enumerate(offset_mapping):
      answer = answers[i]
      start_char = 0
      end_char = 0 + len(answer)
      sequence_ids = inputs.sequence_ids(i)

      # Find the start and end of the context
      idx = 0
      while sequence_ids[idx] != 1:
          idx += 1
      context_start = idx
      while sequence_ids[idx] == 1:
          idx += 1
      context_end = idx - 1

      # If the answer is not fully inside the context, label it (0, 0)
      if offset[context_start][0] > end_char or offset[context_end][1] < start_char:
          start_positions.append(0)
          end_positions.append(0)
      else:
          # Otherwise it's the start and end token positions
          idx = context_start
          while idx <= context_end and offset[idx][0] <= start_char:
              idx += 1
          start_positions.append(idx - 1)

          idx = context_end
          while idx >= context_start and offset[idx][1] >= end_char:
              idx -= 1
          end_positions.append(idx + 1)

  df["start_positions"] = start_positions
  df["end_positions"] = end_positions

  data = {'input_ids': inputs['input_ids'],
          'attention_mask': inputs['attention_mask'],
          'start_positions':start_positions,
          'end_positions': end_positions,
        }
  df = pd.DataFrame(data)
  train = Dataset.from_pandas(df)

  ### VALIDATION SET ###
  # Convert the dataset to a dictionary
  data_dict = dataset["validation"].to_dict()
  # Create a DataFrame from the dictionary
  df = pd.DataFrame.from_dict(data_dict)

  questions = [q.strip() for q in df["input"]]
  context = [q.strip() for q in df["contexts"]]
  inputs = tokenizer(
          questions,
          context,
          max_length=512,
          truncation="only_second",
          return_offsets_mapping=True,
          padding="max_length",
      )

  offset_mapping = inputs.pop("offset_mapping")

  start_positions = []
  end_positions = []
  answers = df['output']
  for i, offset in enumerate(offset_mapping):
      answer = answers[i]
      start_char = 0
      end_char = 0 + len(answer)
      sequence_ids = inputs.sequence_ids(i)

      # Find the start and end of the context
      idx = 0
      while sequence_ids[idx] != 1:
          idx += 1
      context_start = idx
      while sequence_ids[idx] == 1:
          idx += 1
      context_end = idx - 1

      # If the answer is not fully inside the context, label it (0, 0)
      if offset[context_start][0] > end_char or offset[context_end][1] < start_char:
          start_positions.append(0)
          end_positions.append(0)
      else:
          # Otherwise it's the start and end token positions
          idx = context_start
          while idx <= context_end and offset[idx][0] <= start_char:
              idx += 1
          start_positions.append(idx - 1)

          idx = context_end
          while idx >= context_start and offset[idx][1] >= end_char:
              idx -= 1
          end_positions.append(idx + 1)

  df["start_positions"] = start_positions
  df["end_positions"] = end_positions

  data = {'input_ids': inputs['input_ids'],
          'attention_mask': inputs['attention_mask'],
          'start_positions':start_positions,
          'end_positions': end_positions,
        }
  df = pd.DataFrame(data)
  validation = Dataset.from_pandas(df)

In [ ]:

training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="steps",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    report_to=[],
    logging_dir="./logs",
    load_best_model_at_end=True
)

def compute_metrics(p):
    predictions, labels = p.predictions, p.label_ids

    # Convert tuples to numpy arrays
    predictions = np.array(predictions)
    labels = np.array(labels)

    # Assuming your model outputs logits and you want to get predictions
    predictions = np.argmax(predictions, axis=2)
    labels = labels

    # Flatten the predictions and labels
    predictions = predictions.flatten()
    labels = labels.flatten()

    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average='macro')
    recall = recall_score(labels, predictions, average='macro')
    f1 = f1_score(labels, predictions, average='macro')

    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train,
    eval_dataset=validation,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

if TRAIN_MODEL:
  trainer.train()
  trainer.save_model()
  results = trainer.evaluate()
  results

## model evaluation

###BioBert fine-tuning

Inference from the model is performed in batch, test set is splitted into n_splits parts, this is done in order to avoid overloading RAM resources.
And only a part of them is used to make inference

In [ ]:
output_dir = "2_biobert_qa_model"
model = AutoModelForQuestionAnswering.from_pretrained(output_dir)
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.2")

question_answerer = pipeline("question-answering", model=model , tokenizer=tokenizer)
scorer = BERTScorer(lang='en')

n_splits = 160
sections = np.linspace(0, len(dataset["test"]["input"]), n_splits+1)
n_sections = 4

P = np.array([])
R = np.array([])
F1 = np.array([])

for i in range(n_sections):
  print(i)
  first_index = int(sections[i])
  second_index= int(sections[i+1])
  results = question_answerer(question=dataset["test"]["input"][first_index : second_index], context=dataset["test"]["contexts"][first_index : second_index])
  results = pd.DataFrame(results)
  P_2, R_2, F1_2 = scorer.score(list(results["answer"]), dataset["test"]["output"][first_index : second_index])
  P = np.append(P, P_2)
  R = np.append(R, R_2)
  F1 = np.append(F1, F1_2)
  print(f"BERTScore Precision : {np.mean(P):.4f}, Recall: {np.mean(R):.4f}, F1: {np.mean(F1):.4f}  #samples: {int((len(dataset['test']['input'])/n_splits)*(i+1))}")


In [ ]:
def cum_mean(arr):
  cum_sum = np.cumsum(arr)
  return cum_sum/np.arange(1, len(arr) + 1)

plt.plot(cum_mean(P), color='#2ca02c',label='Precision')
plt.plot(cum_mean(R), color='#d62728',label='Recall')
plt.plot(cum_mean(F1),label='F1 score')

plt.ylabel('Mean scores')
plt.xlabel('Iteration')
plt.title("Mean scores  per iteartion")
plt.legend()
plt.show()

### raw model

Here BioBert and is avaluated on the data we have, the context retieved is used to make inference

In [ ]:
### BioBert ###

model_bio = "dmis-lab/biobert-base-cased-v1.2"
model = AutoModelForQuestionAnswering.from_pretrained(model_bio)
tokenizer = AutoTokenizer.from_pretrained(model_bio)

question_answerer = pipeline("question-answering", model=model , tokenizer=tokenizer)
scorer = BERTScorer(lang='en')

n_splits = 160
sections = np.linspace(0, len(dataset["test"]["input"]), n_splits+1)
n_sections = 4

P = np.array([])
R = np.array([])
F1 = np.array([])


for i in range(n_sections):
  print(i)
  first_index = int(sections[i])
  second_index= int(sections[i+1])
  results = question_answerer(question=dataset["test"]["input"][first_index : second_index], context=dataset["test"]["contexts"][first_index : second_index])
  results = pd.DataFrame(results)
  P_2, R_2, F1_2 = scorer.score(list(results["answer"]), dataset["test"]["output"][first_index : second_index])
  P = np.append(P, P_2)
  R = np.append(R, R_2)
  F1 = np.append(F1, F1_2)
  print(f"BERTScore Precision : {np.mean(P):.4f}, Recall: {np.mean(R):.4f}, F1: {np.mean(F1):.4f}  #samples: {int((len(dataset['test']['input'])/n_splits)*(i+1))}")

In [ ]:
def cum_mean(arr):
  cum_sum = np.cumsum(arr)
  return cum_sum/np.arange(1, len(arr) + 1)

plt.plot(cum_mean(P), color='#2ca02c',label='Precision')
plt.plot(cum_mean(R), color='#d62728',label='Recall')
plt.plot(cum_mean(F1),label='F1 score')

plt.ylabel('Mean scores')
plt.xlabel('Iteration')
plt.title("Mean scores  per iteartion")
plt.legend()
plt.show()

##conclusions

Comparing the two models performances have improved thanks to fine tuning however they are lower than other models, this is probably due to the missing starting index, ending index and the quality of the context retrieved.

# Testing

#Models evaluation

Evaluating models in the NLP domain is a challenging task. The primary difficulty in testing Large Language Models (LLMs) stems from the non-deterministic nature of their outputs. Unlike conventional applications where expected outcomes are predictable, LLM-based applications produce varied responses even when given the same inputs.

##Metrics
To overcome this challenge, it's essential to establish metrics that evaluate the accuracy of a generated answer relative to the correct one. In our context, we have identified two key metrics:

* **BERTScore**: Compares the meanings of words in the generated and reference sentences by looking at their context. Unlike simpler methods that just match words or phrases, it considers the full meaning of the sentences using BERT's sophisticated language understanding.
* **Evaluation with Embedding**: Involves representing both the generated and reference sentences as dense vectors using embedding techniques. Subsequently, it calculates the cosine similarity between these vectors to assess their similarity.

We chose these metrics over more statistical ones because in question-answering, generated responses might convey the same meaning as the correct answer but with different wording or additional information. Both BERTScores and embeddings take into account the contextual meaning of sentences when assessing their similarity, making them excellent choices for evaluating our models.

##Qualitative evaluation
Another simple yet effective method for evaluating our model is through manual assessment. We divided our dataset and manually compared the generated answers with the correct ones.

##LLM evaluation
Another approach to evaluate the model could have been to use an LLM trained on evaluation tasks, such as Prometheus. Prometheus is an open-source evaluator LLM specifically designed to assess other LLMs. It stands out for several reasons:
* High Accuracy
* Customizable Scoring
* Open-Source Accessibility

Unfortunately, utilizing Prometheus for evaluation proved unfeasible due to its high computational demands, particularly when employing one LLM to evaluate another. This computational intensity made it impractical to test our model using this method.

##Install and Imports


Let's import everything we need for the execution!

In [ ]:
DOWNLOAD_BIOSENT = False
TEST_MODEL = True

In [ ]:
DATAS_PATH = '/content/drive/MyDrive/Colab Notebooks/NLP/project/Models/Datasets'

In [ ]:
%%capture
# Installs Unsloth, Xformers (Flash Attention) and all other packages!
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
!git clone https://github.com/epfml/sent2vec.git
%cd sent2vec
!make
!pip install .

In [ ]:
!pip install bert-score

In [ ]:
from google.colab import drive

from unsloth import FastLanguageModel

import torch

from datasets import load_dataset
from datasets import Dataset

import os

import sys
import time

from trl import SFTTrainer

from transformers import TrainingArguments
from transformers.utils import logging

from nltk import word_tokenize
from nltk.corpus import stopwords
import nltk

from string import punctuation
from scipy.spatial import distance

from tqdm import tqdm

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from bert_score import BERTScorer

import sent2vec

In [ ]:
sys.setrecursionlimit(4000)

In [ ]:
drive.mount('/content/drive')

In [ ]:
if DOWNLOAD_BIOSENT:
  os.chdir(f'{DATAS_PATH}')
  os.getcwd()
  !wget https://ftp.ncbi.nlm.nih.gov/pub/lu/Suppl/BioSentVec/BioSentVec_PubMed_MIMICIII-bigram_d700.bin

In [ ]:
nltk.download('stopwords')
nltk.download('punkt')

## Model configuration

Let's define some useful constant for our model!

In [ ]:
max_seq_length = 4096   # We use RoPE Scaling to extend it to 4096 with Unsloth!
dtype = None            # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True     # Use 4bit quantization to reduce memory usage. Can be False.

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

##Dataset import
To maintain the integrity of model testing, we pre-saved the dataset split in a dataframe. This ensures that during testing, no training samples are utilized, thereby preserving the model evaluation process.

In [ ]:
if False:
  dataset_path = "medalpaca/medical_meadow_medical_flashcards"
  raw_dataset = load_dataset(dataset_path)
  dataset = raw_dataset["train"].train_test_split(test_size=0.2)
  train_dataset = dataset["train"]
  test_dataset = dataset["test"]
  if False:
    df_train = dataset["train"].to_pandas()
    df_test = dataset["test"].to_pandas()
    os.chdir(f'{DATAS_PATH}')
    os.getcwd()
    df_train.to_csv("train_split.csv", index=False)
    df_test.to_csv("test_split.csv", index=False)
    os.chdir(f'{MODEL_PATH}')
    os.getcwd()
else:
  os.chdir(f'{DATAS_PATH}')
  os.getcwd()
  test_dataset = Dataset.from_pandas(pd.read_csv('test_split.csv'))
  train_dataset = Dataset.from_pandas(pd.read_csv('train_split.csv'))

##Functions

The preprocess_sentence function is utilized to eliminate punctuation and stopwords from the input text.

In [ ]:
stop_words = set(stopwords.words('english'))
def preprocess_sentence(text):
    text = text.replace('/', ' / ')
    text = text.replace('.-', ' .- ')
    text = text.replace('.', ' . ')
    text = text.replace('\'', ' \' ')
    text = text.lower()

    tokens = [token for token in word_tokenize(text) if token not in punctuation and token not in stop_words]

    return ' '.join(tokens)

The generate_answer function utilizes the input model to generate a response to the provided question.

In [ ]:
def generate_answer(model, question):
  # alpaca_prompt = Copied from above
  FastLanguageModel.for_inference(model) # Enable native 2x faster inference
  inputs = tokenizer(
  [
      alpaca_prompt.format(
          "Answer this question truthfully", # instruction
          question, # input
          "", # output - leave this blank for generation!
      )
  ], return_tensors = "pt").to("cuda")

  outputs = model.generate(**inputs, max_new_tokens = 200, use_cache = True, pad_token_id=tokenizer.eos_token_id)
  tokenizer.batch_decode(outputs)
  predicted_answer = tokenizer.batch_decode(outputs[:, inputs['input_ids'].shape[1]:])[0]
  predicted_answer = predicted_answer[:(len(predicted_answer)-5)]
  return predicted_answer

##BioSentVec

Let's now donwload the embedding model!

In [ ]:
if TEST_MODEL:
  model_path = f'{DATAS_PATH}/BioSentVec_PubMed_MIMICIII-bigram_d700.bin'
  testing_model = sent2vec.Sent2vecModel()

The evaluate_answer function is a useful tool that calculates the cosine similarity between the actual text and the generated one.

In [ ]:
if TEST_MODEL:
  def evaluate_answer(real, predicted):
    sentence_vector1 = testing_model.embed_sentence(preprocess_sentence(real))
    sentence_vector2 = testing_model.embed_sentence(preprocess_sentence(predicted))
    cosine_sim = 1 - distance.cosine(sentence_vector1[0], sentence_vector2[0])
    return cosine_sim

In [ ]:
if TEST_MODEL:
  try:
      testing_model.load_model(model_path)
  except Exception as e:
      print(e)
  print('model successfully loaded')

##Models

Every model were tested both before and after fine-tuning, utilizing both BioSentVec embeddings and BERTscore.

To expedite the evaluation process, a stopping criterion was employed. If, in the last 20 iterations of the for loop calculating the mean score, the mean does not change significantly, the execution stops, and the score is printed.

Additionally, the models were evaluated for execution time in answering questions.

##Mistral

###Testing base model

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-v0.3-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

In [ ]:
EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass
train_dataset = train_dataset.map(formatting_prompts_func, batched = True)
test_dataset = test_dataset.map(formatting_prompts_func, batched = True)

In [ ]:
index = 300
#@title Manual testing
real_answer = test_dataset[index]["output"]
generated_answer = generate_answer(model, test_dataset[index]["input"])
question = test_dataset[index]["input"]

print("Question: ", question)
print("Generated answer: ", generated_answer)
print("Real answer: ", real_answer)

In [ ]:
if TEST_MODEL:
  index = 5000

  real_answer = test_dataset[index]["output"]
  generated_answer = generate_answer(model, test_dataset[index]["input"])
  question = test_dataset[index]["input"]

  print("Question: ", question)
  print("Generated answer: ", generated_answer)
  print("Real answer: ", real_answer)
  print("Cosine similarity: ", evaluate_answer(real_answer, generated_answer))

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_list = []
  patience = 20
  k = 0
  for i in range(len(test_dataset)):
    real_answer = test_dataset[i]["output"]
    question = test_dataset[i]["input"]
    generated_answer = generate_answer(model, question)
    if (generated_answer) is not None and (real_answer) is not None:
      score = evaluate_answer(real_answer, generated_answer)
      if i == 0:
        mean = score
      sum += score
      n_try += 1
      old_mean = mean
      mean = sum/n_try
      mean_list.append(mean)

      if abs(mean - old_mean) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot(mean_list, label='cosine similiarity')
  plt.ylabel('Mean cosine similiarity')
  plt.xlabel('Iteration')
  plt.title("Mean cosine similarity per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print(f'The average cosine similarity is: {mean}')

In [ ]:
# Initialize the BERTScorer object
if TEST_MODEL:
  scorer = BERTScorer(lang='en')

In [ ]:
if TEST_MODEL:
  sum = [0, 0, 0]
  n_try = 0
  mean_score_list = []
  patience = 20
  k = 0
  for i in range(len(test_dataset)):
    real_answer = test_dataset[i]["output"]
    question = test_dataset[i]["input"]

    generated_answer = generate_answer(model, question)
    if (generated_answer) is not None and (real_answer) is not None:
      P, R, F1 = scorer.score([generated_answer], [real_answer])
      score = [float(P), float(R), float(F1)]
      if i == 0:
        mean_score = score
      sum = [x + y for x, y in zip(sum, score)]
      n_try += 1
      old_mean = mean_score
      mean_score = [(x / n_try) for x in sum]
      mean_score_list.append(mean_score)

      if abs(mean_score[0] - old_mean[0]) < 0.001 and abs(mean_score[1] - old_mean[1]) < 0.001 and abs(mean_score[2] - old_mean[2]) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean_score} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot([score[0] for score in mean_score_list], color='#2ca02c',label='Precision')
  plt.plot([score[1] for score in mean_score_list], color='#d62728',label='Recall')
  plt.plot([score[2] for score in mean_score_list],label='F1 score')

  plt.ylabel('Mean scores')
  plt.xlabel('Iteration')
  plt.title("Mean scores  per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print("Precision: ", mean_score[0])
  print("Recall: ", mean_score[1])
  print("F1 score: ", mean_score[2])

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_time_list = []
  patience = 20
  k = 0
  for i in tqdm(range(200)):
    question = test_dataset[i]["input"]

    start = time.time()
    generated_answer = generate_answer(model, question)
    end = time.time()

    score_time = end-start
    if i == 0:
      mean_time = score_time
    sum += score_time
    n_try += 1
    old_mean_time = mean_time
    mean_time = sum/n_try
    mean_time_list.append(mean_time)

In [ ]:
plt.plot(mean_time_list)
plt.plot([mean_time for i in mean_time_list], '--', color='blue', label='Mean execution time', linewidth=1)
plt.ylabel('Mean execution time')
plt.xlabel('Iteration')
plt.title("Mean execution time per iteartion")
plt.legend()
plt.show()

In [ ]:
if TEST_MODEL:
  print(f'The average execution time is: {mean_time}')

###Testing fine-tuned model

In [ ]:
MODEL_PATH = '/content/drive/MyDrive/Colab Notebooks/NLP/project/Models/Mistral'

In [ ]:
os.chdir(f'{MODEL_PATH}')
os.getcwd()

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_model",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

In [ ]:
index = 300
#@title Manual testing
real_answer = test_dataset[index]["output"]
generated_answer = generate_answer(test_dataset[index]["input"])
question = test_dataset[index]["input"]

print("Question: ", question)
print("Generated answer: ", generated_answer)
print("Real answer: ", real_answer)

In [ ]:
if TEST_MODEL:
  index = 5000

  real_answer = test_dataset[index]["output"]
  generated_answer = generate_answer(test_dataset[index]["input"])
  question = test_dataset[index]["input"]

  print("Question: ", question)
  print("Generated answer: ", generated_answer)
  print("Real answer: ", real_answer)
  print("Cosine similarity: ", evaluate_answer(real_answer, generated_answer))

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_list = []
  patience = 20
  k = 0
  for i in range(len(test_dataset)):
    real_answer = test_dataset[i]["output"]
    question = test_dataset[i]["input"]
    generated_answer = generate_answer(question)
    if (generated_answer) is not None and (real_answer) is not None:
      score = evaluate_answer(real_answer, generated_answer)
      if i == 0:
        mean = score
      sum += score
      n_try += 1
      old_mean = mean
      mean = sum/n_try
      mean_list.append(mean)

      if abs(mean - old_mean) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot(mean_list, label='cosine similiarity')
  plt.ylabel('Mean cosine similiarity')
  plt.xlabel('Iteration')
  plt.title("Mean cosine similarity per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print(f'The average cosine similarity is: {mean}')

In [ ]:
# Initialize the BERTScorer object
if TEST_MODEL:
  scorer = BERTScorer(lang='en')

In [ ]:
if TEST_MODEL:
  sum = [0, 0, 0]
  n_try = 0
  mean_score_list = []
  patience = 20
  k = 0
  for i in range(len(test_dataset)):
    real_answer = test_dataset[i]["output"]
    question = test_dataset[i]["input"]

    generated_answer = generate_answer(question)
    if (generated_answer) is not None and (real_answer) is not None:
      P, R, F1 = scorer.score([generated_answer], [real_answer])
      score = [float(P), float(R), float(F1)]
      if i == 0:
        mean_score = score
      sum = [x + y for x, y in zip(sum, score)]
      n_try += 1
      old_mean = mean_score
      mean_score = [(x / n_try) for x in sum]
      mean_score_list.append(mean_score)

      if abs(mean_score[0] - old_mean[0]) < 0.001 and abs(mean_score[1] - old_mean[1]) < 0.001 and abs(mean_score[2] - old_mean[2]) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean_score} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot([score[0] for score in mean_score_list], color='#2ca02c',label='Precision')
  plt.plot([score[1] for score in mean_score_list], color='#d62728',label='Recall')
  plt.plot([score[2] for score in mean_score_list],label='F1 score')

  plt.ylabel('Mean scores')
  plt.xlabel('Iteration')
  plt.title("Mean scores  per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print("Precision: ", mean_score[0])
  print("Recall: ", mean_score[1])
  print("F1 score: ", mean_score[2])

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_time_list = []
  patience = 20
  k = 0
  for i in tqdm(range(200)):
    question = test_dataset[i]["input"]

    start = time.time()
    generated_answer = generate_answer(question)
    end = time.time()

    score_time = end-start
    if i == 0:
      mean_time = score_time
    sum += score_time
    n_try += 1
    old_mean_time = mean_time
    mean_time = sum/n_try
    mean_time_list.append(mean_time)

In [ ]:
plt.plot(mean_time_list)
plt.plot([mean_time for i in mean_time_list], '--', color='blue', label='Mean execution time', linewidth=1)
plt.ylabel('Mean execution time')
plt.xlabel('Iteration')
plt.title("Mean execution time per iteartion")
plt.legend()
plt.show()

In [ ]:
if TEST_MODEL:
  print(f'The average execution time is: {mean_time}')

##Gemma

###Testing base model

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-7b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

In [ ]:
EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass
train_dataset = train_dataset.map(formatting_prompts_func, batched = True)
test_dataset = test_dataset.map(formatting_prompts_func, batched = True)

In [ ]:
question

In [ ]:
index = 200
#@title Manual testing
real_answer = test_dataset[index]["output"]
question = test_dataset[index]["input"]
generated_answer = generate_answer(model, question)


print("Question: ", question)
print("Generated answer: ", generated_answer)
print("Real answer: ", real_answer)

In [ ]:
if TEST_MODEL:
  index = 200

  real_answer = test_dataset[index]["output"]
  generated_answer = generate_answer(model, test_dataset[index]["input"])
  question = test_dataset[index]["input"]

  print("Question: ", question)
  print("Generated answer: ", generated_answer)
  print("Real answer: ", real_answer)
  print("Cosine similarity: ", evaluate_answer(real_answer, generated_answer))

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_list = []
  patience = 20
  k = 0
  for i in range(len(test_dataset)):
    real_answer = test_dataset[i]["output"]
    question = test_dataset[i]["input"]
    generated_answer = generate_answer(model, question)
    if i == 30: break
    if (generated_answer) is not None and (real_answer) is not None:
      score = evaluate_answer(real_answer, generated_answer)
      if i == 0:
        mean = score
      sum += score
      n_try += 1
      old_mean = mean
      mean = sum/n_try
      mean_list.append(mean)

      if abs(mean - old_mean) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot(mean_list, label='cosine similiarity')
  plt.ylabel('Mean cosine similiarity')
  plt.xlabel('Iteration')
  plt.title("Mean cosine similarity per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print(f'The average cosine similarity is: {mean}')

In [ ]:
# Initialize the BERTScorer object
if TEST_MODEL:
  scorer = BERTScorer(lang='en')

In [ ]:
if TEST_MODEL:
  sum = [0, 0, 0]
  n_try = 0
  mean_score_list = []
  patience = 20
  k = 0

  for i in range(len(test_dataset)):
    real_answer = test_dataset[i]["output"]
    question = test_dataset[i]["input"]
    generated_answer = generate_answer(model, question)
    if (generated_answer) is not None and (real_answer) is not None:
      P, R, F1 = scorer.score([generated_answer], [real_answer])
      score = [float(P), float(R), float(F1)]
      if i == 0:
        mean_score = score
      sum = [x + y for x, y in zip(sum, score)]
      n_try += 1
      old_mean = mean_score
      mean_score = [(x / n_try) for x in sum]
      mean_score_list.append(mean_score)

      if abs(mean_score[0] - old_mean[0]) < 0.001 and abs(mean_score[1] - old_mean[1]) < 0.001 and abs(mean_score[2] - old_mean[2]) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean_score} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot([score[0] for score in mean_score_list], color='#2ca02c',label='Precision')
  plt.plot([score[1] for score in mean_score_list], color='#d62728',label='Recall')
  plt.plot([score[2] for score in mean_score_list],label='F1 score')

  plt.ylabel('Mean scores')
  plt.xlabel('Iteration')
  plt.title("Mean scores  per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print("Precision: ", mean_score[0])
  print("Recall: ", mean_score[1])
  print("F1 score: ", mean_score[2])

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_time_list = []
  patience = 20
  k = 0
  for i in tqdm(range(200)):
    question = test_dataset[i]["input"]
    start = time.time()
    generated_answer = generate_answer(model, question)
    end = time.time()

    score_time = end-start
    if i == 0:
      mean_time = score_time
    sum += score_time
    n_try += 1
    old_mean_time = mean_time
    mean_time = sum/n_try
    mean_time_list.append(mean_time)

In [ ]:
plt.plot(mean_time_list)
plt.plot([mean_time for i in mean_time_list], '--', color='blue', label='Mean execution time', linewidth=1)
plt.ylabel('Mean execution time')
plt.xlabel('Iteration')
plt.title("Mean execution time per iteartion")
plt.legend()
plt.show()

In [ ]:
if TEST_MODEL:
  print(f'The average execution time is: {mean_time}')

###Testing fine-tuned model

In [ ]:
MODEL_PATH = '/content/drive/MyDrive/Colab Notebooks/NLP/project/Models/Gemma'

In [ ]:
os.chdir(f'{MODEL_PATH}')
os.getcwd()

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_model",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

In [ ]:
index = 300
#@title Manual testing
real_answer = test_dataset[index]["output"]
generated_answer = generate_answer(test_dataset[index]["input"])
question = test_dataset[index]["input"]

print("Question: ", question)
print("Generated answer: ", generated_answer)
print("Real answer: ", real_answer)

In [ ]:
if TEST_MODEL:
  index = 5000

  real_answer = test_dataset[index]["output"]
  generated_answer = generate_answer(test_dataset[index]["input"])
  question = test_dataset[index]["input"]

  print("Question: ", question)
  print("Generated answer: ", generated_answer)
  print("Real answer: ", real_answer)
  print("Cosine similarity: ", evaluate_answer(real_answer, generated_answer))

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_list = []
  patience = 20
  k = 0
  for i in range(len(test_dataset)):
    real_answer = test_dataset[i]["output"]
    question = test_dataset[i]["input"]
    generated_answer = generate_answer(question)
    if (generated_answer) is not None and (real_answer) is not None:
      score = evaluate_answer(real_answer, generated_answer)
      if i == 0:
        mean = score
      sum += score
      n_try += 1
      old_mean = mean
      mean = sum/n_try
      mean_list.append(mean)

      if abs(mean - old_mean) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot(mean_list, label='cosine similiarity')
  plt.ylabel('Mean cosine similiarity')
  plt.xlabel('Iteration')
  plt.title("Mean cosine similarity per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print(f'the average cosine similarity is: {mean}')

In [ ]:
# Initialize the BERTScorer object
if TEST_MODEL:
  scorer = BERTScorer(lang='en')

In [ ]:
if TEST_MODEL:
  sum = [0, 0, 0]
  n_try = 0
  mean_score_list = []
  patience = 20
  k = 0
  for i in range(len(test_dataset)):
    real_answer = test_dataset[i]["output"]
    question = test_dataset[i]["input"]

    generated_answer = generate_answer(question)
    if (generated_answer) is not None and (real_answer) is not None:
      P, R, F1 = scorer.score([generated_answer], [real_answer])
      score = [float(P), float(R), float(F1)]
      if i == 0:
        mean_score = score
      sum = [x + y for x, y in zip(sum, score)]
      n_try += 1
      old_mean = mean_score
      mean_score = [(x / n_try) for x in sum]
      mean_score_list.append(mean_score)

      if abs(mean_score[0] - old_mean[0]) < 0.001 and abs(mean_score[1] - old_mean[1]) < 0.001 and abs(mean_score[2] - old_mean[2]) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean_score} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot([score[0] for score in mean_score_list], color='#2ca02c',label='Precision')
  plt.plot([score[1] for score in mean_score_list], color='#d62728',label='Recall')
  plt.plot([score[2] for score in mean_score_list],label='F1 score')

  plt.ylabel('Mean scores')
  plt.xlabel('Iteration')
  plt.title("Mean scores  per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print("Precision: ", mean_score[0])
  print("Recall: ", mean_score[1])
  print("F1 score: ", mean_score[2])

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_time_list = []
  patience = 20
  k = 0
  for i in tqdm(range(200)):
    question = test_dataset[i]["input"]

    start = time.time()
    generated_answer = generate_answer(question)
    end = time.time()

    score_time = end-start
    if i == 0:
      mean_time = score_time
    sum += score_time
    n_try += 1
    old_mean_time = mean_time
    mean_time = sum/n_try
    mean_time_list.append(mean_time)

In [ ]:
plt.plot(mean_time_list)
plt.plot([mean_time for i in mean_time_list], '--', color='blue', label='Mean execution time', linewidth=1)
plt.ylabel('Mean execution time')
plt.xlabel('Iteration')
plt.title("Mean execution time per iteartion")
plt.legend()
plt.show()

In [ ]:
if TEST_MODEL:
  print(f'The average execution time is: {mean_time}')

##Llama

###Testing base model

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

In [ ]:
EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass
train_dataset = train_dataset.map(formatting_prompts_func, batched = True)
test_dataset = test_dataset.map(formatting_prompts_func, batched = True)

In [ ]:
index = 200
#@title Manual testing
real_answer = test_dataset[index]["output"]
generated_answer = generate_answer(model, test_dataset[index]["input"])
question = test_dataset[index]["input"]

print("Question: ", question)
print("Generated answer: ", generated_answer)
print("Real answer: ", real_answer)

In [ ]:
if TEST_MODEL:
  index = 5000

  real_answer = test_dataset[index]["output"]
  generated_answer = generate_answer(model, test_dataset[index]["input"])
  question = test_dataset[index]["input"]

  print("Question: ", question)
  print("Generated answer: ", generated_answer)
  print("Real answer: ", real_answer)
  print("Cosine similarity: ", evaluate_answer(real_answer, generated_answer))

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_list = []
  patience = 20
  k = 0
  for i in range(len(test_dataset)):
    real_answer = test_dataset[i]["output"]
    question = test_dataset[i]["input"]
    generated_answer = generate_answer(model, question)
    if (generated_answer) is not None and (real_answer) is not None:
      score = evaluate_answer(real_answer, generated_answer)
      if i == 0:
        mean = score
      sum += score
      n_try += 1
      old_mean = mean
      mean = sum/n_try
      mean_list.append(mean)

      if abs(mean - old_mean) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot(mean_list, label='cosine similiarity')
  plt.ylabel('Mean cosine similiarity')
  plt.xlabel('Iteration')
  plt.title("Mean cosine similarity per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print(f'The average cosine similarity is: {mean}')

In [ ]:
# Initialize the BERTScorer object
if TEST_MODEL:
  scorer = BERTScorer(lang='en')

In [ ]:
if TEST_MODEL:
  sum = [0, 0, 0]
  n_try = 0
  mean_score_list = []
  patience = 20
  k = 0
  for i in range(len(test_dataset)):
    real_answer = test_dataset[i]["output"]
    question = test_dataset[i]["input"]

    generated_answer = generate_answer(model, question)
    if (generated_answer) is not None and (real_answer) is not None:
      P, R, F1 = scorer.score([generated_answer], [real_answer])
      score = [float(P), float(R), float(F1)]
      if i == 0:
        mean_score = score
      sum = [x + y for x, y in zip(sum, score)]
      n_try += 1
      old_mean = mean_score
      mean_score = [(x / n_try) for x in sum]
      mean_score_list.append(mean_score)

      if abs(mean_score[0] - old_mean[0]) < 0.001 and abs(mean_score[1] - old_mean[1]) < 0.001 and abs(mean_score[2] - old_mean[2]) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean_score} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot([score[0] for score in mean_score_list], color='#2ca02c',label='Precision')
  plt.plot([score[1] for score in mean_score_list], color='#d62728',label='Recall')
  plt.plot([score[2] for score in mean_score_list],label='F1 score')

  plt.ylabel('Mean scores')
  plt.xlabel('Iteration')
  plt.title("Mean scores  per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print("Precision: ", mean_score[0])
  print("Recall: ", mean_score[1])
  print("F1 score: ", mean_score[2])

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_time_list = []
  patience = 20
  k = 0
  for i in tqdm(range(200)):
    question = test_dataset[i]["input"]

    start = time.time()
    generated_answer = generate_answer(model, question)
    end = time.time()

    score_time = end-start
    if i == 0:
      mean_time = score_time
    sum += score_time
    n_try += 1
    old_mean_time = mean_time
    mean_time = sum/n_try
    mean_time_list.append(mean_time)

In [ ]:
plt.plot(mean_time_list)
plt.plot([mean_time for i in mean_time_list], '--', color='blue', label='Mean execution time', linewidth=1)
plt.ylabel('Mean execution time')
plt.xlabel('Iteration')
plt.title("Mean execution time per iteartion")
plt.legend()
plt.show()

In [ ]:
if TEST_MODEL:
  print(f'The average execution time is: {mean_time}')

###Testing fine-tuned model

In [ ]:
MODEL_PATH = '/content/drive/MyDrive/Colab Notebooks/NLP/project/Models/llama'

In [ ]:
os.chdir(f'{MODEL_PATH}')
os.getcwd()

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_model",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

In [ ]:
index = 300
#@title Manual testing
real_answer = test_dataset[index]["output"]
generated_answer = generate_answer(test_dataset[index]["input"])
question = test_dataset[index]["input"]

print("Question: ", question)
print("Generated answer: ", generated_answer)
print("Real answer: ", real_answer)

In [ ]:
if TEST_MODEL:
  index = 5000

  real_answer = test_dataset[index]["output"]
  generated_answer = generate_answer(test_dataset[index]["input"])
  question = test_dataset[index]["input"]

  print("Question: ", question)
  print("Generated answer: ", generated_answer)
  print("Real answer: ", real_answer)
  print("Cosine similarity: ", evaluate_answer(real_answer, generated_answer))

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_list = []
  patience = 20
  k = 0
  for i in range(len(test_dataset)):
    real_answer = test_dataset[i]["output"]
    question = test_dataset[i]["input"]
    generated_answer = generate_answer(question)
    if (generated_answer) is not None and (real_answer) is not None:
      score = evaluate_answer(real_answer, generated_answer)
      if i == 0:
        mean = score
      sum += score
      n_try += 1
      old_mean = mean
      mean = sum/n_try
      mean_list.append(mean)

      if abs(mean - old_mean) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot(mean_list, label='cosine similiarity')
  plt.ylabel('Mean cosine similiarity')
  plt.xlabel('Iteration')
  plt.title("Mean cosine similarity per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print(f'The average cosine similarity is: {mean}')

In [ ]:
# Initialize the BERTScorer object
if TEST_MODEL:
  scorer = BERTScorer(lang='en')

In [ ]:
if TEST_MODEL:
  sum = [0, 0, 0]
  n_try = 0
  mean_score_list = []
  patience = 20
  k = 0
  for i in range(len(test_dataset)):
    real_answer = test_dataset[i]["output"]
    question = test_dataset[i]["input"]

    generated_answer = generate_answer(question)
    if (generated_answer) is not None and (real_answer) is not None:
      P, R, F1 = scorer.score([generated_answer], [real_answer])
      score = [float(P), float(R), float(F1)]
      if i == 0:
        mean_score = score
      sum = [x + y for x, y in zip(sum, score)]
      n_try += 1
      old_mean = mean_score
      mean_score = [(x / n_try) for x in sum]
      mean_score_list.append(mean_score)

      if abs(mean_score[0] - old_mean[0]) < 0.001 and abs(mean_score[1] - old_mean[1]) < 0.001 and abs(mean_score[2] - old_mean[2]) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean_score} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot([score[0] for score in mean_score_list], color='#2ca02c',label='Precision')
  plt.plot([score[1] for score in mean_score_list], color='#d62728',label='Recall')
  plt.plot([score[2] for score in mean_score_list],label='F1 score')

  plt.ylabel('Mean scores')
  plt.xlabel('Iteration')
  plt.title("Mean scores  per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print("Precision: ", mean_score[0])
  print("Recall: ", mean_score[1])
  print("F1 score: ", mean_score[2])

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_time_list = []
  patience = 20
  k = 0
  for i in tqdm(range(200)):
    question = test_dataset[i]["input"]

    start = time.time()
    generated_answer = generate_answer(question)
    end = time.time()

    score_time = end-start
    if i == 0:
      mean_time = score_time
    sum += score_time
    n_try += 1
    old_mean_time = mean_time
    mean_time = sum/n_try
    mean_time_list.append(mean_time)

In [ ]:
if TEST_MODEL:
  plt.plot(mean_time_list)
  plt.plot([mean_time for i in mean_time_list], '--', color='blue', label='Mean', linewidth=1)
  plt.ylabel('Mean execution time')
  plt.xlabel('Iteration')
  plt.title("Mean execution time per iteartion")
  plt.show()

In [ ]:
if TEST_MODEL:
  print(f'The average execution time is: {mean_time}')

##Tiny Llama

###Testing base model

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/tinyllama-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

In [ ]:
EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass
train_dataset = train_dataset.map(formatting_prompts_func, batched = True)
test_dataset = test_dataset.map(formatting_prompts_func, batched = True)

In [ ]:
index = 300
#@title Manual testing
real_answer = test_dataset[index]["output"]
generated_answer = generate_answer(model, test_dataset[index]["input"])
question = test_dataset[index]["input"]

print("Question: ", question)
print("Generated answer: ", generated_answer)
print("Real answer: ", real_answer)

In [ ]:
if TEST_MODEL:
  index = 5000

  real_answer = test_dataset[index]["output"]
  generated_answer = generate_answer(model, test_dataset[index]["input"])
  question = test_dataset[index]["input"]

  print("Question: ", question)
  print("Generated answer: ", generated_answer)
  print("Real answer: ", real_answer)
  print("Cosine similarity: ", evaluate_answer(real_answer, generated_answer))

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_list = []
  patience = 20
  k = 0
  for i in range(len(test_dataset)):
    real_answer = test_dataset[i]["output"]
    question = test_dataset[i]["input"]
    generated_answer = generate_answer(model, question)
    if (generated_answer) is not None and (real_answer) is not None:
      score = evaluate_answer(real_answer, generated_answer)
      if i == 0:
        mean = score
      sum += score
      n_try += 1
      old_mean = mean
      mean = sum/n_try
      mean_list.append(mean)

      if abs(mean - old_mean) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot(mean_list, label='cosine similiarity')
  plt.ylabel('Mean cosine similiarity')
  plt.xlabel('Iteration')
  plt.title("Mean cosine similarity per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print(f'The average cosine similarity is: {mean}')

In [ ]:
# Initialize the BERTScorer object
if TEST_MODEL:
  scorer = BERTScorer(lang='en')

In [ ]:
if TEST_MODEL:
  sum = [0, 0, 0]
  n_try = 0
  mean_score_list = []
  patience = 20
  k = 0
  for i in range(len(test_dataset)):
    real_answer = test_dataset[i]["output"]
    question = test_dataset[i]["input"]

    generated_answer = generate_answer(model, question)
    if (generated_answer) is not None and (real_answer) is not None:
      P, R, F1 = scorer.score([generated_answer], [real_answer])
      score = [float(P), float(R), float(F1)]
      if i == 0:
        mean_score = score
      sum = [x + y for x, y in zip(sum, score)]
      n_try += 1
      old_mean = mean_score
      mean_score = [(x / n_try) for x in sum]
      mean_score_list.append(mean_score)

      if abs(mean_score[0] - old_mean[0]) < 0.001 and abs(mean_score[1] - old_mean[1]) < 0.001 and abs(mean_score[2] - old_mean[2]) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean_score} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot([score[0] for score in mean_score_list], color='#2ca02c',label='Precision')
  plt.plot([score[1] for score in mean_score_list], color='#d62728',label='Recall')
  plt.plot([score[2] for score in mean_score_list],label='F1 score')

  plt.ylabel('Mean scores')
  plt.xlabel('Iteration')
  plt.title("Mean scores  per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print("Precision: ", mean_score[0])
  print("Recall: ", mean_score[1])
  print("F1 score: ", mean_score[2])

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_time_list = []
  patience = 20
  k = 0
  for i in tqdm(range(200)):
    question = test_dataset[i]["input"]

    start = time.time()
    generated_answer = generate_answer(model, question)
    end = time.time()

    score_time = end-start
    if i == 0:
      mean_time = score_time
    sum += score_time
    n_try += 1
    old_mean_time = mean_time
    mean_time = sum/n_try
    mean_time_list.append(mean_time)

In [ ]:
plt.plot(mean_time_list)
plt.plot([mean_time for i in mean_time_list], '--', color='blue', label='Mean execution time', linewidth=1)
plt.ylabel('Mean execution time')
plt.xlabel('Iteration')
plt.title("Mean execution time per iteartion")
plt.legend()
plt.show()

In [ ]:
if TEST_MODEL:
  print(f'The average execution time is: {mean_time}')

###Testing fine-tuned model

In [ ]:
MODEL_PATH = '/content/drive/MyDrive/Colab Notebooks/NLP/project/Models/Tiny llama'

In [ ]:
os.chdir(f'{MODEL_PATH}')
os.getcwd()

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_model",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

In [ ]:
index = 300
#@title Manual testing
real_answer = test_dataset[index]["output"]
generated_answer = generate_answer(test_dataset[index]["input"])
question = test_dataset[index]["input"]

print("Question: ", question)
print("Generated answer: ", generated_answer)
print("Real answer: ", real_answer)

In [ ]:
if TEST_MODEL:
  index = 5000

  real_answer = test_dataset[index]["output"]
  generated_answer = generate_answer(test_dataset[index]["input"])
  question = test_dataset[index]["input"]

  print("Question: ", question)
  print("Generated answer: ", generated_answer)
  print("Real answer: ", real_answer)
  print("Cosine similarity: ", evaluate_answer(real_answer, generated_answer))

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_list = []
  patience = 20
  k = 0
  for i in range(len(test_dataset)):
    real_answer = test_dataset[i]["output"]
    question = test_dataset[i]["input"]
    generated_answer = generate_answer(question)
    if (generated_answer) is not None and (real_answer) is not None:
      score = evaluate_answer(real_answer, generated_answer)
      if i == 0:
        mean = score
      sum += score
      n_try += 1
      old_mean = mean
      mean = sum/n_try
      mean_list.append(mean)

      if abs(mean - old_mean) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot(mean_list, label='cosine similiarity')
  plt.ylabel('Mean cosine similiarity')
  plt.xlabel('Iteration')
  plt.title("Mean cosine similarity per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print(f'the average cosine similarity is: {mean}')

In [ ]:
# Initialize the BERTScorer object
if TEST_MODEL:
  scorer = BERTScorer(lang='en')

In [ ]:
if TEST_MODEL:
  sum = [0, 0, 0]
  n_try = 0
  mean_score_list = []
  patience = 20
  k = 0
  for i in range(len(test_dataset)):
    real_answer = test_dataset[i]["output"]
    question = test_dataset[i]["input"]

    generated_answer = generate_answer(question)
    if (generated_answer) is not None and (real_answer) is not None:
      P, R, F1 = scorer.score([generated_answer], [real_answer])
      score = [float(P), float(R), float(F1)]
      if i == 0:
        mean_score = score
      sum = [x + y for x, y in zip(sum, score)]
      n_try += 1
      old_mean = mean_score
      mean_score = [(x / n_try) for x in sum]
      mean_score_list.append(mean_score)

      if abs(mean_score[0] - old_mean[0]) < 0.001 and abs(mean_score[1] - old_mean[1]) < 0.001 and abs(mean_score[2] - old_mean[2]) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean_score} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot([mean_score[0] for score in mean_score_list], color='#2ca02c',label='Precision')
  plt.plot([mean_score[1] for score in mean_score_list], color='#d62728',label='Recall')
  plt.plot([mean_score[2] for score in mean_score_list],label='F1 score')

  plt.ylabel('Mean scores')
  plt.xlabel('Iteration')
  plt.title("Mean scores  per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print("Precision: ", mean_score[0])
  print("Recall: ", mean_score[1])
  print("F1 score: ", mean_score[2])

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_time_list = []
  patience = 20
  k = 0
  for i in tqdm(range(200)):
    question = test_dataset[i]["input"]

    start = time.time()
    generated_answer = generate_answer(question)
    end = time.time()

    score_time = end-start
    if i == 0:
      mean_time = score_time
    sum += score_time
    n_try += 1
    old_mean_time = mean_time
    mean_time = sum/n_try
    mean_time_list.append(mean_time)

In [ ]:
if TEST_MODEL:
  plt.plot(mean_time_list)
  plt.plot([mean_time for i in mean_time_list], '--', color='blue', label='Mean', linewidth=1)
  plt.ylabel('Mean execution time')
  plt.xlabel('Iteration')
  plt.title("Mean execution time per iteartion")
  plt.show()

In [ ]:
if TEST_MODEL:
  print(f'The average execution time is: {mean_time}')

##GPT-2

###GPT-2 fine tuned

In [ ]:

def load_tokenizer(tokenizer_path):
    tokenizer = GPT2Tokenizer.from_pretrained(tokenizer_path)
    return tokenizer

def load_model(model_path):
    model = GPT2LMHeadModel.from_pretrained(model_path)
    return model

def generate_text_parall(tokenizer, sequence, max_length):
    tokenizer.pad_token = tokenizer.eos_token

    ids = tokenizer(sequence, return_tensors='pt', padding=True ,truncation=True)

    final_outputs = model.generate(
        **ids,
        do_sample=True,
        max_length=max_length,
        pad_token_id=model.config.eos_token_id,
        top_k=50,
        top_p=0.95,
    )

    return [ tokenizer.decode(x, skip_special_tokens=True).split("[Q]")[1].split("[A]")[1] for x in final_outputs ]

#retrieve model and tokenizer
model_path = "medium-val-gpt2-custom_q_and_a"
model = load_model(model_path)
tokenizer = load_tokenizer(model_path)
tokenizer.padding_side ='left'

scorer = BERTScorer(lang='en')

In [ ]:
sum = 0
n_try = 0
mean_list = []
patience = 20
k = 0
for i in range(len(test_dataset)):
  real_answer = test_dataset[i]["output"]
  question = test_dataset[i]["input"]
  generated_answer = generate_answer(model, question)
  if (generated_answer) is not None and (real_answer) is not None:
    score = evaluate_answer(real_answer, generated_answer)
    if i == 0:
      mean = score
    sum += score
    n_try += 1
    old_mean = mean
    mean = sum/n_try
    mean_list.append(mean)

    if abs(mean - old_mean) < 0.001:
      k += 1
      if k == patience:
        print(f'Early stopping at iteration: {i}')
        break
    else:
      k=0
    print(f'Iteration: {i}, Mean: {mean} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot(mean_list, label='cosine similiarity')
  plt.ylabel('Mean cosine similiarity')
  plt.xlabel('Iteration')
  plt.title("Mean cosine similarity per iteartion")
  plt.legend()
  plt.show()

Test evaluation with BertScore

In [ ]:
max_len = 150
questions = ["[Q]" + x for x in dataset["test"]["input"]]

In [ ]:
n_splits = 160
sections = np.linspace(0, len(questions), n_splits+1)
n_sections = 3

In [ ]:
P = np.array([])
R = np.array([])
F1 = np.array([])

for i in range(n_sections):
  print(i)
  first_index = int(sections[i])
  second_index= int(sections[i+1])
  responses = generate_text_parall(tokenizer, questions[first_index : second_index], max_len)

  P_2, R_2, F1_2 = scorer.score(responses, dataset["test"]["output"][first_index : second_index])
  P = np.append(P, P_2)
  R = np.append(R, R_2)
  F1 = np.append(F1, F1_2)
  print(f"BERTScore Precision : {np.mean(P):.4f}, Recall: {np.mean(R):.4f}, F1: {np.mean(F1):.4f}  #samples: {int((len(questions)/n_splits)*(i+1))}")


In [ ]:
def cum_mean(arr):
  cum_sum = np.cumsum(arr)
  return cum_sum/np.arange(1, len(arr) + 1)

plt.plot(cum_mean(P), color='#2ca02c',label='Precision')
plt.plot(cum_mean(R), color='#d62728',label='Recall')
plt.plot(cum_mean(F1),label='F1 score')

plt.ylabel('Mean scores')
plt.xlabel('Iteration')
plt.title("Mean scores  per iteartion")
plt.legend()
plt.show()

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_list = []
  patience = 20
  k = 0
  for i in range(len(gpt_true)):
    real_answer = gpt_true[i]
    generated_answer = gpt_generated[i]
    if (generated_answer) is not None and (real_answer) is not None:
      score = evaluate_answer(real_answer, generated_answer)
      if i == 0:
        mean = score
      sum += score
      n_try += 1
      old_mean = mean
      mean = sum/n_try
      mean_list.append(mean)

      if abs(mean - old_mean) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot(mean_list, label='cosine similiarity')
  plt.ylabel('Mean cosine similiarity')
  plt.xlabel('Iteration')
  plt.title("Mean cosine similarity per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print(f'the average cosine similarity is: {mean}')

Exection time

In [ ]:
question = ["Q"] + dataset["test"]["input"][1]
start = time.time()
generated_response = generate_tex_parall(tokenizer, [question], max_len)
end = time.time()

print("exection time: "+ str(end - start))

###GPT-2

In [ ]:
#retrieve model and tokenizer
model_path = "gpt2-medium"
model = load_model(model_path)
tokenizer = load_tokenizer(model_path)
tokenizer.padding_side ='left'

scorer = BERTScorer(lang='en')

In [ ]:
def generate_text_parall_raw(tokenizer, sequence, max_length):
    tokenizer.pad_token = tokenizer.eos_token

    ids = tokenizer(sequence, return_tensors='pt', padding=True ,truncation=True)

    final_outputs = model.generate(
        **ids,
        do_sample=True,
        max_length=max_length,
        pad_token_id=model.config.eos_token_id,
        top_k=50,
        top_p=0.95,
    )

    return [ tokenizer.decode(x, skip_special_tokens=True) for x in final_outputs ]

In [ ]:
P = np.array([])
R = np.array([])
F1 = np.array([])
merged_true_answers = [questions[i] + dataset["test"]["output"][i] for i in range(len(questions))]

n_sections = 6

for i in range(n_sections):
  print(i)
  first_index = int(sections[i])
  second_index= int(sections[i+1])
  responses = generate_text_parall_raw(tokenizer, questions[first_index : second_index], max_len)

  P_2, R_2, F1_2 = scorer.score(responses, merged_true_answers[first_index : second_index])
  P = np.append(P, P_2)
  R = np.append(R, R_2)
  F1 = np.append(F1, F1_2)
  print(f"BERTScore Precision : {np.mean(P):.4f}, Recall: {np.mean(R):.4f}, F1: {np.mean(F1):.4f}  #samples: {int((len(questions)/n_splits)*(i+1))}")

In [ ]:
def cum_mean(arr):
  cum_sum = np.cumsum(arr)
  return cum_sum/np.arange(1, len(arr) + 1)

plt.plot(cum_mean(P), color='#2ca02c',label='Precision')
plt.plot(cum_mean(R), color='#d62728',label='Recall')
plt.plot(cum_mean(F1),label='F1 score')

plt.ylabel('Mean scores')
plt.xlabel('Iteration')
plt.title("Mean scores  per iteartion")
plt.legend()
plt.show()

In [ ]:
gpt_true, gpt_generated = raw_gpt["true"], raw_gpt["generated"]
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_list = []
  patience = 20
  k = 0
  for i in range(len(gpt_true)):
    real_answer = gpt_true[i]
    generated_answer = gpt_generated[i]
    if (generated_answer) is not None and (real_answer) is not None:
      score = evaluate_answer(real_answer, generated_answer)
      if i == 0:
        mean = score
      sum += score
      n_try += 1
      old_mean = mean
      mean = sum/n_try
      mean_list.append(mean)

      if abs(mean - old_mean) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot(mean_list, label='cosine similiarity')
  plt.ylabel('Mean cosine similiarity')
  plt.xlabel('Iteration')
  plt.title("Mean cosine similarity per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print(f'the average cosine similarity is: {mean}')

##BioBert

### BioBert fine-tuned

In [ ]:
output_dir = "2_biobert_qa_model"
model = AutoModelForQuestionAnswering.from_pretrained(output_dir)
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.2")

question_answerer = pipeline("question-answering", model=model , tokenizer=tokenizer)
scorer = BERTScorer(lang='en')

n_splits = 160
sections = np.linspace(0, len(dataset["test"]["input"]), n_splits+1)
n_sections = 4

P = np.array([])
R = np.array([])
F1 = np.array([])

for i in range(n_sections):
  print(i)
  first_index = int(sections[i])
  second_index= int(sections[i+1])
  results = question_answerer(question=dataset["test"]["input"][first_index : second_index], context=dataset["test"]["contexts"][first_index : second_index])
  results = pd.DataFrame(results)
  P_2, R_2, F1_2 = scorer.score(list(results["answer"]), dataset["test"]["output"][first_index : second_index])
  P = np.append(P, P_2)
  R = np.append(R, R_2)
  F1 = np.append(F1, F1_2)
  print(f"BERTScore Precision : {np.mean(P):.4f}, Recall: {np.mean(R):.4f}, F1: {np.mean(F1):.4f}  #samples: {int((len(dataset['test']['input'])/n_splits)*(i+1))}")


In [ ]:
def cum_mean(arr):
  cum_sum = np.cumsum(arr)
  return cum_sum/np.arange(1, len(arr) + 1)

plt.plot(cum_mean(P), color='#2ca02c',label='Precision')
plt.plot(cum_mean(R), color='#d62728',label='Recall')
plt.plot(cum_mean(F1),label='F1 score')

plt.ylabel('Mean scores')
plt.xlabel('Iteration')
plt.title("Mean scores  per iteartion")
plt.legend()
plt.show()

In [ ]:
biobert_true, biobert_gen = biobert_tuned["true_answer"], biobert_tuned["generated"]

In [ ]:
index = 10
biobert_true[index]

In [ ]:
biobert_gen[index]

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_list = []
  patience = 20
  k = 0
  for i in range(len(biobert_gen)):
    real_answer = biobert_true[i]
    generated_answer = biobert_gen[i]
    if (generated_answer) is not None and (real_answer) is not None:
      score = evaluate_answer(real_answer, generated_answer)
      if i == 0:
        mean = score
      sum += score
      n_try += 1
      old_mean = mean
      mean = sum/n_try
      mean_list.append(mean)

      if abs(mean - old_mean) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot(mean_list, label='cosine similiarity')
  plt.ylabel('Mean cosine similiarity')
  plt.xlabel('Iteration')
  plt.title("Mean cosine similarity per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print(f'the average cosine similarity is: {mean}')

### BioBert raw model

In [ ]:
### BioBert ###

model_bio = "dmis-lab/biobert-base-cased-v1.2"
model = AutoModelForQuestionAnswering.from_pretrained(model_bio)
tokenizer = AutoTokenizer.from_pretrained(model_bio)

question_answerer = pipeline("question-answering", model=model , tokenizer=tokenizer)
scorer = BERTScorer(lang='en')

n_splits = 160
sections = np.linspace(0, len(dataset["test"]["input"]), n_splits+1)
n_sections = 4

P = np.array([])
R = np.array([])
F1 = np.array([])


for i in range(n_sections):
  print(i)
  first_index = int(sections[i])
  second_index= int(sections[i+1])
  results = question_answerer(question=dataset["test"]["input"][first_index : second_index], context=dataset["test"]["contexts"][first_index : second_index])
  results = pd.DataFrame(results)
  P_2, R_2, F1_2 = scorer.score(list(results["answer"]), dataset["test"]["output"][first_index : second_index])
  P = np.append(P, P_2)
  R = np.append(R, R_2)
  F1 = np.append(F1, F1_2)
  print(f"BERTScore Precision : {np.mean(P):.4f}, Recall: {np.mean(R):.4f}, F1: {np.mean(F1):.4f}  #samples: {int((len(dataset['test']['input'])/n_splits)*(i+1))}")

In [ ]:
def cum_mean(arr):
  cum_sum = np.cumsum(arr)
  return cum_sum/np.arange(1, len(arr) + 1)

plt.plot(cum_mean(P), color='#2ca02c',label='Precision')
plt.plot(cum_mean(R), color='#d62728',label='Recall')
plt.plot(cum_mean(F1),label='F1 score')

plt.ylabel('Mean scores')
plt.xlabel('Iteration')
plt.title("Mean scores  per iteartion")
plt.legend()
plt.show()

In [ ]:
if TEST_MODEL:
  sum = 0
  n_try = 0
  mean_list = []
  patience = 20
  k = 0
  for i in range(len(gpt_true)):
    real_answer = gpt_true[i]
    generated_answer = gpt_generated[i]
    if (generated_answer) is not None and (real_answer) is not None:
      score = evaluate_answer(real_answer, generated_answer)
      if i == 0:
        mean = score
      sum += score
      n_try += 1
      old_mean = mean
      mean = sum/n_try
      mean_list.append(mean)

      if abs(mean - old_mean) < 0.001:
        k += 1
        if k == patience:
          print(f'Early stopping at iteration: {i}')
          break
      else:
        k=0
      print(f'Iteration: {i}, Mean: {mean} \nLast relevant mean change is {k} iteration ago\n\n')

In [ ]:
if TEST_MODEL:
  plt.plot(mean_list, label='cosine similiarity')
  plt.ylabel('Mean cosine similiarity')
  plt.xlabel('Iteration')
  plt.title("Mean cosine similarity per iteartion")
  plt.legend()
  plt.show()

In [ ]:
if TEST_MODEL:
  print(f'the average cosine similarity is: {mean}')

##Promotheus

To use Prometheus for scoring the answers, it is necessary to define a prompt with specific evaluation criteria to serve as a scoring rubric for the model.
In our case, we defined the evaluation criteria as follows:

* Score 0: The model does not answer correctly

* Score 1: The model does not answer correctly to the user's question omitting important informations (synonyms are accepted).

* Score 2: The model answer correctly to the user's question omitting more relavant informations and adding incorrect and irrelevant informations (synonyms are accepted).

* Score 3: The model answer correctly to the user's question omitting more relevant informations and adding only correct and relevant informations (synonyms are accepted).

* Score 4: The model answer correctly to the user's question omitting only one relevant information and adding only correct and relevant informations (synonyms are accepted).

* Score 5: The model answer correctly to the user's question without omitting any relevant information and adding only correct and relevant informations (synonyms are accepted).


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-chat-hf")
model = LlamaForCausalLM.from_pretrained("kaist-ai/Prometheus-7b-v1.0")

input_text = """###Task Description: An instruction (might include an Input inside it), a response to evaluate, a reference answer that gets a score of 5, and a score rubric representing a evaluation criteria are given.

1. Write a detailed feedback that assess the quality of the response strictly based on the given score rubric, not evaluating in general.

2. After writing a feedback, write a score that is an integer between 1 and 5. You should refer to the score rubric.

3. The output format should look as follows: "Feedback: (write a feedback for criteria) [RESULT] (an integer number between 1 and 5)"

4. Please do not generate any other opening, closing, and explanations.

###The instruction to evaluate: What does it mean if the relative risk is less than 1 in relation to exposure and disease occurrence?

###Response to evaluate: If the relative risk is less than 1 in relation to exposure and disease occurrence, it means that there is a significant difference between the risk of disease occurrence and the risk of exposure to the disease

###Reference Answer (Score 5):  If the relative risk is less than 1, it means that exposure to a particular factor or variable is associated with a decreased occurrence of the disease in question. In other words, individuals who are exposed to the factor or variable are less likely to develop the disease than those who are not exposed. The relative risk is a measure of the strength of the association between exposure and disease occurrence, and a value less than 1 indicates a negative association.

###Score Rubrics:

[Is the model able to answer correctly to the user's question?]

Score 0: The model does not answer correctly

Score 1: The model does not answer correctly to the user's question omitting important informations (synonyms are accepted).

Score 2: The model answer correctly to the user's question omitting more relavant informations and adding incorrect and irrelevant informations (synonyms are accepted).

Score 3: The model answer correctly to the user's question omitting more relevant informations and adding only correct and relevant informations (synonyms are accepted).

Score 4: The model answer correctly to the user's question omitting only one relevant information and adding only correct and relevant informations (synonyms are accepted).

Score 5: The model answer correctly to the user's question without omitting any relevant information and adding only correct and relevant informations (synonyms are accepted).

###Feedback:

"""


In [ ]:
input_ids = tokenizer(input_text, return_tensors="pt").input_ids

outputs = model.generate(input_ids, max_new_tokens = 128, pad_token_id=2)
print(tokenizer.decode(outputs[0]))

The model is highly capable of evaluating generated answers with a level of precision and adherence to the rubric comparable to that of a human evaluator. This stringent evaluation method could provide excellent insights into the model's performance. However, despite its potential, this approach is impractical due to the substantial computational resources it demands, especially when compared to statistical or embedding-based evaluation methods.

##Results


In [ ]:
os.chdir(f'/content/')
os.getcwd()
score_dataset = pd.read_csv('scores.csv')

In [ ]:
score_dataset

As per the above table we can see that the **MedTinyLlama** and **MedGemma** model are the Best models with **90.6** and **91.6** **F1 score** respectively, which was evaluated using the BertScore.

If we also check the cosine similarity of the best two models they are giving good results.

**MedTinyLlama:** 0.78

**MedGemma:** 0.81

We choose TinyLlama as the best model for us due to following reasons mentioned below:


*   The Execution Time of the TinyLlama is less compared to the MedGemma model which helps us to inference in less time.
*   TinyLlama, which boasts 1.1 billion parameters, its applications are mainly suited to environments where larger models might not be as feasible due to hardware limitations or greater efficiency.
* As the F1 Score of TinyLlamma is just 1% less than the MedGemma model we choose it to be our best model.


The best cosine similarity was from the model **MedMistral** as the result from the model where more similar in terms of embeddings.






# Extensions

# Voice Interactive Systems - Introduction

**Enhancing Question-Answering System with TTS and STT**

Text-to-Speech (TTS) and Speech-to-Text (STT) technologies significantly enhance the usability of applications like chatbots and question-answering systems. TTS allows to provide spoken responses, making interactions more natural and accessible, especially for users who prefer or require auditory communication. STT enables users to ask questions verbally, making the interaction hands-free and more convenient. Together, these technologies create a more engaging and efficient user experience, broadening the accessibility and practicality of question-answering applications in various contexts.

**Our Work**

 In the following code blocks, we present an implementation of a voice-interactive question-answering system. The main components are:



*   **Whisper model as SST Technology**: Utilized for automatic speech recognition

*   **Question-answering System**: Based on a pre-trained Large Language Model fine-tuned specifically for medical question-answering taks. Given a medical question obtained through the STT technology, the system is capable of providing a sufficiently accurate and relevant answer.

*   **Tacotron2 and WaveGlow models as TTS Technologies**: Employed to synthetize natural-sounding speech from raw trnascripts , which is going to correspond to the answer of the question-answering systems, without additional prosody information

This combination of advanced STT and TTS technologies, along with a specialized LLM, creates a robust and efficient voice-interactive system capable of handling complex medical inquiries.



## Install and Import

Let's install and then impot some useful libraries

In [ ]:
! pip install --upgrade pip
! pip install --upgrade git+https://github.com//huggingface/transformers.git accelerate datasets[audio]

In [ ]:
!pip install --upgrade transformers optimum accelerate

In [ ]:
!pip install ffmpeg-python

In [ ]:
!pip install numpy scipy librosa unidecode inflect

In [ ]:
%%capture
# Installs Unsloth, Xformers (Flash Attention) and all other packages!
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
import torch

from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from transformers.utils import is_flash_attn_2_available

from datasets import load_dataset

from google.colab.output import eval_js
from google.colab import drive

from IPython.display import HTML, Audio

from base64 import b64decode

import numpy as np

import ffmpeg

import io
from scipy.io.wavfile import read as wav_read
from scipy.io.wavfile import write

import os

from unsloth import FastLanguageModel

import time

## Setup

In [ ]:
MODEL_PATH = '/content/drive/MyDrive/Colab Notebooks/NLP/project/Models/Mistral'
drive.mount('/content/drive')
os.chdir(f'{MODEL_PATH}')
os.getcwd()

Set to true **FAST_WHISPER** to use the fast Whisper model. Note that a GPU is required for this mode.

In [ ]:
FAST_WHISPER= True

# Speech-To-Text Technologies: Whisper Model

Whisper is an automatic speech recognition (ASR) system trained on 680,000 hours of multilingual and multitask supervised data collected from the web. The use of such a large and diverse dataset leads to improved robustness to accents, background noise, and technical language. Moreover, it enables transcription in multiple languages, as well as translation from those languages into English. Whisper's performance varies widely depending on the language.

There are five model sizes, four with English-only versions, offering speed and accuracy tradeoffs.

In order to guarantee a pleasant user experience, tiny, base, and small sizes of Whisper should be used. Even though they have 39M, 74M, and 244M parameters, respectively, they are capable of providing correct transcriptions in a short amount of time without strict constraints on resources and hardware.

Later, we'll explore a faster implementation that will allow us to use larger sizes without having to wait too long for an answer.


In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-small"
model = AutoModelForSpeechSeq2Seq.from_pretrained(
      model_id, torch_dtype = torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
      "automatic-speech-recognition",
      model = model,
      tokenizer= processor.tokenizer,
      feature_extractor= processor.feature_extractor,
      max_new_tokens=128,
      chunk_length_s = 30,
      batch_size= 16,
      return_timestamps=True,
      torch_dtype=torch_dtype,
      device=device
  )

A faster implmentation of Whisper has recently been released, leveraging flash attention 2. To utilize this implementation, a GPU must be available.  Flash Attention provides a more memory-efficient approach and increases efficiency due to optimized GPU memory utilization.
Flash Attention 2 is an evolution of the original Flash Attention. It exploits the asymmetric GPU memory hierarchy to bring significant memory saving (linear instead of quadratic) and runtime speedup (2-4× compared to optimized baselines) with no approximation

The latter model enables us to utilize larger sizes, such as large-v3, which boasts 1550M parameters.

In [ ]:
pipe_fast = pipeline(
      "automatic-speech-recognition",
      model="openai/whisper-large-v3", # select checkpoint from https://huggingface.co/openai/whisper-large-v3#model-details
      torch_dtype=torch.float16,
      device="cuda:0", # or mps for Mac devices
      model_kwargs={"attn_implementation": "flash_attention_2"} if is_flash_attn_2_available() else {"attn_implementation": "sdpa"},
  )

# Text-To-Speech Technologies: Tacotron2 and WaveGlow

Tacotron 2 and WaveGlow form a text-to-speech system, synthesizing natural-sounding speech from raw transcripts without additional prosody information. Tacotron 2 utilizes an encoder-decoder architecture to produce mel spectrograms from input text, while WaveGlow is a flow-based model that generates speech from these spectrograms. Hence, A Vocoder which convert the spectrogram into an audio signal

Moreover, models like Bark offer an alternative approach to synthesizing natural speech. Bark skips the spectrogram stage by using transformers directly from text to audio waveform.

Tacotron remains the preferred choice in a multitude of cases. Its nuanced approach to predicting spectrograms allows for precise control over various aspects of synthesized speech, including intonation, rhythm, and overall naturalness.


Load the Tacotron model from Torchhub:

In [ ]:
tacotron2 = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub', 'nvidia_tacotron2', model_math='fp16')
tacotron2 = tacotron2.to('cuda')
tacotron2.eval()

Load the waveglow model as follows:

In [ ]:
waveglow = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub', 'nvidia_waveglow', model_math='fp16')
waveglow = waveglow.remove_weightnorm(waveglow)
waveglow = waveglow.to('cuda')
waveglow.eval()

# Question-Answering model

The upcoming lines of code loads a pre-trained LLM and tokenizer with specified parameters and enables native 2x faster inference

In [ ]:
max_seq_length = 4096   # TinyLlama's internal maximum sequence length is 2048. We use RoPE Scaling to extend it to 4096 with Unsloth!
dtype = None            # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True     # Use 4bit quantization to reduce memory usage. Can be False.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass


The following block of code uses the pre-trained fine-tuned model to generate an answer for a given question

In [ ]:
def generate_answer(question):
  # alpaca_prompt = Copied from above
  FastLanguageModel.for_inference(model) # Enable native 2x faster inference
  inputs = tokenizer(
  [
      alpaca_prompt.format(
          "Answer this question truthfully", # instruction
          question, # input
          "", # output - leave this blank for generation!
      )
  ], return_tensors = "pt").to("cuda")

  outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True, pad_token_id=tokenizer.eos_token_id)
  tokenizer.batch_decode(outputs)
  predicted_answer = tokenizer.batch_decode(outputs[:, inputs['input_ids'].shape[1]:])[0]
  predicted_answer = predicted_answer[:(len(predicted_answer)-5)]
  return predicted_answer

# Audio Recording

Let's define a function to record our own voice

In [ ]:
AUDIO_HTML = """
<script>
var my_div = document.createElement("DIV");
var my_p = document.createElement("P");
var my_btn = document.createElement("BUTTON");
var t = document.createTextNode("Press to start recording");

my_btn.appendChild(t);
//my_p.appendChild(my_btn);
my_div.appendChild(my_btn);
document.body.appendChild(my_div);

var base64data = 0;
var reader;
var recorder, gumStream;
var recordButton = my_btn;

var handleSuccess = function(stream) {
  gumStream = stream;
  var options = {
    //bitsPerSecond: 8000, //chrome seems to ignore, always 48k
    mimeType : 'audio/webm;codecs=opus'
    //mimeType : 'audio/webm;codecs=pcm'
  };
  //recorder = new MediaRecorder(stream, options);
  recorder = new MediaRecorder(stream);
  recorder.ondataavailable = function(e) {
    var url = URL.createObjectURL(e.data);
    var preview = document.createElement('audio');
    preview.controls = true;
    preview.src = url;
    document.body.appendChild(preview);

    reader = new FileReader();
    reader.readAsDataURL(e.data);
    reader.onloadend = function() {
      base64data = reader.result;
      //console.log("Inside FileReader:" + base64data);
    }
  };
  recorder.start();
  };

recordButton.innerText = "Recording... press to stop";

navigator.mediaDevices.getUserMedia({audio: true}).then(handleSuccess);


function toggleRecording() {
  if (recorder && recorder.state == "recording") {
      recorder.stop();
      gumStream.getAudioTracks()[0].stop();
      recordButton.innerText = "Saving the recording... pls wait!"
  }
}

// https://stackoverflow.com/a/951057
function sleep(ms) {
  return new Promise(resolve => setTimeout(resolve, ms));
}

var data = new Promise(resolve=>{
//recordButton.addEventListener("click", toggleRecording);
recordButton.onclick = ()=>{
toggleRecording()

sleep(2000).then(() => {
  // wait 2000ms for the data to be available...
  // ideally this should use something like await...
  //console.log("Inside data:" + base64data)
  resolve(base64data.toString())

});

}
});

</script>
"""

def get_audio():
  display(HTML(AUDIO_HTML))
  data = eval_js("data")
  binary = b64decode(data.split(',')[1])

  process = (ffmpeg
    .input('pipe:0')
    .output('pipe:1', format='wav')
    .run_async(pipe_stdin=True, pipe_stdout=True, pipe_stderr=True, quiet=True, overwrite_output=True)
  )
  output, err = process.communicate(input=binary)

  with open('audio.wav', 'wb') as f:
    f.write(output)

  return 'audio.wav'

# Interactive Question-Answering Function

This function processes user question via audio input, genereates textual responses and convert them into synthesized speech.

In [ ]:
def voice_interactive_QnA(pipe, isFast):
  audio = get_audio()
  if isFast:
    question = pipe(audio, chunk_length_s=30, batch_size=24, return_timestamps=True)
  else:
    question = pipe(audio, generate_kwargs={"language": "english"})

  text = generate_answer(question["text"])
  # Split text into sentences
  sentences = text.split('. ')

  # Generate audio for each sentence and concatenate
  audio_clips = []
  utils = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub', 'nvidia_tts_utils')
  for sentence in sentences:
      # Ensure each sentence ends with a period for proper pronunciation
      if not sentence.endswith('.'):
          sentence += '.'

      # Process the sentence text
      sequences, lengths = utils.prepare_input_sequence([sentence])
      with torch.no_grad():
        mel, _, _ = tacotron2.infer(sequences, lengths)

      # Get the sampling rate and the generated speech
      with torch.no_grad():
        audio = waveglow.infer(mel)
      audio_numpy = audio[0].data.cpu().numpy()
      rate = 22050

      audio_clips.append(audio_numpy)

  # Concatenate all audio arrays
  final_audio_array = np.concatenate(audio_clips)

  # Save the concatenated audio to a file


  # If you have librosa and soundfile installed, you can save the audio like this:
  #import soundfile as sf
  #sf.write("final_output.wav", final_audio_array, rate)

  return final_audio_array, question["text"]
  # Or, if you want to play it directly in Python
  #from IPython.display import Audio
  #Audio(data=final_audio_array, rate=rate)

# Welcome to MedCortana

This is a brief guide on how to use it:


1.  **Start Recording**: Once the code cell is executed, a button will appear, signaling that the recording has commenced. Speak your question clearly into the microphone

2.  **Stop Recording**: Once you've finished your question, press the button again to stop the recording

3. **Processing**: The system will take 30 to 60 seconds to generate a response, depending on the length of the question

4. **Playback**: Your recording will appear, and you can listen to the generated response

5. **Continue or Quit**: After the playback, the system will ask if you want to continue with further questions or quit. Press Ener to continue or type 'exit' to quit

That being said, enjoy MedCortana!



In [ ]:
while True:
  answer, question = voice_interactive_QnA(pipe_fast, FAST_WHISPER)
  print(f'User: {question}')
  print("MedCortana:")
  display(Audio(data=answer, rate=22050))
  time.sleep(3)
  user_input = input("Press Enter to continue or type 'exit' to quit: ")
  print("--------------------------------------------------------------------------------------------------------------------------------------------")
  # Check if the user wants to continue or exit
  if user_input.lower() == 'exit':
      break